<a href="https://colab.research.google.com/github/miray7yuce/quadcopter-rl-copilot/blob/main/notebooks/quadcopter_rl_dogfight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q stable-baselines3 gymnasium
!pip install -q jsbsim==1.2.4
!pip install -q pyyaml
!pip install -q optuna

import jsbsim
print(jsbsim.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 13.9 MB/s eta 0:00:00
1.2.4


In [2]:
!pip install fastapi uvicorn websockets --quiet


In [3]:
#repodaki güncel dosyaları çeker PUSHLAMADAN ÇALIŞTIRMA
from google.colab import userdata
import os

USER  = "miray7yuce"
REPO  = "quadcopter-rl-copilot"
TOKEN = userdata.get('GH_TOKEN')

!git config --global user.email "miray7yuce@gmail.com"
!git config --global user.name "miray7yuce"

os.environ['REMOTE'] = f"https://{TOKEN}@github.com/{USER}/{REPO}.git"
!rm -rf /content/repo
!git clone -q $REMOTE /content/repo
!ls -a /content/repo

.			   f450-drone-framestl.stl   ppo_vs_sac_comparison.png
..			   .git			     README.md
configs			   .gitignore		     requirements.txt
dogfightSim_realtime.html  main.py		     runs
droneSim_realtime.html	   notebooks		     sac_final.acmi
export_acmi.py		   ppo_final.acmi	     src
export_episode_csv.py	   ppo_flight_final.acmi     telemetry_for_html.json
exports			   ppo_flight_telemetry.csv  train_a_log.txt
export_tacview_csv.py	   ppo_telemetry.csv	     train_b_log.txt


In [4]:
import torch
print("torch OK:", torch.__version__)
import stable_baselines3
print("sb3 OK:", stable_baselines3.__version__)
import jsbsim
print("jsbsim OK:", jsbsim.__version__)
import fastapi, uvicorn
print("fastapi/uvicorn OK")

torch OK: 2.11.0+cpu


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


sb3 OK: 2.9.0
jsbsim OK: 1.2.4


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


fastapi/uvicorn OK


In [5]:
import os
os.chdir('/content')
print(os.getcwd())

/content


In [6]:
!pip freeze | grep -iE "^(jsbsim|stable-baselines3|gymnasium|torch|numpy)=" > /content/repo/requirements.txt
!cat /content/repo/requirements.txt

gymnasium==1.3.0
jsbsim==1.2.4
numpy==2.1.3


In [7]:
import os, sys

BASE = "/content/repo"

for d in ["src/drone_rl/envs", "src/drone_rl/utils", "configs"]:
    os.makedirs(f"{BASE}/{d}", exist_ok=True)

for p in ["src/drone_rl", "src/drone_rl/envs", "src/drone_rl/utils"]:
    open(f"{BASE}/{p}/__init__.py", "a").close()

with open(f"{BASE}/.gitignore", "w") as f:
    f.write("__pycache__/\n*.zip\n*.pkl\nlogs/\nruns/\n.ipynb_checkpoints/\n")

sys.path.insert(0, f"{BASE}/src")

!find /content/repo -not -path '*/.git/*' -type f | sort

os.environ['PYTHONPATH'] = f"{BASE}/src"

/content/repo/configs/dogfight_stage_a.yaml
/content/repo/configs/dogfight_stage_b.yaml
/content/repo/configs/ppo_flight_stage1.yaml
/content/repo/configs/ppo_flight.yaml
/content/repo/configs/ppo_hover_customnet.yaml
/content/repo/configs/ppo_hover.yaml
/content/repo/dogfightSim_realtime.html
/content/repo/droneSim_realtime.html
/content/repo/export_acmi.py
/content/repo/export_episode_csv.py
/content/repo/exports/Quadrotor (BEST) [Red] - ep1.csv
/content/repo/exports/Quadrotor (BEST) [Red] - rec1.csv
/content/repo/exports/Quadrotor (TRAINING) [Blue] - ep1.csv
/content/repo/exports/Quadrotor (TRAINING) [Blue] - rec1.csv
/content/repo/export_tacview_csv.py
/content/repo/f450-drone-framestl.stl
/content/repo/.gitignore
/content/repo/main.py
/content/repo/notebooks/quadcopter_rl_dogfight.ipynb
/content/repo/notebooks/quadcopter_rl.ipynb
/content/repo/ppo_final.acmi
/content/repo/ppo_flight_final.acmi
/content/repo/ppo_flight_telemetry.csv
/content/repo/ppo_telemetry.csv
/content/repo/ppo

In [125]:
#pull gerekmiyorsa git push
%%bash
# Repo dizinine geç
cd /content/repo

# Tüm değişiklikleri ekle
git add .

git commit -m "dogfight takip hareket düzenlemeleri"
git pull origin main --no-edit
git push origin main

[main 2ae1091] dogfight takip hareket düzenlemeleri
 21 files changed, 5231 insertions(+), 36 deletions(-)
 create mode 100644 exports/dogfight_recording_1.acmi
Already up to date.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
From https://github.com/miray7yuce/quadcopter-rl-copilot
 * branch            main       -> FETCH_HEAD
To https://github.com/miray7yuce/quadcopter-rl-copilot.git
   17cb049..2ae1091  main -> main


In [ ]:
#pull gerekiyorsa git push
%%bash
cd /content/repo

# Önce Colab'daki mevcut değişiklikleri geçici olarak sakla
git stash push -u -m "colab-local-changes"

# GitHub'daki güncel hali al
git pull origin main --no-rebase

# Colab'daki değişiklikleri geri getir
git stash pop

# Değişiklikleri commit et
git add .
git commit -m "ppo aşamalı training ve simülasyon kontrol fix"

# GitHub'a gönder
git push origin main

No local changes to save
Merge made by the 'ort' strategy.
 notebooks/quadcopter_rl.ipynb | 8525 +++++++++++++++++++++--------------------
 1 file changed, 4386 insertions(+), 4139 deletions(-)
On branch main
Your branch is ahead of 'origin/main' by 2 commits.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
From https://github.com/miray7yuce/quadcopter-rl-copilot
 * branch            main       -> FETCH_HEAD
No stash entries found.
To https://github.com/miray7yuce/quadcopter-rl-copilot.git
   89cfcc0..1cc330c  main -> main


In [37]:
!head -100 /content/repo/train_b_log.txt
print("====================")
!wc -l /content/repo/train_b_log.txt
print("====================")
!ls -la /content/repo/runs/dogfight_stage_b/
print("====================")
!ls -la /content/repo/runs/dogfight_stage_b/live_snapshot/ 2>&1
print("====================")
!cat /content/repo/runs/dogfight_pool/manifest.json 2>&1


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
Traceback (most recent call last):
  File "/content/repo/main.py", line 164, in <module>
    main()
    ~~~~^^
  File "/content/repo/main.py", line 160, in main
    args.func(args)
    ~~~~~~~~~^^^^^^
  File "/content/repo/main.py", line 60, in cmd_train_b
    train_mod.main()
    ~~~~~~~~~~~~~~^^
  File "/content/repo/src/drone_rl/dogfight/train.py", line 378, in main
    cfg = load_dogfight_config(args.config)
  File "/content/repo/src/drone_rl/dogfight/config.py", line 266, in load_dogfight_config
    env=DogfightEnvConfig(**_filter_env(raw.get("env", {}))),
        ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeErro

In [110]:
%%writefile /content/repo/src/drone_rl/dogfight/config.py
"""Dogfight konfigurasyonu - v8.

v8'de ne degisti (ozet):
  * Odul fonksiyonu tamamen yeniden tasarlandi: "burun hizalama" (align)
    yerine HIZ VEKTORU tabanli ATA/AA geometrisi + Gaussian shaping
    (Benati 2025 tezi, Bolum 4.3.6/4.5.6), mesafe potansiyeli (r_dist),
    yaklasma hizi odulu (r_close), HP/WEZ hasar modeli ve terminal
    kazanma odulu (Chen et al. 2025 - aerospace-12-00265).
  * standoff_* parametreleri KALDIRILDI (yerine dist/too_close terimleri).
  * crash mantigi yumusatildi: egim/donus hizi artik ANINDA sonlandirma
    degil, kademeli ceza; sert sonlandirma sadece gercekten kurtarilamaz
    durumlarda (cok buyuk egim, yer/tavan, arena disi).
  * shaped odul agirligi egitim boyunca sonumleniyor (AOS makalesindeki
    lambda_r decay fikri) - boylece terminal (kazanma) sinyali gitgide
    baskin hale geliyor.
  * gamma 0.99 -> 0.995 (20 Hz'de 2.5 s -> ~5 s efektif ufuk).
  * ent_coef 0 -> 0.005, lineer lr sonumlemesi.
  * Mufredat (kappa-PPG benzeri) parametreleri eklendi.
"""

from dataclasses import dataclass, field
from typing import Optional, List
import yaml


@dataclass
class DogfightEnvConfig:
    # ------------------------------------------------------------------
    # Zamanlama
    # ------------------------------------------------------------------
    episode_seconds: float = 45.0
    physics_hz: int = 240
    control_hz: int = 20

    # ------------------------------------------------------------------
    # Kontrol
    # ------------------------------------------------------------------
    hover_throttle: float = 0.420
    throttle_range: float = 0.25
    roll_authority: float = 0.6
    pitch_authority: float = 0.6
    yaw_authority: float = 0.45
    control_surface_tau_s: float = 0.08

    # YENI: pitch_cmd'nin HANGI isaretinin dronu BURNU YONUNDE
    # ilerlettigi. F450 + ScasEngage kombinasyonunda bu isaret model
    # dosyasina bagli oldugu icin varsayim yapmiyoruz.
    # `python main.py calibrate` komutu bu degeri olcup soyler.
    forward_pitch_sign: float = 1.0

    # ------------------------------------------------------------------
    # Gozlem normalizasyonu (O1)
    # ------------------------------------------------------------------
    velocity_norm_fps: float = 30.0
    range_norm_ft: float = 100.0
    closing_norm_fps: float = 20.0
    dz_norm_ft: float = 50.0
    # Hiz cok dusukken (hover) hiz vektorunun yonu tanimsizdir; bu esigin
    # altinda angajman ekseni yumusak sekilde burun vektorune kayar.
    engage_axis_blend_fps: float = 8.0

    # ------------------------------------------------------------------
    # WEZ / koni
    # ------------------------------------------------------------------
    cone_half_angle_deg: float = 25.0
    cone_range_ft: float = 60.0

    # ------------------------------------------------------------------
    # Odul: takip geometrisi (R1 - Gaussian ATA/AA shaping)
    # ------------------------------------------------------------------
    reward_track_weight: float = 0.50
    reward_threat_weight: float = 0.35
    # Benati tezindeki grid-search sonucu: s = 0.8 rad (~36 deg).
    track_sigma_rad: float = 0.80

    # ------------------------------------------------------------------
    # Odul: mesafe / yaklasma (R2)
    # ------------------------------------------------------------------
    reward_dist_weight: float = 0.15
    dist_ref_ft: float = 150.0
    reward_close_weight: float = 0.30
    closing_ref_fps: float = 15.0
    too_close_ft: float = 25.0
    reward_too_close_weight: float = 0.30

    # ------------------------------------------------------------------
    # Odul: kilitlenme / maruz kalma
    # ------------------------------------------------------------------
    reward_cone_hold: float = 0.15

    # ------------------------------------------------------------------
    # Odul: kontrol duzgunlugu (kucultuldu - eskiden pasifligi tesvik
    # edecek kadar buyuktu)
    # ------------------------------------------------------------------
    reward_tilt_weight: float = 0.02
    reward_spin_weight: float = 0.03
    reward_yawrate_weight: float = 0.02
    reward_jerk_weight: float = 0.03

    # ------------------------------------------------------------------
    # HP / WEZ hasar modeli + terminal kazanma (R3)
    # ------------------------------------------------------------------
    hp_initial: float = 3.0            # "kac saniye kilitte kalinca duser"
    hp_damage_rate: float = 1.0        # HP/saniye (yakinlikla olceklenir)
    win_bonus: float = 50.0            # dusurme / dusurulme
    timeout_hp_bonus: float = 15.0     # sure dolunca HP farkina gore

    # ------------------------------------------------------------------
    # Guvenlik (R4) - yumusak cezalar + sadece kurtarilamaz durumda
    # sert sonlandirma
    # ------------------------------------------------------------------
    crash_penalty: float = 25.0
    opponent_fault_bonus: float = 10.0
    # DUZELTME (yer carpmasi sorunu): eskiden 8 ft idi - sert tabanla
    # arasinda neredeyse hic tepki payi yoktu. 20 ft'e cikarildi.
    crash_min_alt_ft: float = 20.0
    crash_max_alt_ft: float = 300.0
    crash_max_tilt_rad: float = 1.40   # ~80 deg: gercekten kurtarilamaz
    tilt_soft_rad: float = 0.60        # bu acinin uzerinde kademeli ceza
    tilt_soft_weight: float = 0.25
    yawrate_soft_rps: float = 4.0
    yawrate_soft_weight: float = 0.10
    max_horizontal_range_ft: float = 400.0
    boundary_soft_margin_ft: float = 120.0
    boundary_soft_weight: float = 0.25
    # YENI (KRITIK guvenlik freni): terminate_on_fault=False olsa BILE
    # bu esikler HER ZAMAN sonlandirir - collision/HP/numeric_divergence
    # ile ayni oncelikte. Amac 'hafif/orta dengesizligi cezalandirmadan
    # izlemek' (mentor istegi) DEGIL, JSBSim'in fizik motorunun GERCEKTEN
    # KIRILMADAN once bir emniyet freni koymak. crash_max_tilt_rad (80
    # derece) gibi ESKI sert sinirlardan COK daha gevsek tutuldu - amac
    # 'agresif ama fiziksel olarak anlamli' manevralari HALA
    # yasaklamamak, sadece JSBSim'in aerodinamik tablolarinin hic
    # tanimli olmadigi asiri bolgelere girmeden durdurmak.
    extreme_tilt_rad: float = 2.5          # ~143 derece
    extreme_altitude_min_ft: float = -100.0
    extreme_altitude_max_ft: float = 2000.0
    extreme_boundary_ft: float = 2000.0

    min_separation_ft: float = 8.0

    # YENI (mentor istegi): rakip/kendi dengesizlik (asiri egim, irtifa
    # tabani/tavani asma, sinir disi) artik EPISODE'U SONLANDIRMIYOR -
    # SADECE cezalandiriliyor (bkz. _soft_safety_penalty, bu zaten HER
    # ZAMAN calisiyordu, bagimsiz). False yapildiginda _hard_terminate()
    # sonuclari sadece BILGI amacli raporlanir (info['self_fault']/
    # info['opp_fault']), episode'u BITIRMEZ - boylece TRAINING/BEST
    # dengesini kaybettiginde ani 'reset' olmadan davranisi uzun sureli
    # izlemek mumkun olur.
    # NOT: collision (carpisma) ve HP=0 (kazanma/kaybetme) BU ANAHTARDAN
    # BAGIMSIZ, HER ZAMAN episode'u bitirir - bunlar instabilite degil,
    # anlamli mucadele sonuclaridir. Ayrica JSBSim'in aerodinamik
    # tablolari normal ucus zarfinin cok disinda (orn. uzun sureli asiri
    # egim) tanimsiz olabileceginden, SAYISAL SAPMA (NaN/Inf) tespit
    # edilirse bu anahtardan BAGIMSIZ olarak yine de sonlandirilir -
    # bu bir 'instabilite cezasi' degil, simulasyon cokmesin diye
    # gereken bir guvenlik agidir.
    terminate_on_fault: bool = True

    # YENI: irtifa tabani/tavani icin de tilt/yawrate/boundary'deki gibi
    # KADEMELI ceza. Eskiden bu SADECE sert sinirdi (crash_min_alt_ft/
    # crash_max_alt_ft) - ajan yere yaklastigini hic 'hissetmeden' aniden
    # carpiyordu, cunku hicbir erken uyari sinyali yoktu (diger 3 guvenlik
    # terimi icin vardi, bu bir eksiklikti). Taban icin marj daha genis
    # tutuldu cunku yere carpma cok daha sik/tehlikeli.
    alt_floor_soft_margin_ft: float = 60.0
    alt_floor_soft_weight: float = 0.35
    alt_ceiling_soft_margin_ft: float = 40.0
    alt_ceiling_soft_weight: float = 0.15

    # YENI: hizli inis (yuksek negatif dikey hiz) dogrudan cezalandirilir -
    # ozellikle rakip asagidayken 'menzili kapatma' odulu dalisi tesvik
    # ediyordu; bu terim dalis HIZINI irtifadan bagimsiz olarak sinirlar.
    descent_rate_soft_fps: float = 12.0
    descent_rate_soft_weight: float = 0.12

    # ------------------------------------------------------------------
    # Shaped odul sonumlemesi + mufredat ilerlemesi
    # ------------------------------------------------------------------
    shaped_weight_start: float = 1.00
    shaped_weight_end: float = 0.45
    shaped_ramp_steps: int = 400_000
    curriculum_ramp_steps: int = 250_000

    # ------------------------------------------------------------------
    # Spawn randomizasyonu (O3)
    # ------------------------------------------------------------------
    base_altitude_ft: float = 150.0
    altitude_jitter_ft: float = 25.0
    spawn_range_min_ft: float = 80.0
    spawn_range_max_ft: float = 220.0
    # DUZELTME: eskiden 18.0/0.12 idi - Stage B'de taze/rastgele agirlikli
    # bir TRAINING politikasi ile henuz mufredata karsi egitilmis (gercek
    # rakiple hic karsilasmamis) bir BEST karsi karsiya gelince, agresif
    # baslangic kosullari JSBSim'in aerodinamik modelinin hic test
    # edilmemis bir bolgesine dusup GERCEK sayisal sapmaya (NaN) yol
    # acabiliyordu (ozellikle episode'un DAHA ILK ADIMINDA). Daha
    # yumusak baslangic, ilk adimdaki patlama riskini azaltir.
    spawn_speed_max_fps: float = 10.0
    spawn_attitude_jitter_rad: float = 0.05

    # ------------------------------------------------------------------
    # Self-play
    # ------------------------------------------------------------------
    opponent_latest_prob: float = 0.60
    # PFSP-lite: eski checkpoint'ler arasinda win_rate'e gore softmax
    # agirlikli ornekleme sicakligi. Kucuk deger = guclu rakiplere daha
    # cok agirlik, buyuk deger = uniform'a yakin.
    opponent_pfsp_temp: float = 0.25
    # Egitim sirasinda rakip politikayi stokastik calistirmak cesitliligi
    # artirir (degerlendirmede yine deterministic kullanilir).
    opponent_deterministic: bool = False


@dataclass
class PPOConfig:
    policy: str = "MlpPolicy"
    n_steps: int = 1024
    batch_size: int = 256
    n_epochs: int = 10
    gamma: float = 0.995          # 20 Hz -> ~5 s efektif ufuk
    gae_lambda: float = 0.95
    clip_range: float = 0.2
    learning_rate: float = 3e-4
    lr_final_frac: float = 0.1    # lineer sonumleme hedefi
    ent_coef: float = 0.005
    vf_coef: float = 0.5
    max_grad_norm: float = 0.5
    target_kl: Optional[float] = 0.05
    clip_reward: float = 10.0
    net_arch_pi: Optional[List[int]] = None
    net_arch_vf: Optional[List[int]] = None
    activation_fn: Optional[str] = None


@dataclass
class TrainConfig:
    timesteps: int = 500_000
    n_envs: int = 4
    # "subproc" | "dummy" | "auto" (auto: n_envs > 1 ise subproc)
    vec: str = "auto"
    seed: Optional[int] = None


@dataclass
class PromotionConfig:
    eval_freq: int = 30_000
    n_eval_episodes: int = 20
    win_rate_threshold: float = 0.55
    mean_reward_improve_pct: float = 5.0
    consecutive_passes_required: int = 2


@dataclass
class DogfightConfig:
    env: DogfightEnvConfig = field(default_factory=DogfightEnvConfig)
    ppo: PPOConfig = field(default_factory=PPOConfig)
    train: TrainConfig = field(default_factory=TrainConfig)
    promotion: PromotionConfig = field(default_factory=PromotionConfig)


_LEGACY_ENV_KEYS = {
    "standoff_target_ft", "standoff_weight_start", "standoff_weight_end",
    "standoff_ramp_steps", "standoff_penalty_cap",
    "reward_align_weight", "reward_exposure_weight", "reward_closing_weight",
    "crash_max_yawrate_rps",
}


def _filter_env(raw_env: dict) -> dict:
    """Eski (v7 ve oncesi) yaml dosyalarinda kalmis, artik kullanilmayan
    anahtarlari sessizce atar - boylece eski bir config dosyasi
    TypeError ile patlamaz, sadece uyari basar."""
    unknown = [k for k in raw_env if k in _LEGACY_ENV_KEYS]
    if unknown:
        print(f"[config] UYARI: artik kullanilmayan env anahtarlari yok "
              f"sayildi (v8'de odul fonksiyonu degisti): {sorted(unknown)}")
    return {k: v for k, v in raw_env.items() if k not in _LEGACY_ENV_KEYS}


def load_dogfight_config(path: Optional[str]) -> DogfightConfig:
    if path is None:
        return DogfightConfig()
    with open(path, "r") as f:
        raw = yaml.safe_load(f) or {}
    return DogfightConfig(
        env=DogfightEnvConfig(**_filter_env(raw.get("env", {}))),
        ppo=PPOConfig(**raw.get("ppo", {})),
        train=TrainConfig(**raw.get("train", {})),
        promotion=PromotionConfig(**raw.get("promotion", {})),
    )




Overwriting /content/repo/src/drone_rl/dogfight/config.py


In [9]:
%%writefile /content/repo/src/drone_rl/dogfight/__init__.py

Overwriting /content/repo/src/drone_rl/dogfight/__init__.py


In [10]:
%%writefile /content/repo/src/drone_rl/dogfight/checkpoint_pool.py
"""Self-play icin checkpoint havuzu - PFSP-lite.

Korunan davranislar:
  * Havuz KENDI KENDINI DOGRULAR: manifest'te olup diskte dosyasi
    eksik olan entry'ler otomatik temizlenir (yarida kalmis bir
    promotion egitimi cokertmesin diye).
  * Versiyon numarasi "son entry + 1" (prune sonrasi cakismayi onler).
  * sample()/latest()/add() manifest'i DISKTEN TAZE okur - egitim
    surecinin ayri bir CheckpointPool nesnesiyle yaptigi promotion'lari
    ortamlar hemen gorur (stale opponent problemi).

v8'de eklenen (T2 - sadelestirilmis):
  * PFSP-lite ornekleme. Eskiden: %70 en son, %30 eskiler arasinda
    UNIFORM. Uniform ornekleme, cok eski/zayif politikalarin surekli
    secilmesine ve egitimin bosa harcanmasina yol aciyordu (bkz.
    aerospace-12-00265, Bolum 3.1 - "obsolete strategies are more
    likely to be sampled, potentially degrading RL performance").
    Simdi eskiler arasinda secim, kayitli win_rate uzerinden softmax
    agirligiyla yapiliyor: guclu checkpoint'ler daha sik secilir.
    Tam Elo + SA-Boltzmann meta-solver yerine, ayni etkiyi veren
    birkac satirlik sade bir surum.
"""

import json
import shutil
from pathlib import Path
from typing import Optional, Tuple

import numpy as np


class CheckpointPool:
    def __init__(self, pool_dir: str):
        self.pool_dir = Path(pool_dir)
        self.pool_dir.mkdir(parents=True, exist_ok=True)
        self.manifest_path = self.pool_dir / "manifest.json"
        self._load()
        self._prune_missing()

    # ------------------------------------------------------------------
    def _load(self):
        if self.manifest_path.exists():
            try:
                self.entries = json.loads(self.manifest_path.read_text())
            except json.JSONDecodeError:
                # baska bir surec tam o anda yaziyor olabilir; eldekini koru
                self.entries = getattr(self, "entries", [])
        else:
            self.entries = []

    def _save(self):
        tmp = self.manifest_path.with_suffix(".json.tmp")
        tmp.write_text(json.dumps(self.entries, indent=2))
        tmp.replace(self.manifest_path)

    def _prune_missing(self):
        valid, removed = [], []
        for e in self.entries:
            if Path(e["model"]).exists() and Path(e["vecnorm"]).exists():
                valid.append(e)
            else:
                removed.append(e["version"])
        if removed:
            print(f"[CheckpointPool] UYARI: diskte dosyasi eksik oldugu icin "
                  f"manifest'ten cikarilan versiyonlar: {removed}")
            self.entries = valid
            self._save()

    def __len__(self):
        return len(self.entries)

    # ------------------------------------------------------------------
    def add(self, model_src: str, vecnorm_src: str, mean_reward: float,
            win_rate: float, note: str = "", obs_dim: Optional[int] = None) -> int:
        self._load()
        next_version = (self.entries[-1]["version"] + 1) if self.entries else 1
        dst_dir = self.pool_dir / f"v{next_version}"
        dst_dir.mkdir(parents=True, exist_ok=True)
        model_dst = dst_dir / "model.zip"
        vecnorm_dst = dst_dir / "vecnormalize.pkl"

        shutil.copy(model_src, model_dst)
        shutil.copy(vecnorm_src, vecnorm_dst)

        entry = {
            "version": next_version,
            "model": str(model_dst),
            "vecnorm": str(vecnorm_dst),
            "mean_reward": float(mean_reward),
            "win_rate": float(win_rate),
            "note": note,
        }
        if obs_dim is not None:
            entry["obs_dim"] = int(obs_dim)
        self.entries.append(entry)
        self._save()
        return next_version

    def latest(self) -> Optional[Tuple[str, str]]:
        self._load()
        if not self.entries:
            return None
        e = self.entries[-1]
        return e["model"], e["vecnorm"]

    def latest_mean_reward(self) -> Optional[float]:
        self._load()
        if not self.entries:
            return None
        return self.entries[-1]["mean_reward"]

    def latest_version(self) -> Optional[int]:
        self._load()
        if not self.entries:
            return None
        return self.entries[-1]["version"]

    # ------------------------------------------------------------------
    def _pfsp_weights(self, entries, temperature: float) -> np.ndarray:
        """win_rate uzerinden softmax. Dusuk sicaklik = guclu
        checkpoint'lere daha cok agirlik; yuksek sicaklik = uniform."""
        wr = np.array([float(e.get("win_rate", 0.5)) for e in entries], dtype=np.float64)
        t = max(float(temperature), 1e-3)
        logits = (wr - wr.max()) / t
        w = np.exp(logits)
        total = w.sum()
        if not np.isfinite(total) or total <= 0:
            return np.full(len(entries), 1.0 / len(entries))
        return w / total

    def sample(self, latest_prob: float = 0.6,
               rng: Optional[np.random.Generator] = None,
               temperature: float = 0.25) -> Optional[Tuple[str, str]]:
        self._load()
        if not self.entries:
            return None
        rng = rng or np.random.default_rng()

        if len(self.entries) == 1 or rng.random() < latest_prob:
            e = self.entries[-1]
        else:
            older = self.entries[:-1]
            probs = self._pfsp_weights(older, temperature)
            idx = int(rng.choice(len(older), p=probs))
            e = older[idx]

        if not (Path(e["model"]).exists() and Path(e["vecnorm"]).exists()):
            print(f"[CheckpointPool] UYARI: v{e['version']} diskte bulunamadi, "
                  f"havuz yeniden dogrulaniyor.")
            self._load()
            self._prune_missing()
            if not self.entries:
                return None
            e = self.entries[-1]

        return e["model"], e["vecnorm"]

    def summary(self) -> str:
        self._load()
        lines = [f"Pool: {self.pool_dir} ({len(self.entries)} versiyon)"]
        for e in self.entries:
            lines.append(f"  v{e['version']}: mean_reward={e['mean_reward']:.2f} "
                         f"win_rate={e['win_rate']:.2f} obs_dim={e.get('obs_dim', '?')} "
                         f"note={e['note']}")
        return "\n".join(lines)

Overwriting /content/repo/src/drone_rl/dogfight/checkpoint_pool.py


In [93]:
%%writefile /content/repo/src/drone_rl/dogfight/dogfight_env.py
"""Iki F450 arasinda 'dogfight' gorevi - TAM 3D fizik.

================================================================
v8 - "NEDEN BIRBIRLERINI TAKIP ETMIYORLARDI" DUZELTMELERI
================================================================

(1) AJAN HEDEFIN HANGI YONDE OLDUGUNU BILMIYORDU.  [KRITIK]
    Eski gozlemde yon bilgisi olarak SADECE `align_owner = cos(ATA)`
    vardi. Kosinus simetriktir: hedef 30 derece SOLDA da 30 derece
    SAGDA da ayni degeri uretir. Yani ajan "ne kadar sapmisim"i
    goruyor ama "hangi yone donmeliyim"i GOREMIYORDU. Tek kareden
    (Markov) dogru karar vermesi matematiksel olarak imkansizdi;
    ancak salinarak/deneyerek arayabiliyordu.
    COZUM: LOS (line-of-sight) vektoru artik GOVDE EKSENINDE, ISARETLI
    3 bilesen olarak veriliyor (los_bx = on, los_by = sag, los_bz =
    asagi). "Sag tarafta" ile "sol tarafta" artik farkli isaretli.

(2) GOZLEMDE KENDI HIZI YOKTU.  [KRITIK]
    17 boyutlu eski vektorde `hdot` disinda hicbir hiz yoktu: u, v, w
    (govde eksen hizlari) yok, rakibin hizi/yonu yok. Bir multikopter
    icin bu gozu kapali ucmaktir - ileri gidip gitmedigini bilemez,
    kesme (lead pursuit) yapmasi imkansizdir.
    COZUM: kendi govde hizlari (u,v,w), hiz buyuklugu ve RAKIBIN hiz
    vektoru (kendi govde ekseninde) gozleme eklendi.

(3) `closing_n` GOZLEMDE HER ZAMAN SIFIRDI.  [GERCEK BUG]
    step() icinde `self._prev_range_ft = rng_ft` atamasi, sonundaki
    `_get_obs_for()` cagrisindan ONCE yapiliyordu; _get_obs_for menzili
    yeniden hesaplayip AYNI _prev_range_ft'ten cikariyordu -> (x-x)/dt
    = 0. info["closing_fps"] dogruydu ama POLITIKAYA giden ozellik
    oluydu. Ayrica _get_obs_for rakip icin closing'i acikca 0.0
    sabitliyordu -> self-play'de dagilim kaymasi.
    COZUM: closing bir kez step()/reset() icinde dogru hesaplanip
    self._closing_fps'te saklaniyor; her iki taraf da ayni (fiziksel
    olarak simetrik) gercek degeri goruyor.

(4) "NISAN AL" ODULU QUADROTOR FIZIGIYLE CELISIYORDU.  [KRITIK]
    _nose_vector burnu psi+theta'dan uretiyordu. Bir F450 ilerlemek
    icin burnunu ASAGI egmek zorundadir (theta < 0). Hedef ayni
    irtifadaysa, yaklasmak icin pitch yaptigin anda align_cos DUSER.
    Yani odul fonksiyonu fiilen "yaklasma, sadece burnunu cevir"
    diyordu; ajanin buldugu lokal optimum tam olarak buydu.
    COZUM: ATA artik BURUN ile degil HIZ VEKTORU ile LOS arasinda
    olculuyor (Benati 2025 tezi, Bolum 4.3.2 - "ATA: angle between the
    agent's velocity vector and the line-of-sight"). Koni/WEZ de ayni
    "angajman ekseni" uzerinde tanimli. Hiz cok dusukken (hover) yon
    tanimsiz oldugu icin eksen yumusak sekilde burun vektorune kayar.

Ikincil duzeltmeler:
  * r_dist (mesafe potansiyeli) + r_close (yaklasma hizi) odulleri
    eklendi - eskiden menzili kapatmak icin net gradyan yoktu.
  * HP / WEZ hasar modeli + terminal kazanma-kaybetme odulu eklendi -
    eskiden "kazanmak" diye bir kavram yoktu (Chen et al. 2025,
    Denklem 3; Benati 2025, Bolum 4.5).
  * Egim ve donus hizi artik ANINDA "crash" degil, kademeli ceza.
    Sert sonlandirma sadece kurtarilamaz durumlarda.
  * Spawn'da hiz + yonelim randomizasyonu (her episode hover'dan
    baslamiyor).
  * Kolaydan zora rakip mufredati (HoverOpponent -> daire -> kappa-PPG
    saf takip), AOS makalesindeki kappa-PPG fikrinin sade hali.

--- KIM RL ILE CALISIYOR? ---
- fdm_self: HER ZAMAN disaridan (PPO) gelen action ile suruluyor.
- fdm_opp: self.opponent_controller uzerinden - Stage A'da scripted,
  Stage B'de dondurulmus/inference-only bir PPO modeli.
- reward SADECE fdm_self icin ogrenme sinyalidir. info'daki opp_reward
  sadece demo arayuzu icin hesaplanan simetrik bir gosterim degeridir.
"""

import math
from typing import Optional

import numpy as np
import gymnasium as gym
from gymnasium import spaces
import jsbsim

LAT0_DEG = 0.0
LON0_DEG = 0.0
FT_PER_DEG_LAT = 364567.2

# Gozlem boyutu. v7'de 17 idi -> v8'de 30.
# DIKKAT: bu degisiklik ESKI checkpoint'leri ve vecnormalize.pkl
# dosyalarini GECERSIZ kilar. Havuzu silip sifirdan egitmek gerekir.
OBS_DIM = 30
ACT_DIM = 4

# YENI: sayisal sapma (NaN/Inf) TEK SEFERLIK teshis ciktisi icin
# modul seviyesinde sayac. opponent_numeric_divergence her bolumde
# tekrarlanan bir sorun oldugunda log'u SPAM'lememek icin sadece
# ILK BIRKAC OLAYI detayli yazdirir, sonra sessizce durur.
_DEBUG_DIVERGENCE_PRINTS_REMAINING = [5]


# ======================================================================
# Kucuk matematik yardimcilari
# ======================================================================

def _wrap_pi(a: float) -> float:
    return math.atan2(math.sin(a), math.cos(a))


def _norm3(x, y, z):
    return math.sqrt(x * x + y * y + z * z)


def _world_to_body(phi, theta, psi, n, e, up):
    """(kuzey, dogu, yukari) dunya vektorunu govde eksenine cevirir.
    Govde ekseni standart havacilik konvansiyonu: x = on, y = sag,
    z = asagi.  R_ned->body = Rx(phi) * Ry(theta) * Rz(psi)."""
    d = -up  # NED'de asagi pozitif
    cps, sps = math.cos(psi), math.sin(psi)
    cth, sth = math.cos(theta), math.sin(theta)
    cph, sph = math.cos(phi), math.sin(phi)

    x1 = n * cps + e * sps
    y1 = -n * sps + e * cps
    z1 = d

    x2 = x1 * cth - z1 * sth
    y2 = y1
    z2 = x1 * sth + z1 * cth

    xb = x2
    yb = y2 * cph + z2 * sph
    zb = -y2 * sph + z2 * cph
    return xb, yb, zb


def _gauss(angle_rad: float, sigma: float) -> float:
    """Benati 2025, Denklem 4.9: exp(-angle^2 / s^2).
    Mutlak-deger yerine karesel (Gaussian) form, kucuk acilarda daha
    duzgun turev ve daha hizli yakinsama veriyor (tezde grid-search ile
    s = 0.8 rad ~ 36 derece en iyi bulunmus)."""
    s = max(sigma, 1e-6)
    return math.exp(-(angle_rad * angle_rad) / (s * s))


# ======================================================================
# Rakip kontrolculeri
# ======================================================================

class BaseOpponentController:
    name = "base"

    def reset(self):
        pass

    def compute_action(self, env: "DogfightEnv") -> np.ndarray:
        raise NotImplementedError


def _pitch_cmd_for_target_theta(cfg, theta_now, q_now, theta_des,
                                kp=2.5, kd=0.45) -> float:
    """pitch_cmd'nin isaret semantigi model dosyasina bagli oldugu icin
    (bkz. cfg.forward_pitch_sign) tum scripted kontrolculer pitch'i
    BURADAN uretir. cfg.forward_pitch_sign, "pozitif pitch_cmd dronu
    burnu yonunde ilerletir" (yani burnu asagi eger) anlamina gelir;
    dolayisiyla theta'yi AZALTMAK icin pozitif komut gerekir."""
    s = cfg.forward_pitch_sign
    cmd = s * (kp * (theta_now - theta_des) + kd * q_now)
    return float(np.clip(cmd, -1.0, 1.0))


class HoverOpponent(BaseOpponentController):
    """Mufredatin en kolay seviyesi: yerinde durur, seviyeli kalir.
    Ajanin once "yaklas ve nisan al"i ogrenmesi icin."""

    name = "hover"

    def __init__(self, cfg, kp_alt=0.10, kd_alt=0.30, kp_roll=3.0, kd_roll=0.5):
        self.cfg = cfg
        self.kp_alt, self.kd_alt = kp_alt, kd_alt
        self.kp_roll, self.kd_roll = kp_roll, kd_roll
        self._target_alt = None

    def reset(self):
        self._target_alt = None

    def compute_action(self, env):
        f = env.fdm_opp
        if self._target_alt is None:
            self._target_alt = f["position/h-agl-ft"]

        phi = f["attitude/phi-rad"]
        theta = f["attitude/theta-rad"]
        p = f["velocities/p-rad_sec"]
        q = f["velocities/q-rad_sec"]

        roll_cmd = float(np.clip(self.kp_roll * (0.0 - phi) - self.kd_roll * p, -1.0, 1.0))
        pitch_cmd = _pitch_cmd_for_target_theta(self.cfg, theta, q, 0.0)
        yaw_cmd = 0.0
        alt_err = self._target_alt - f["position/h-agl-ft"]
        throttle_cmd = float(np.clip(
            self.kp_alt * alt_err - self.kd_alt * f["velocities/h-dot-fps"], -1.0, 1.0))
        return np.array([roll_cmd, pitch_cmd, yaw_cmd, throttle_cmd], dtype=np.float32)


class ScriptedCircleOpponent(BaseOpponentController):
    """Sabit bankli daire - mufredatin ikinci seviyesi. Kovalamaz, ama
    hareketli bir hedef olarak ajana 'lead pursuit' gerektirir."""

    name = "circle"

    def __init__(self, cfg, bank_deg=15.0, kp_roll=3.5, kd_roll=0.3,
                 kp_alt=0.12, kd_alt=0.35):
        self.cfg = cfg
        self.bank_deg = bank_deg
        self.kp_roll, self.kd_roll = kp_roll, kd_roll
        self.kp_alt, self.kd_alt = kp_alt, kd_alt
        self._target_alt_ft = None

    def reset(self):
        self._target_alt_ft = None

    def compute_action(self, env):
        f = env.fdm_opp
        if self._target_alt_ft is None:
            self._target_alt_ft = f["position/h-agl-ft"]

        phi = f["attitude/phi-rad"]
        theta = f["attitude/theta-rad"]
        p = f["velocities/p-rad_sec"]
        q = f["velocities/q-rad_sec"]

        target_roll = math.radians(self.bank_deg)
        roll_cmd = float(np.clip(
            self.kp_roll * (target_roll - phi) - self.kd_roll * p, -1.0, 1.0))
        # hafif ileri egim -> daire cizerken gercekten yol alsin
        pitch_cmd = _pitch_cmd_for_target_theta(self.cfg, theta, q, -0.12)
        yaw_cmd = 0.0
        alt_err = self._target_alt_ft - f["position/h-agl-ft"]
        throttle_cmd = float(np.clip(
            self.kp_alt * alt_err - self.kd_alt * f["velocities/h-dot-fps"], -1.0, 1.0))
        return np.array([roll_cmd, pitch_cmd, yaw_cmd, throttle_cmd], dtype=np.float32)


class KappaPursuitOpponent(BaseOpponentController):
    """kappa-PPG: kappa olasilikla RASTGELE bir yone, aksi halde LOS
    uzerine (saf takip) ucar. Chen et al. 2025 (aerospace-12-00265,
    Denklem 23-24) icindeki kappa-PPG mufredat politikasinin sade hali.
    kappa buyudukce rakip zayiflar/ongorulemezlesir, kucuk kappa ise
    gercek bir kovalayicidir.

    Hiz kontrolu ACIK DONGU DEGIL: istenen ileri hiz ile fiili ileri
    hiz arasindaki hatadan bir hedef egim acisi uretilir, boylece
    pitch isaret/olcek belirsizligi kendini duzeltir."""

    def __init__(self, cfg, kappa=0.3, redirect_period_s=2.5,
                 speed_gain=0.25, max_speed_fps=25.0, max_tilt_rad=0.45,
                 kp_roll=3.5, kd_roll=0.45, kp_yaw=1.2, kd_yaw=0.15,
                 bank_limit_deg=45.0, kp_alt=0.10, kd_alt=0.30,
                 speed_err_gain=0.05):
        self.cfg = cfg
        self.kappa = float(kappa)
        self.redirect_period_s = redirect_period_s
        self.speed_gain = speed_gain
        self.max_speed_fps = max_speed_fps
        self.max_tilt_rad = max_tilt_rad
        self.kp_roll, self.kd_roll = kp_roll, kd_roll
        self.kp_yaw, self.kd_yaw = kp_yaw, kd_yaw
        self.bank_limit = math.radians(bank_limit_deg)
        self.kp_alt, self.kd_alt = kp_alt, kd_alt
        self.speed_err_gain = speed_err_gain
        self.name = f"kappa{int(round(kappa * 100)):03d}"
        self._rand_dir = None
        self._t_since_redirect = 1e9
        self._rng = np.random.default_rng()

    def seed(self, rng):
        self._rng = rng

    def reset(self):
        self._rand_dir = None
        self._t_since_redirect = 1e9

    def _maybe_redirect(self, dt):
        self._t_since_redirect += dt
        if self._t_since_redirect < self.redirect_period_s:
            return
        self._t_since_redirect = 0.0
        if self._rng.random() < self.kappa:
            yaw = self._rng.uniform(-math.pi, math.pi)
            pitch = self._rng.uniform(-0.25, 0.25)
            self._rand_dir = (
                math.cos(pitch) * math.cos(yaw),
                math.cos(pitch) * math.sin(yaw),
                math.sin(pitch),
            )
        else:
            self._rand_dir = None  # saf takip

    def compute_action(self, env):
        f = env.fdm_opp
        self._maybe_redirect(env.control_dt)

        if self._rand_dir is not None:
            dn, de, dup = self._rand_dir
            target_alt = f["position/h-agl-ft"] + dup * 40.0
        else:
            on, oe, oalt = env._position(env.fdm_opp)
            sn, se, salt = env._position(env.fdm_self)
            dn, de, dup = sn - on, se - oe, salt - oalt
            target_alt = salt

        horiz = max(math.hypot(dn, de), 1e-6)
        desired_yaw = math.atan2(de, dn)

        phi = f["attitude/phi-rad"]
        theta = f["attitude/theta-rad"]
        psi = f["attitude/psi-rad"]
        p = f["velocities/p-rad_sec"]
        q = f["velocities/q-rad_sec"]
        r = f["velocities/r-rad_sec"]

        yaw_err = _wrap_pi(desired_yaw - psi)

        # Koordineli donus: yaw hatasina orantili banka acisi
        target_roll = float(np.clip(yaw_err * 1.2, -self.bank_limit, self.bank_limit))
        roll_cmd = float(np.clip(
            self.kp_roll * (target_roll - phi) - self.kd_roll * p, -1.0, 1.0))
        yaw_cmd = float(np.clip(self.kp_yaw * yaw_err - self.kd_yaw * r, -1.0, 1.0))

        # Ileri hiz kontrolu (kapali dongu)
        vn = f["velocities/v-north-fps"]
        ve = f["velocities/v-east-fps"]
        fwd_speed = (vn * dn + ve * de) / horiz
        v_des = float(np.clip(self.speed_gain * horiz, 0.0, self.max_speed_fps))
        v_des *= max(math.cos(yaw_err), 0.0)  # cok sapmisken once don
        tilt_des = float(np.clip(
            self.speed_err_gain * (v_des - fwd_speed), -0.15, self.max_tilt_rad))
        theta_des = -tilt_des  # burun asagi = ileri
        pitch_cmd = _pitch_cmd_for_target_theta(self.cfg, theta, q, theta_des)

        alt_err = target_alt - f["position/h-agl-ft"]
        throttle_cmd = float(np.clip(
            self.kp_alt * alt_err - self.kd_alt * f["velocities/h-dot-fps"], -1.0, 1.0))

        return np.array([roll_cmd, pitch_cmd, yaw_cmd, throttle_cmd], dtype=np.float32)


class OpponentCurriculum:
    """Kolaydan zora rakip havuzu. Egitim ilerledikce daha zor
    seviyelerin 'kilidi acilir'; her reset()'te agirlikli olarak en zor
    acik seviye, bazen de daha kolay seviyeler secilir (AOS makalesinde
    kappa = 0.95 kopyasinin havuzda tutulmasiyla ayni amac: onceki
    becerileri unutmamak)."""

    def __init__(self, cfg, hardest_prob: float = 0.6):
        self.cfg = cfg
        self.hardest_prob = hardest_prob
        self.levels = [
            HoverOpponent(cfg),
            ScriptedCircleOpponent(cfg),
            KappaPursuitOpponent(cfg, kappa=0.70),
            KappaPursuitOpponent(cfg, kappa=0.35),
            KappaPursuitOpponent(cfg, kappa=0.05),
        ]

    def sample(self, rng, progress: float) -> BaseOpponentController:
        n = len(self.levels)
        unlocked = 1 + int(round(float(np.clip(progress, 0.0, 1.0)) * (n - 1)))
        unlocked = int(np.clip(unlocked, 1, n))
        if unlocked == 1 or rng.random() < self.hardest_prob:
            idx = unlocked - 1
        else:
            idx = int(rng.integers(0, unlocked))
        ctrl = self.levels[idx]
        if hasattr(ctrl, "seed"):
            ctrl.seed(rng)
        return ctrl


class NormalizerStats:
    def __init__(self, vecnorm):
        self.mean = vecnorm.obs_rms.mean.astype(np.float32)
        self.var = vecnorm.obs_rms.var.astype(np.float32)
        self.epsilon = vecnorm.epsilon
        self.clip_obs = vecnorm.clip_obs

    @property
    def obs_dim(self) -> int:
        return int(self.mean.shape[-1])

    def normalize(self, obs: np.ndarray) -> np.ndarray:
        normed = (obs - self.mean) / np.sqrt(self.var + self.epsilon)
        return np.clip(normed, -self.clip_obs, self.clip_obs).astype(np.float32)


class PPOOpponentController(BaseOpponentController):
    name = "ppo"

    def __init__(self, model, stats: NormalizerStats, deterministic: bool = False):
        self.model = model
        self.stats = stats
        self.deterministic = deterministic

    def compute_action(self, env: "DogfightEnv") -> np.ndarray:
        obs = env._get_obs_for(env.fdm_opp, env.fdm_self, env.prev_action_opp)
        norm_obs = self.stats.normalize(obs).reshape(1, -1)
        action, _ = self.model.predict(norm_obs, deterministic=self.deterministic)
        return action[0]


# ======================================================================
# Ortam
# ======================================================================

class DogfightEnv(gym.Env):
    metadata = {"render_modes": []}

    def __init__(self, cfg, opponent_controller: Optional[BaseOpponentController] = None,
                 opponent_pool=None, opponent_latest_prob: float = 0.6,
                 opponent_curriculum: Optional[OpponentCurriculum] = None):
        super().__init__()
        self.cfg = cfg

        self.physics_hz = int(cfg.physics_hz)
        self.control_hz = int(cfg.control_hz)
        self.physics_dt = 1.0 / self.physics_hz
        self.substeps = self.physics_hz // self.control_hz
        self.max_steps = int(cfg.episode_seconds * self.control_hz)

        self.action_space = spaces.Box(-1.0, 1.0, shape=(ACT_DIM,), dtype=np.float32)
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(OBS_DIM,), dtype=np.float32)

        self.fdm_self = jsbsim.FGFDMExec(None)
        self.fdm_self.set_debug_level(0)
        if not self.fdm_self.load_model("F450"):
            raise RuntimeError("F450 (self) yuklenemedi")
        self.fdm_self.set_dt(self.physics_dt)

        self.fdm_opp = jsbsim.FGFDMExec(None)
        self.fdm_opp.set_debug_level(0)
        if not self.fdm_opp.load_model("F450"):
            raise RuntimeError("F450 (opp) yuklenemedi")
        self.fdm_opp.set_dt(self.physics_dt)

        self.opponent_curriculum = opponent_curriculum
        self.opponent_controller = opponent_controller or HoverOpponent(cfg)
        self.opponent_pool = opponent_pool
        self.opponent_latest_prob = opponent_latest_prob

        self._surface_self = np.zeros(3, dtype=np.float64)
        self._surface_opp = np.zeros(3, dtype=np.float64)
        self.prev_action_self = np.zeros(ACT_DIM, dtype=np.float32)
        self.prev_action_opp = np.zeros(ACT_DIM, dtype=np.float32)

        self.step_count = 0
        self.my_score = 0
        self.opp_score = 0
        self.hp_self = float(cfg.hp_initial)
        self.hp_opp = float(cfg.hp_initial)
        self._prev_range_ft = None
        self._closing_fps = 0.0

        # Egitim ilerlemesine bagli olarak disaridan set edilir
        self.shaped_weight = float(cfg.shaped_weight_start)
        self.curriculum_progress = 0.0

        self._arena_center_n = 0.0
        self._arena_center_e = 0.0
        self._spawn_debug = {}

    # ------------------------------------------------------------------
    @property
    def control_dt(self):
        return self.substeps * self.physics_dt

    def set_shaped_weight(self, w: float):
        """Shaped (yogun) odulun agirligi. Egitim ilerledikce
        sonumlenir -> terminal kazanma sinyali baskin hale gelir."""
        self.shaped_weight = float(w)

    def set_curriculum_progress(self, p: float):
        self.curriculum_progress = float(np.clip(p, 0.0, 1.0))

    def set_terminate_on_fault(self, enabled: bool):
        """YENI: EGITIM config'ine (yaml) HIC DOKUNMADAN, sadece BU
        ortam nesnesi icin dengesizlik/egim/irtifa/sinir ihlallerinin
        bolumu sonlandirip sonlandirmayacagini calisma aninda degistirir.
        Kullanim alani: offline kayit (ACMI/Tacview CSV) veya canli demo
        - bunlar EGITIMDEN TAMAMEN AYRI, kendi ortam nesnelerini kurar,
        bu yuzden burada False yapmak egitimi hicbir sekilde etkilemez.
        Collision, HP=0, sayisal sapma ve 'felaket' esikleri (bkz.
        _is_catastrophic) BU AYARDAN BAGIMSIZ, HER ZAMAN calismaya
        devam eder - yani kayit sirasinda bile JSBSim'in GERCEKTEN
        kirilmasina izin verilmez."""
        self.cfg.terminate_on_fault = bool(enabled)

    def set_max_episode_seconds(self, seconds: float):
        """YENI: EGITIM config'ine (yaml) DOKUNMADAN, sadece BU ortam
        nesnesi icin bolum (episode) suresini degistirir. Kullanim
        alani: offline ACMI/Tacview kaydi - egitimdeki episode_seconds
        (varsayilan 45s) bir kayit icin YETERSIZ kalirsa (orn. mentor
        60s kesintisiz kayit istiyorsa), bolum suresi kayit hedefinden
        UZUN yapilip TEK BIR bolumun TAMAMI kayit edilir - boylece
        art arda birlestirilmis (reset'li/isinlanmali) bir kayit
        DEGIL, gercekten TEK, kesintisiz bir ucus elde edilir.
        NOT: bu metod ortamin ILK reset()/step() cagrisindan ONCE
        cagrilmalidir - max_steps sadece burada yeniden hesaplanir."""
        self.cfg.episode_seconds = float(seconds)
        self.max_steps = int(seconds * self.control_hz)

    def set_opponent_controller(self, controller: BaseOpponentController):
        self.opponent_controller = controller

    def set_opponent_controller_from_pool(self, model_vecnorm_tuple):
        from drone_rl.dogfight.env_factory import load_opponent_controller
        model_path, vecnorm_path = model_vecnorm_tuple
        self.opponent_controller = load_opponent_controller(
            model_path, vecnorm_path, deterministic=self.cfg.opponent_deterministic)

    # ------------------------------------------------------------------
    # Kinematik yardimcilari
    # ------------------------------------------------------------------
    def _attitude(self, fdm):
        return (fdm["attitude/phi-rad"], fdm["attitude/theta-rad"], fdm["attitude/psi-rad"])

    def _nose_vector(self, fdm):
        _, theta, psi = self._attitude(fdm)
        return (math.cos(theta) * math.cos(psi),
                math.cos(theta) * math.sin(psi),
                math.sin(theta))

    def _world_velocity(self, fdm):
        """(kuzey, dogu, yukari) ft/s."""
        return (fdm["velocities/v-north-fps"],
                fdm["velocities/v-east-fps"],
                -fdm["velocities/v-down-fps"])

    def _engage_axis(self, fdm):
        """DUZELTME (4): angajman ekseni artik BURUN degil HIZ VEKTORU.
        Bir quadrotor ilerlemek icin burnunu asagi eger; burun tabanli
        nisan alma ile yaklasma hareketi birbirini iptal ediyordu.
        Hiz cok dusukken (hover, |v| -> 0) yon tanimsiz olacagi icin
        eksen yumusak sekilde burun vektorune kayar - boylece odulde
        sicrama/sureksizlik olmaz."""
        vn, ve, vu = self._world_velocity(fdm)
        speed = _norm3(vn, ve, vu)
        nn, ne, nu = self._nose_vector(fdm)
        blend = max(self.cfg.engage_axis_blend_fps - speed, 0.0)
        ax, ay, az = vn + blend * nn, ve + blend * ne, vu + blend * nu
        mag = _norm3(ax, ay, az)
        if mag < 1e-6:
            return (nn, ne, nu), speed
        return (ax / mag, ay / mag, az / mag), speed

    def _position(self, fdm):
        lat = fdm["position/lat-gc-deg"]
        lon = fdm["position/long-gc-deg"]
        ft_per_deg_lon = FT_PER_DEG_LAT * math.cos(math.radians(LAT0_DEG))
        north_ft = (lat - LAT0_DEG) * FT_PER_DEG_LAT
        east_ft = (lon - LON0_DEG) * ft_per_deg_lon
        alt_ft = fdm["position/h-agl-ft"]
        return north_ft, east_ft, alt_ft

    def _geom(self, fdm_a, fdm_b):
        """a -> b yonunde tam angajman geometrisi."""
        na, ea, ua = self._position(fdm_a)
        nb, eb, ub = self._position(fdm_b)
        dn, de, dz = nb - na, eb - ea, ub - ua
        rng_ft = max(_norm3(dn, de, dz), 1e-3)
        los = (dn / rng_ft, de / rng_ft, dz / rng_ft)

        axis_a, speed_a = self._engage_axis(fdm_a)
        axis_b, speed_b = self._engage_axis(fdm_b)

        # ATA: a'nin hiz vektoru ile LOS arasindaki aci
        cos_ata = float(np.clip(axis_a[0] * los[0] + axis_a[1] * los[1] + axis_a[2] * los[2],
                                -1.0, 1.0))
        # AA (aspect angle): LOS ile b'nin hiz yonu arasindaki aci.
        # 0 -> b bizden kaciyor (biz onun kuyrugundayiz) = ideal.
        cos_aa = float(np.clip(los[0] * axis_b[0] + los[1] * axis_b[1] + los[2] * axis_b[2],
                               -1.0, 1.0))
        return {
            "rng": rng_ft, "los": los, "dn": dn, "de": de, "dz": dz,
            "axis_a": axis_a, "axis_b": axis_b,
            "speed_a": speed_a, "speed_b": speed_b,
            "cos_ata": cos_ata, "cos_aa": cos_aa,
        }

    def _in_cone(self, cos_axis_los: float, rng_ft: float) -> bool:
        cone_half_cos = math.cos(math.radians(self.cfg.cone_half_angle_deg))
        return bool(cos_axis_los >= cone_half_cos and rng_ft <= self.cfg.cone_range_ft)

    def _boundary_dist(self, fdm) -> float:
        n_ft, e_ft, _ = self._position(fdm)
        return math.hypot(n_ft - self._arena_center_n, e_ft - self._arena_center_e)

    # ------------------------------------------------------------------
    # Gozlem
    # ------------------------------------------------------------------
    def _get_obs_for(self, fdm_owner, fdm_other, prev_action_owner):
        cfg = self.cfg
        g = self._geom(fdm_owner, fdm_other)
        phi, theta, psi = self._attitude(fdm_owner)

        # (1) Gövde eksenindeki ISARETLI LOS - "hangi yone donmeliyim"
        los_bx, los_by, los_bz = _world_to_body(phi, theta, psi, *g["los"])

        # (2) Kendi govde hizlari + rakibin hizi (kendi govde ekseninde)
        vnorm = max(cfg.velocity_norm_fps, 1e-6)
        u = fdm_owner["velocities/u-fps"] / vnorm
        v = fdm_owner["velocities/v-fps"] / vnorm
        w = fdm_owner["velocities/w-fps"] / vnorm
        speed_n = g["speed_a"] / vnorm

        ovn, ove, ovu = self._world_velocity(fdm_other)
        ov_bx, ov_by, ov_bz = _world_to_body(phi, theta, psi, ovn, ove, ovu)
        ov_bx, ov_by, ov_bz = ov_bx / vnorm, ov_by / vnorm, ov_bz / vnorm

        # (3) Gercek yaklasma hizi (artik sifir degil)
        closing_n = self._closing_fps / max(cfg.closing_norm_fps, 1e-6)

        cos_ata = g["cos_ata"]
        cos_aa = g["cos_aa"]
        in_my_cone = 1.0 if self._in_cone(cos_ata, g["rng"]) else 0.0
        # rakibin ATA'si: kendi ekseni ile BANA giden LOS (-los) arasinda
        cos_ata_other = -cos_aa
        in_other_cone = 1.0 if self._in_cone(cos_ata_other, g["rng"]) else 0.0

        alt = fdm_owner["position/h-agl-ft"]
        alt_n = (alt - cfg.base_altitude_ft) / max(cfg.base_altitude_ft, 1e-6)
        boundary_n = self._boundary_dist(fdm_owner) / max(cfg.max_horizontal_range_ft, 1e-6)

        if fdm_owner is self.fdm_self:
            hp_own, hp_other = self.hp_self, self.hp_opp
        else:
            hp_own, hp_other = self.hp_opp, self.hp_self
        hp0 = max(cfg.hp_initial, 1e-6)

        obs = np.array([
            phi, theta,                                     # 0-1
            fdm_owner["velocities/p-rad_sec"] / 5.0,        # 2
            fdm_owner["velocities/q-rad_sec"] / 5.0,        # 3
            fdm_owner["velocities/r-rad_sec"] / 5.0,        # 4
            u, v, w,                                        # 5-7
            speed_n,                                        # 8
            los_bx, los_by, los_bz,                         # 9-11
            g["rng"] / max(cfg.range_norm_ft, 1e-6),        # 12
            closing_n,                                      # 13
            g["dz"] / max(cfg.dz_norm_ft, 1e-6),            # 14
            cos_ata,                                        # 15
            cos_aa,                                         # 16
            ov_bx, ov_by, ov_bz,                            # 17-19
            in_my_cone, in_other_cone,                      # 20-21
            alt_n,                                          # 22
            boundary_n,                                     # 23
            hp_own / hp0, hp_other / hp0,                   # 24-25
            *prev_action_owner,                             # 26-29
        ], dtype=np.float32)

        # YENI (KRITIK): savunma amacli kirpma/temizleme. terminate_on_fault
        # =False oldugundan beri bir drone COK UZUN sure kontrolsuz
        # kalabiliyor (artik aninda sonlandirilmiyor); bu sirada closing_fps,
        # acisal hizlar (p/q/r), rakip hiz vektoru gibi bazi bilesenler HICBIR
        # ZAMAN kirpilmiyordu (eskiden zaten hizla sonlandirildigi icin bu
        # hic sorun olmamisti). Asiri buyuk (ama sonlu) degerler VecNormalize'in
        # calisan varyans hesabini (Welford) TASIRIP NaN'a donusturuyor, bu da
        # politika agina "bulasip" TUM aksiyon tahminlerini NaN yapiyor
        # (egitimi tamamen coken bir ValueError ile durduruyor). _is_finite_state
        # SADECE gercek NaN/Inf'i yakaliyordu, 'cok buyuk ama sonlu' degerleri
        # DEGIL - bu ikinci, tamamlayici bir guvenlik katmani.
        obs = np.nan_to_num(obs, nan=0.0, posinf=50.0, neginf=-50.0)
        obs = np.clip(obs, -50.0, 50.0)
        return obs

    # ------------------------------------------------------------------
    # Reset
    # ------------------------------------------------------------------
    def _init_fdm(self, fdm, alt_ft, heading_deg, north_ft=0.0, east_ft=0.0,
                  speed_fps=0.0, phi=0.0, theta=0.0):
        ft_per_deg_lon = FT_PER_DEG_LAT * math.cos(math.radians(LAT0_DEG))
        fdm["ic/lat-gc-deg"] = LAT0_DEG + north_ft / FT_PER_DEG_LAT
        fdm["ic/long-gc-deg"] = LON0_DEG + east_ft / ft_per_deg_lon
        fdm["ic/h-agl-ft"] = alt_ft
        # (O3) hover'dan degil, govde-ileri bir hizla basla
        fdm["ic/u-fps"] = float(speed_fps)
        fdm["ic/v-fps"] = 0.0
        fdm["ic/w-fps"] = 0.0
        fdm["ic/phi-rad"] = float(phi)
        fdm["ic/theta-rad"] = float(theta)
        fdm["ic/psi-true-rad"] = math.radians(heading_deg)
        fdm.run_ic()
        for i in range(4):
            fdm[f"propulsion/engine[{i}]/set-running"] = 1
        fdm["fcs/ScasEngage"] = 1
        fdm["fcs/aileron-cmd-norm"] = 0.0
        fdm["fcs/elevator-cmd-norm"] = 0.0
        fdm["fcs/rudder-cmd-norm"] = 0.0
        fdm["fcs/throttle-cmd-norm"] = self.cfg.hover_throttle

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        cfg = self.cfg
        rng = self.np_random

        # Stage B: havuzdan rakip ornekle
        if self.opponent_pool is not None:
            sampled = self.opponent_pool.sample(
                cfg.opponent_latest_prob, rng=rng, temperature=cfg.opponent_pfsp_temp)
            if sampled is not None:
                from drone_rl.dogfight.env_factory import load_opponent_controller
                self.opponent_controller = load_opponent_controller(
                    *sampled, deterministic=cfg.opponent_deterministic)
        # Stage A: mufredattan rakip ornekle
        elif self.opponent_curriculum is not None:
            self.opponent_controller = self.opponent_curriculum.sample(
                rng, self.curriculum_progress)

        rng_range = rng.uniform(cfg.spawn_range_min_ft, cfg.spawn_range_max_ft)
        bearing_deg = rng.uniform(0.0, 360.0)
        alt_self = cfg.base_altitude_ft + rng.uniform(-cfg.altitude_jitter_ft, cfg.altitude_jitter_ft)
        alt_opp = cfg.base_altitude_ft + rng.uniform(-cfg.altitude_jitter_ft, cfg.altitude_jitter_ft)
        heading_self = rng.uniform(0.0, 360.0)
        heading_opp = rng.uniform(0.0, 360.0)

        jit = cfg.spawn_attitude_jitter_rad
        spd = cfg.spawn_speed_max_fps

        dn_target = rng_range * math.cos(math.radians(bearing_deg))
        de_target = rng_range * math.sin(math.radians(bearing_deg))

        self._arena_center_n = dn_target / 2.0
        self._arena_center_e = de_target / 2.0

        speed_self = float(rng.uniform(0.0, spd))
        speed_opp = float(rng.uniform(0.0, spd))
        phi_self, theta_self = float(rng.uniform(-jit, jit)), float(rng.uniform(-jit, jit))
        phi_opp, theta_opp = float(rng.uniform(-jit, jit)), float(rng.uniform(-jit, jit))
        # YENI: teshis amacli - bir sayisal sapma olursa HANGI spawn
        # kosullariyla basladigini gorebilmek icin saklaniyor.
        self._spawn_debug = {
            "alt_self": alt_self, "alt_opp": alt_opp,
            "heading_self": heading_self, "heading_opp": heading_opp,
            "speed_self": speed_self, "speed_opp": speed_opp,
            "phi_self": phi_self, "theta_self": theta_self,
            "phi_opp": phi_opp, "theta_opp": theta_opp,
            "opponent_name": getattr(self.opponent_controller, "name", "?"),
        }

        self._init_fdm(self.fdm_self, alt_self, heading_self,
                       north_ft=0.0, east_ft=0.0,
                       speed_fps=speed_self, phi=phi_self, theta=theta_self)
        self._init_fdm(self.fdm_opp, alt_opp, heading_opp,
                       north_ft=dn_target, east_ft=de_target,
                       speed_fps=speed_opp, phi=phi_opp, theta=theta_opp)

        self._surface_self[:] = 0.0
        self._surface_opp[:] = 0.0
        self.prev_action_self = np.zeros(ACT_DIM, dtype=np.float32)
        self.prev_action_opp = np.zeros(ACT_DIM, dtype=np.float32)
        self.step_count = 0
        self.my_score = 0
        self.opp_score = 0
        self.hp_self = float(cfg.hp_initial)
        self.hp_opp = float(cfg.hp_initial)
        self.opponent_controller.reset()

        g = self._geom(self.fdm_self, self.fdm_opp)
        self._prev_range_ft = g["rng"]
        self._closing_fps = 0.0

        return self._get_obs_for(self.fdm_self, self.fdm_opp, self.prev_action_self), {}

    # ------------------------------------------------------------------
    # Aksiyon
    # ------------------------------------------------------------------
    def _apply_action(self, fdm, surface_state, action):
        roll_cmd, pitch_cmd, yaw_cmd, throttle_cmd = action
        aileron_t = float(np.clip(roll_cmd * self.cfg.roll_authority, -1.0, 1.0))
        elevator_t = float(np.clip(-pitch_cmd * self.cfg.pitch_authority, -1.0, 1.0))
        rudder_t = float(np.clip(yaw_cmd * self.cfg.yaw_authority, -1.0, 1.0))
        throttle = float(np.clip(
            self.cfg.hover_throttle + throttle_cmd * self.cfg.throttle_range, 0.0, 1.0))
        targets = np.array([aileron_t, elevator_t, rudder_t])
        alpha = self.physics_dt / (self.cfg.control_surface_tau_s + self.physics_dt)
        surface_state += alpha * (targets - surface_state)
        fdm["fcs/aileron-cmd-norm"] = float(surface_state[0])
        fdm["fcs/elevator-cmd-norm"] = float(surface_state[1])
        fdm["fcs/rudder-cmd-norm"] = float(surface_state[2])
        fdm["fcs/throttle-cmd-norm"] = throttle

    # ------------------------------------------------------------------
    # Sonlandirma
    # ------------------------------------------------------------------
    def _hard_terminate(self, fdm) -> Optional[str]:
        """DUZELTME (R4): egim ve donus hizi artik burada DEGIL, kademeli
        ceza olarak ele aliniyor. Eskiden 0.9 rad (51 derece) egim aninda
        -30 ceza + episode sonu demekti; bu, agresif manevrayi olumcul
        kilip ajani 'duz uc, yaklasma' politikasina itiyordu.

        NOT: bu fonksiyonun DONDURDUGU deger artik cfg.terminate_on_fault
        FALSE oldugunda step() icinde SONLANDIRMA icin KULLANILMIYOR -
        sadece bilgi/teshis amacli (info['self_fault']/info['opp_fault'])
        hesaplanmaya devam ediyor."""
        alt = fdm["position/h-agl-ft"]
        if alt < self.cfg.crash_min_alt_ft:
            return "ground"
        if alt > self.cfg.crash_max_alt_ft:
            return "ceiling"
        phi, theta, _ = self._attitude(fdm)
        if abs(phi) > self.cfg.crash_max_tilt_rad or abs(theta) > self.cfg.crash_max_tilt_rad:
            return "tumble"
        if self._boundary_dist(fdm) > self.cfg.max_horizontal_range_ft:
            return "boundary"
        return None

    def _is_finite_state(self, fdm) -> bool:
        """YENI: sayisal SAPMA (NaN/Inf) guvenlik agi. terminate_on_fault
        FALSE oldugunda drone artik SERT sinirlarla korunmuyor - asiri
        egimde COK uzun sure kalabilir. JSBSim'in aerodinamik tablolari
        boyle bir ucus zarfinin disinda TANIMSIZ olabilir ve teorik
        olarak NaN/Inf uretebilir. Bu durum instabilite CEZASI degil,
        simulasyonun COKMESINI onleyen zorunlu bir kontrol - terminate_on_fault
        ayarindan BAGIMSIZ olarak her zaman calisir."""
        vals = (
            fdm["position/h-agl-ft"],
            fdm["position/lat-gc-deg"], fdm["position/long-gc-deg"],
            fdm["attitude/phi-rad"], fdm["attitude/theta-rad"], fdm["attitude/psi-rad"],
            fdm["velocities/u-fps"], fdm["velocities/v-fps"], fdm["velocities/w-fps"],
        )
        return all(math.isfinite(v) for v in vals)

    def _debug_dump_divergence(self, which: str, fdm, action, prev_surface):
        """YENI: bir sayisal sapma (NaN/Inf) OLDUGU ANDA, TEK SEFERLIK
        (ilk birkac olay icin) TUM ilgili durumu yazdirir - HANGI
        spawn kosullariyla basladigi, HANGI aksiyon uygulandigi, ve
        TAM OLARAK hangi FDM ozelliginin NaN/Inf oldugu. Log'u
        spam'lememek icin modul seviyesindeki sayac tukenince sessizce
        durur."""
        if _DEBUG_DIVERGENCE_PRINTS_REMAINING[0] <= 0:
            return
        _DEBUG_DIVERGENCE_PRINTS_REMAINING[0] -= 1

        props = {
            "h-agl-ft": fdm["position/h-agl-ft"],
            "lat-gc-deg": fdm["position/lat-gc-deg"],
            "long-gc-deg": fdm["position/long-gc-deg"],
            "phi-rad": fdm["attitude/phi-rad"],
            "theta-rad": fdm["attitude/theta-rad"],
            "psi-rad": fdm["attitude/psi-rad"],
            "u-fps": fdm["velocities/u-fps"],
            "v-fps": fdm["velocities/v-fps"],
            "w-fps": fdm["velocities/w-fps"],
            "p-rad_sec": fdm["velocities/p-rad_sec"],
            "q-rad_sec": fdm["velocities/q-rad_sec"],
            "r-rad_sec": fdm["velocities/r-rad_sec"],
            "h-dot-fps": fdm["velocities/h-dot-fps"],
        }
        bad = {k: v for k, v in props.items() if not math.isfinite(v)}

        print(f"\n[DIVERGENCE DEBUG #{5 - _DEBUG_DIVERGENCE_PRINTS_REMAINING[0]}] "
             f"taraf={which} step_count={self.step_count}")
        print(f"  spawn kosullari : {self._spawn_debug}")
        print(f"  uygulanan aksiyon (roll,pitch,yaw,throttle) = {list(np.asarray(action).tolist())}")
        print(f"  yuzey durumu (aileron,elevator,rudder)      = {list(np.asarray(prev_surface).tolist())}")
        print(f"  TUM fdm ozellikleri: {props}")
        print(f"  NaN/Inf OLAN ozellikler: {bad if bad else '(hicbiri - tuhaf, tekrar kontrol edin)'}")
        print("")

    def _is_catastrophic(self, fdm) -> bool:
        """YENI: terminate_on_fault=False olsa BILE HER ZAMAN sonlandiran
        gevsek bir emniyet freni. _hard_terminate() ile KARISTIRMAYIN -
        o eski, siki sinirlar (80 derece egim, 20-300ft irtifa) artik
        SADECE terminate_on_fault=True iken sonlandiriyor. Bu fonksiyon
        COK daha gevsek esikler kullanir (~143 derece egim, -100/+2000ft
        irtifa, 2000ft sinir) - amac 'dengesizligi cezalandirmak' degil,
        JSBSim'in fizik motoru GERCEKTEN sayisal olarak KIRILMADAN once
        (NaN/Inf uretmeden once) bir sinir koymak. _is_finite_state zaten
        NaN olustuktan SONRA yakaliyordu; bu fonksiyon bir onceki adimda,
        NaN olusmadan ONCE devreye girmeyi hedefler."""
        alt = fdm["position/h-agl-ft"]
        if alt < self.cfg.extreme_altitude_min_ft or alt > self.cfg.extreme_altitude_max_ft:
            return True
        phi, theta, _ = self._attitude(fdm)
        if abs(phi) > self.cfg.extreme_tilt_rad or abs(theta) > self.cfg.extreme_tilt_rad:
            return True
        if self._boundary_dist(fdm) > self.cfg.extreme_boundary_ft:
            return True
        return False

    def _soft_safety_penalty(self, fdm) -> float:
        cfg = self.cfg
        phi, theta, _ = self._attitude(fdm)
        tilt = max(abs(phi), abs(theta))
        tilt_excess = max(tilt - cfg.tilt_soft_rad, 0.0)
        tilt_span = max(cfg.crash_max_tilt_rad - cfg.tilt_soft_rad, 1e-3)
        tilt_pen = cfg.tilt_soft_weight * min(tilt_excess / tilt_span, 1.0) ** 2

        yaw_excess = max(abs(fdm["velocities/r-rad_sec"]) - cfg.yawrate_soft_rps, 0.0)
        yaw_pen = cfg.yawrate_soft_weight * min(yaw_excess / max(cfg.yawrate_soft_rps, 1e-3), 1.0) ** 2

        soft_edge = cfg.max_horizontal_range_ft - cfg.boundary_soft_margin_ft
        margin = max(cfg.boundary_soft_margin_ft, 1e-3)
        bnd_excess = max(self._boundary_dist(fdm) - soft_edge, 0.0)
        bnd_pen = cfg.boundary_soft_weight * min(bnd_excess / margin, 1.0) ** 2

        # YENI: irtifa tabani - tilt/yaw/boundary ile AYNI kademeli desen,
        # eskiden eksikti. Taban esiginin UZERINDE kalan mesafe azaldikca
        # (crash_min_alt_ft'e yaklastikca) ceza kareselce artar.
        alt = fdm["position/h-agl-ft"]
        floor_soft_edge = cfg.crash_min_alt_ft + cfg.alt_floor_soft_margin_ft
        floor_margin = max(cfg.alt_floor_soft_margin_ft, 1e-3)
        floor_excess = max(floor_soft_edge - alt, 0.0)
        floor_pen = cfg.alt_floor_soft_weight * min(floor_excess / floor_margin, 1.0) ** 2

        ceil_soft_edge = cfg.crash_max_alt_ft - cfg.alt_ceiling_soft_margin_ft
        ceil_margin = max(cfg.alt_ceiling_soft_margin_ft, 1e-3)
        ceil_excess = max(alt - ceil_soft_edge, 0.0)
        ceil_pen = cfg.alt_ceiling_soft_weight * min(ceil_excess / ceil_margin, 1.0) ** 2

        # YENI: dalis HIZI dogrudan cezalandirilir - irtifadan bagimsiz,
        # erken bir uyari. 'closing' odulu rakip asagidayken dalisi
        # tesvik ediyordu; bu terim asiri hizli inisi caydirir (yatay
        # kovalamaya dokunmaz, sadece dikey hizi sinirlar).
        hdot = fdm["velocities/h-dot-fps"]
        descent_excess = max(-hdot - cfg.descent_rate_soft_fps, 0.0)
        descent_pen = cfg.descent_rate_soft_weight * min(
            descent_excess / max(cfg.descent_rate_soft_fps, 1e-3), 1.0) ** 2

        return tilt_pen + yaw_pen + bnd_pen + floor_pen + ceil_pen + descent_pen

    # ------------------------------------------------------------------
    # Step
    # ------------------------------------------------------------------
    def step(self, action):
        cfg = self.cfg
        action = np.asarray(action, dtype=np.float32).reshape(ACT_DIM)
        opp_action = np.asarray(self.opponent_controller.compute_action(self),
                                dtype=np.float32).reshape(ACT_DIM)

        for _ in range(self.substeps):
            self._apply_action(self.fdm_self, self._surface_self, action)
            self._apply_action(self.fdm_opp, self._surface_opp, opp_action)
            self.fdm_self.run()
            self.fdm_opp.run()

        self.step_count += 1
        dt = self.control_dt

        g = self._geom(self.fdm_self, self.fdm_opp)
        rng_ft = g["rng"]
        cos_ata = g["cos_ata"]          # benim hiz vektorum vs LOS
        cos_aa = g["cos_aa"]            # LOS vs rakibin hiz vektoru
        cos_ata_opp = -cos_aa           # rakibin ATA'si
        cos_aa_opp = -cos_ata           # rakibin gordugu AA

        # (3) closing bir kez, dogru sekilde
        closing_fps = (self._prev_range_ft - rng_ft) / dt
        self._closing_fps = closing_fps
        self._prev_range_ft = rng_ft

        ata = math.acos(float(np.clip(cos_ata, -1.0, 1.0)))
        aa = math.acos(float(np.clip(cos_aa, -1.0, 1.0)))
        ata_opp = math.acos(float(np.clip(cos_ata_opp, -1.0, 1.0)))
        aa_opp = math.acos(float(np.clip(cos_aa_opp, -1.0, 1.0)))

        # --- R1: Gaussian ATA/AA takip odulu -------------------------
        s = cfg.track_sigma_rad
        track_self = _gauss(ata, s) * _gauss(aa, s)
        track_opp = _gauss(ata_opp, s) * _gauss(aa_opp, s)
        r_track = cfg.reward_track_weight * track_self
        r_threat = cfg.reward_threat_weight * track_opp

        # --- R2: mesafe + yaklasma -----------------------------------
        r_dist = cfg.reward_dist_weight * min(rng_ft / max(cfg.dist_ref_ft, 1e-6), 2.0)
        close_gate = 1.0 if rng_ft > cfg.too_close_ft else 0.0
        closing_n = float(np.clip(closing_fps / max(cfg.closing_ref_fps, 1e-6), -1.0, 1.0))
        r_close = cfg.reward_close_weight * closing_n * close_gate
        r_close_opp = cfg.reward_close_weight * closing_n * close_gate
        too_close = max(cfg.too_close_ft - rng_ft, 0.0) / max(cfg.too_close_ft, 1e-6)
        r_tooclose = cfg.reward_too_close_weight * (too_close ** 2)

        # --- Koni / kilitlenme ---------------------------------------
        opp_in_my_cone = self._in_cone(cos_ata, rng_ft)
        me_in_opp_cone = self._in_cone(cos_ata_opp, rng_ft)
        r_lock = cfg.reward_cone_hold if opp_in_my_cone else 0.0
        r_exposed = cfg.reward_cone_hold if me_in_opp_cone else 0.0
        if opp_in_my_cone:
            self.my_score += 1
        if me_in_opp_cone:
            self.opp_score += 1

        # --- R3: HP / WEZ hasar modeli -------------------------------
        dmg_to_opp = 0.0
        dmg_to_self = 0.0
        if opp_in_my_cone:
            prox = 1.0 - 0.5 * min(rng_ft / max(cfg.cone_range_ft, 1e-6), 1.0)
            dmg_to_opp = cfg.hp_damage_rate * prox * dt
            self.hp_opp = max(self.hp_opp - dmg_to_opp, 0.0)
        if me_in_opp_cone:
            prox = 1.0 - 0.5 * min(rng_ft / max(cfg.cone_range_ft, 1e-6), 1.0)
            dmg_to_self = cfg.hp_damage_rate * prox * dt
            self.hp_self = max(self.hp_self - dmg_to_self, 0.0)

        # --- Kontrol duzgunlugu --------------------------------------
        def _control_pen(fdm, act, prev_act):
            phi, theta, _ = self._attitude(fdm)
            tilt = abs(phi) + abs(theta)
            # YENI (KRITIK): spin/yawr eskiden HICBIR SEKILDE sinirli
            # degildi - diger tum guvenlik terimleri min(...,1.0)**2
            # ile sinirliyken bu ikisi degildi. terminate_on_fault=False
            # sayesinde bir drone artik GERCEK bir kontrolsuz donuste
            # (spin) 900 adim (45s) boyunca kalabiliyor; p/q/r JSBSim'in
            # modellemedigi bolgede asiri buyuyebiliyor. Sinirlanmamis
            # bu deger, VecNormalize'in ODUL tarafindaki calisan varyans
            # hesabini (observation'daki gibi) tasirip NaN'a donusturuyordu.
            spin = min(abs(fdm["velocities/p-rad_sec"]) + abs(fdm["velocities/q-rad_sec"]), 50.0)
            yawr = min(abs(fdm["velocities/r-rad_sec"]), 50.0)
            jerk = float(np.sum(np.abs(act - prev_act)))
            return (cfg.reward_tilt_weight * tilt + cfg.reward_spin_weight * spin
                    + cfg.reward_yawrate_weight * yawr + cfg.reward_jerk_weight * jerk)

        control_penalty = _control_pen(self.fdm_self, action, self.prev_action_self)
        opp_control_penalty = _control_pen(self.fdm_opp, opp_action, self.prev_action_opp)

        safety_self = self._soft_safety_penalty(self.fdm_self)
        safety_opp = self._soft_safety_penalty(self.fdm_opp)

        # --- Shaped toplam -------------------------------------------
        shaped_self = (r_track - r_threat - r_dist + r_close - r_tooclose
                       + r_lock - r_exposed - control_penalty - safety_self)
        shaped_opp = (cfg.reward_track_weight * track_opp
                      - cfg.reward_threat_weight * track_self
                      - r_dist + r_close_opp - r_tooclose
                      + r_exposed - r_lock - opp_control_penalty - safety_opp)

        reward = self.shaped_weight * shaped_self
        opp_reward = self.shaped_weight * shaped_opp

        # --- Terminal olaylar (shaped_weight ile OLCEKLENMEZ) --------
        self_fault = self._hard_terminate(self.fdm_self)
        opp_fault = self._hard_terminate(self.fdm_opp)
        # YENI: sayisal sapma kontrolu - terminate_on_fault ayarindan
        # BAGIMSIZ, her zaman calisan zorunlu guvenlik agi.
        self_numeric_ok = self._is_finite_state(self.fdm_self)
        opp_numeric_ok = self._is_finite_state(self.fdm_opp)
        if not self_numeric_ok:
            self._debug_dump_divergence("self", self.fdm_self, action, self._surface_self)
        if not opp_numeric_ok:
            self._debug_dump_divergence("opponent", self.fdm_opp, opp_action, self._surface_opp)
        collided = rng_ft < cfg.min_separation_ft

        crashed = False
        terminated = False
        reset_reason = None

        if collided:
            reward -= cfg.crash_penalty
            opp_reward -= cfg.crash_penalty
            crashed = True
            terminated = True
            reset_reason = "collision"
        elif self.hp_self <= 0.0 and self.hp_opp <= 0.0:
            terminated = True
            reset_reason = "mutual_down"
        elif self.hp_opp <= 0.0:
            reward += cfg.win_bonus
            opp_reward -= cfg.win_bonus
            terminated = True
            reset_reason = "opponent_down"
        elif self.hp_self <= 0.0:
            reward -= cfg.win_bonus
            opp_reward += cfg.win_bonus
            terminated = True
            reset_reason = "self_down"
        elif not self_numeric_ok:
            # YENI: guvenlik agi - terminate_on_fault=False olsa BILE
            # sayisal sapma (NaN/Inf) durumunda sonlandirilir.
            reward -= cfg.crash_penalty
            opp_reward += cfg.opponent_fault_bonus
            crashed = True
            terminated = True
            reset_reason = "self_numeric_divergence"
        elif not opp_numeric_ok:
            reward += cfg.opponent_fault_bonus
            opp_reward -= cfg.crash_penalty
            terminated = True
            reset_reason = "opponent_numeric_divergence"
        elif self._is_catastrophic(self.fdm_self):
            # YENI: terminate_on_fault ayarindan BAGIMSIZ, gevsek
            # ('felaket') emniyet freni - bkz. _is_catastrophic().
            reward -= cfg.crash_penalty
            opp_reward += cfg.opponent_fault_bonus
            crashed = True
            terminated = True
            reset_reason = "self_catastrophic"
        elif self._is_catastrophic(self.fdm_opp):
            reward += cfg.opponent_fault_bonus
            opp_reward -= cfg.crash_penalty
            terminated = True
            reset_reason = "opponent_catastrophic"
        elif cfg.terminate_on_fault and self_fault is not None:
            # DUZELTME (mentor istegi): bu dal artik SADECE
            # cfg.terminate_on_fault=True iken calisir. False oldugunda
            # egim/irtifa/sinir ihlalleri episode'u BITIRMEZ - sadece
            # _soft_safety_penalty uzerinden (asagida zaten hesaplandi,
            # BAGIMSIZ olarak) cezalandirilmaya devam eder.
            reward -= cfg.crash_penalty
            opp_reward += cfg.opponent_fault_bonus
            crashed = True
            terminated = True
            reset_reason = f"self_{self_fault}"
        elif cfg.terminate_on_fault and opp_fault is not None:
            reward += cfg.opponent_fault_bonus
            opp_reward -= cfg.crash_penalty
            terminated = True
            reset_reason = f"opponent_{opp_fault}"

        self.prev_action_self = action.copy()
        self.prev_action_opp = opp_action.copy()

        truncated = bool(self.step_count >= self.max_steps)
        if truncated and reset_reason is None:
            reset_reason = "timeout"
            hp_edge = (self.hp_self - self.hp_opp) / max(cfg.hp_initial, 1e-6)
            reward += cfg.timeout_hp_bonus * hp_edge
            opp_reward -= cfg.timeout_hp_bonus * hp_edge

        # YENI (KRITIK, obs'daki gibi): kaynagi ne olursa olsun (bilinen
        # ya da HENUZ ONGORULEMEYEN bir terim) odulun kendisi de artik
        # savunma amacli sinirlaniyor. Boylece VecNormalize'in odul
        # normalizasyon istatistikleri (calisan varyans) hicbir zaman
        # tasip NaN'a donusemiyor - terminate_on_fault=False'un actigi
        # 'uzun sureli kontrolsuz ucus' senaryosunda HANGI terimin
        # buyudugunu tek tek avlamak yerine, ceviri son bir garanti.
        reward = float(np.clip(np.nan_to_num(reward, nan=0.0, posinf=100.0, neginf=-100.0), -100.0, 100.0))
        opp_reward = float(np.clip(np.nan_to_num(opp_reward, nan=0.0, posinf=100.0, neginf=-100.0), -100.0, 100.0))

        obs = self._get_obs_for(self.fdm_self, self.fdm_opp, self.prev_action_self)

        axis_self = g["axis_a"]
        axis_opp = g["axis_b"]

        info = {
            # geometri
            "range_ft": rng_ft,
            "closing_fps": closing_fps,
            "ata_deg": math.degrees(ata),
            "aa_deg": math.degrees(aa),
            "opp_ata_deg": math.degrees(ata_opp),
            "cos_ata": cos_ata,
            "cos_aa": cos_aa,
            "self_speed_fps": g["speed_a"],
            "opp_speed_fps": g["speed_b"],
            # DUZELTME (gorsellestirme): koni artik Euler acilarindan
            # degil, dogrudan angajman ekseni vektorunden cizilir.
            "self_axis": axis_self,
            "opp_axis": axis_opp,
            # durum
            "opp_in_my_cone": bool(opp_in_my_cone),
            "me_in_opp_cone": bool(me_in_opp_cone),
            "my_score": self.my_score,
            "opp_score": self.opp_score,
            "hp_self": float(self.hp_self),
            "hp_opp": float(self.hp_opp),
            "hp_initial": float(cfg.hp_initial),
            "crashed": crashed,
            "reset_reason": reset_reason,
            # YENI: episode bitmese bile (terminate_on_fault=False)
            # dengesizlik bilgisini KAYBETMEMEK icin ham fault etiketi.
            "self_fault": self_fault,
            "opp_fault": opp_fault,
            "shaped_weight": float(self.shaped_weight),
            "curriculum_progress": float(self.curriculum_progress),
            "opponent_name": getattr(self.opponent_controller, "name", "?"),
            # odul kirilimi (self)
            "track_reward": r_track,
            "threat_penalty": r_threat,
            "dist_penalty": r_dist,
            "close_reward": r_close,
            "tooclose_penalty": r_tooclose,
            "lock_reward": r_lock,
            "exposed_penalty": r_exposed,
            "control_penalty": control_penalty,
            "safety_penalty": safety_self,
            # odul kirilimi (opp - sadece gosterim)
            "opp_track_reward": cfg.reward_track_weight * track_opp,
            "opp_threat_penalty": cfg.reward_threat_weight * track_self,
            "opp_dist_penalty": r_dist,
            "opp_close_reward": r_close_opp,
            "opp_control_penalty": opp_control_penalty,
            "opp_safety_penalty": safety_opp,
            "self_reward": float(reward),
            "opp_reward": float(opp_reward),
            # kinematik (demo)
            "self_hdot_fps": float(self.fdm_self["velocities/h-dot-fps"]),
            "opp_hdot_fps": float(self.fdm_opp["velocities/h-dot-fps"]),
            "self_pos": self._position(self.fdm_self),
            "opp_pos": self._position(self.fdm_opp),
            "self_attitude": self._attitude(self.fdm_self),
            "opp_attitude": self._attitude(self.fdm_opp),
        }

        return obs, float(reward), terminated, truncated, info





Overwriting /content/repo/src/drone_rl/dogfight/dogfight_env.py


In [39]:
!rm -rf /content/repo/runs/dogfight_stage_b

In [12]:
%%writefile /content/repo/src/drone_rl/dogfight/env_factory.py
"""Dogfight env / VecEnv kurulum yardimcilari - v8.

v7'den gelen iyilestirmeler korundu:
  * load_opponent_controller() VecNormalize'i yuklemek icin gercek bir
    DogfightEnv (2 JSBSim FDM'i) kurmuyor; sadece observation/action
    space'i eslesen sahte bir env kullaniyor.
  * Ayni (model, vecnorm) ciftinin denetleyicisi onbellekten donuyor.

v8'de eklenenler:
  * _DummyObsEnv artik OBS_DIM'i dogfight_env'den ALIYOR - gozlem
    boyutu degistiginde burayi guncellemeyi unutma riski kalmadi.
  * load_opponent_controller() gozlem boyutu uyusmazligini ACIK bir
    hata mesajiyla bildiriyor (v7 checkpoint'leri 17 boyutlu, v8 ise
    30 boyutlu - eski havuz kullanilirsa sessizce sacmalamak yerine
    net bir hata verir).
  * Stage A artik kolaydan-zora bir rakip mufredati kullaniyor
    (OpponentCurriculum): hover -> daire -> kappa-PPG saf takip.
  * SubprocVecEnv destegi (T4).
"""

import numpy as np
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv, VecNormalize

from drone_rl.dogfight.dogfight_env import (
    OBS_DIM, ACT_DIM, DogfightEnv, OpponentCurriculum,
    PPOOpponentController, NormalizerStats,
)
from drone_rl.dogfight.checkpoint_pool import CheckpointPool


class _DummyObsEnv(gym.Env):
    """VecNormalize.load() icin JSBSim GEREKTIRMEYEN yer tutucu env."""

    def __init__(self):
        super().__init__()
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(OBS_DIM,), dtype=np.float32)
        self.action_space = spaces.Box(-1.0, 1.0, shape=(ACT_DIM,), dtype=np.float32)

    def reset(self, seed=None, options=None):
        return np.zeros(OBS_DIM, dtype=np.float32), {}

    def step(self, action):
        return np.zeros(OBS_DIM, dtype=np.float32), 0.0, True, False, {}


def make_dummy_vecnorm_env():
    return DummyVecEnv([lambda: Monitor(_DummyObsEnv())])


_controller_cache: dict = {}


def load_opponent_controller(model_path: str, vecnorm_path: str,
                             deterministic: bool = False):
    cache_key = (model_path, vecnorm_path, bool(deterministic))
    if cache_key in _controller_cache:
        return _controller_cache[cache_key]

    from stable_baselines3 import PPO
    vecnorm = VecNormalize.load(vecnorm_path, make_dummy_vecnorm_env())
    stats = NormalizerStats(vecnorm)

    if stats.obs_dim != OBS_DIM:
        raise ValueError(
            f"Gozlem boyutu uyusmuyor: checkpoint {stats.obs_dim} boyutlu, "
            f"bu surum {OBS_DIM} boyutlu bekliyor.\n"
            f"  -> {vecnorm_path}\n"
            f"v8'de gozlem uzayi genisletildi (isaretli govde-eksen LOS, "
            f"govde hizlari, rakip hizi, HP vb.). ESKI havuz ve "
            f"checkpoint'ler GECERSIZDIR: havuz klasorunu silip Stage A'yi "
            f"sifirdan calistirin."
        )

    model = PPO.load(model_path, device="cpu")
    controller = PPOOpponentController(model, stats, deterministic=deterministic)
    _controller_cache[cache_key] = controller
    return controller


def make_dogfight_env(env_cfg, stage: str = "a", pool_dir: str = None,
                      fixed_opponent=None, deterministic_opponent=None) -> DogfightEnv:
    det = env_cfg.opponent_deterministic if deterministic_opponent is None else deterministic_opponent

    if fixed_opponent is not None:
        controller = load_opponent_controller(*fixed_opponent, deterministic=det)
        return DogfightEnv(env_cfg, opponent_controller=controller)

    if stage == "b" and pool_dir:
        pool = CheckpointPool(pool_dir)
        return DogfightEnv(
            env_cfg,
            opponent_pool=pool,
            opponent_latest_prob=env_cfg.opponent_latest_prob,
        )

    # Stage A: kolaydan zora mufredat
    return DogfightEnv(env_cfg, opponent_curriculum=OpponentCurriculum(env_cfg))


def make_dogfight_training_vec_env(env_cfg, n_envs: int, stage: str, pool_dir: str,
                                   training: bool, norm_reward: bool,
                                   clip_obs: float = 10.0, clip_reward: float = 10.0,
                                   vec: str = "auto", seed=None,
                                   vecnormalize_path: str = None):
    """DUZELTME (resume destegi): vecnormalize_path verilirse, YENI bir
    VecNormalize olusturmak yerine diskteki kaydedilmis normalizasyon
    istatistikleri (mean/var vb.) YUKLENIR. Bu olmadan resume edilen bir
    egitim, sifirdan sifirlanmis (yanlis olcekli) bir gozlem
    normalizasyonuyla devam eder - bu da modelin ogrendigi politikayla
    UYUMSUZ girdi dagilimina yol acar (fiilen egitim bozulur)."""
    def _make(rank):
        def _init():
            env = make_dogfight_env(env_cfg, stage=stage, pool_dir=pool_dir)
            if seed is not None:
                env.reset(seed=int(seed) + rank)
            return Monitor(env)
        return _init

    fns = [_make(i) for i in range(n_envs)]

    if vec == "auto":
        vec = "subproc" if n_envs > 1 else "dummy"

    if vec == "subproc" and n_envs > 1:
        try:
            venv = SubprocVecEnv(fns, start_method="spawn")
        except Exception as e:  # Colab/Windows gibi ortamlarda guvenli geri donus
            print(f"[env_factory] SubprocVecEnv basarisiz ({e}); DummyVecEnv'e dusuluyor.")
            venv = DummyVecEnv(fns)
    else:
        venv = DummyVecEnv(fns)

    if vecnormalize_path:
        venv = VecNormalize.load(vecnormalize_path, venv)
        venv.training = training
        venv.norm_reward = norm_reward
        venv.clip_obs = clip_obs
        venv.clip_reward = clip_reward
    else:
        venv = VecNormalize(venv, norm_obs=True, norm_reward=norm_reward,
                            clip_obs=clip_obs, clip_reward=clip_reward, training=training)
    return venv


Overwriting /content/repo/src/drone_rl/dogfight/env_factory.py


In [94]:
%%writefile /content/repo/export_acmi.py
"""Iki drone'un (TRAINING = egitim modeli, BEST = havuzdaki en guncel
dondurulmus rakip) ucusunu Tacview'un KENDI formatinda (.acmi, metin
tabanli ACMI 2.1) kaydeder - dogrudan Tacview'a surukleyip 3B tekrar
izleyebilirsiniz.

DIKKAT: CSV dosyalari Tacview'da ACILMAZ (farkli format). Bu script
TAMAMEN BAGIMSIZ, baska hicbir export script'ine dokunmaz/bagli
degildir - CSV export script'ine hic dokunulmadi.

DUZELTME (surekli kayit): Eskiden TEK bolum kaydediliyordu ve bir
carpisma/crash olunca kayit o anda kesiliyordu - bolumler bazen
~10 saniyede bitince 'daha ne oldugunu anlamadan' kayit sona eriyordu.
Simdi varsayilan olarak EN AZ --min-duration-s (varsayilan 60s) uzunlugunda
KESINTISIZ tek bir dosya uretiliyor: bir bolum biterse (crash/timeout)
ortam otomatik resetlenip kayit AYNI dosyada, AYNI zaman ekseninde
devam ediyor - hedef sureye ulasana kadar. Boylece kisa crash'ler
kaydi erken kesmiyor, en az 1 dakikalik anlamli bir sahne garantileniyor.

Egitim (Stage B) ARKA PLANDA calisirken de guvenle calistirilabilir:
sadece diskteki mevcut snapshot/havuz dosyalarini OKUR, kendi ayri bir
degerlendirme ortaminda calisir - egitim surecine hic mudahale etmez.

Kullanim (repo kokunden):
    cd /content/repo && python tools/export_acmi.py --min-duration-s 60

Cikti (exports/ klasorunde):
    dogfight_recording_1.acmi
    dogfight_recording_2.acmi   (--recordings > 1 verilirse)
    ...
(her 'recording' AYRI bir dosya, ama HER BIRI ICINDE birden fazla
bolum sureklilik icinde birlesik olabilir)

Koordinat notu: DogfightEnv, kuzey/dogu-feet ofsetlerini LAT0_DEG=0.0,
LON0_DEG=0.0 etrafinda enlem/boylama cevirip JSBSim'e veriyor. Bu
script AYNI sabitleri kullanarak north/east ft degerlerini enlem/
boylama geri ceviriyor - yani ACMI'deki konum, ortamin kendi ic
tutarliligiyla BIREBIR eslesir (round-trip test edildi).
"""

import argparse
import math
from datetime import datetime, timezone
from pathlib import Path

from drone_rl.dogfight.config import load_dogfight_config
from drone_rl.dogfight.dogfight_env import (
    DogfightEnv, NormalizerStats, LAT0_DEG, LON0_DEG, FT_PER_DEG_LAT,
)
from drone_rl.dogfight.env_factory import load_opponent_controller, make_dummy_vecnorm_env
from drone_rl.dogfight.checkpoint_pool import CheckpointPool

FT_TO_M = 0.3048


def _find_repo_root() -> Path:
    """Repo koku calisma dizinine (cwd) gore bulunur - script'in
    nereye yazildigina bagimli degildir."""
    candidates = [Path.cwd()]
    here = Path(__file__).resolve()
    candidates += [here.parent, here.parent.parent, here.parent.parent.parent]
    for c in candidates:
        if (c / "configs").is_dir() and (c / "src" / "drone_rl").is_dir():
            return c
    tried = "\n".join(f"  - {c}" for c in candidates)
    raise FileNotFoundError(
        "Repo koku otomatik bulunamadi (icinde hem 'configs/' hem "
        "'src/drone_rl/' olan bir dizin araniyor). Denenen yerler:\n"
        f"{tried}\n"
        "Bu betigi 'cd /content/repo && python tools/export_acmi.py' "
        "seklinde, repo kokunden calistirdiginizdan emin olun; ya da "
        "--config, --live-snapshot-dir, --pool argumanlarini elle verin."
    )


REPO_ROOT = _find_repo_root()
RUNS_DIR = REPO_ROOT / "runs"
EXPORT_DIR = REPO_ROOT / "exports"


def _north_east_to_lonlat(north_ft: float, east_ft: float):
    """DogfightEnv._position()'un TAM TERSI - ayni sabitlerle."""
    ft_per_deg_lon = FT_PER_DEG_LAT * math.cos(math.radians(LAT0_DEG))
    lat = LAT0_DEG + north_ft / FT_PER_DEG_LAT
    lon = LON0_DEG + east_ft / ft_per_deg_lon
    return lon, lat


def _acmi_object_line(obj_id, lon, lat, alt_m, roll_deg, pitch_deg, yaw_deg,
                      name, color):
    t = f"{lon:.9f}|{lat:.9f}|{alt_m:.2f}|{roll_deg:.2f}|{pitch_deg:.2f}|{yaw_deg:.2f}"
    return (f"{obj_id},T={t},Name={name},Color={color},"
           f"Type=Air+Rotorcraft,CallSign={name}")


class _TrainingController:
    def __init__(self, model_path, vecnorm_path):
        from stable_baselines3 import PPO
        from stable_baselines3.common.vec_env import VecNormalize

        vecnorm = VecNormalize.load(str(vecnorm_path), make_dummy_vecnorm_env())
        self.stats = NormalizerStats(vecnorm)
        self.model = PPO.load(str(model_path), device="cpu")

    def compute_action(self, env):
        obs = env._get_obs_for(env.fdm_self, env.fdm_opp, env.prev_action_self)
        norm_obs = self.stats.normalize(obs).reshape(1, -1)
        action, _ = self.model.predict(norm_obs, deterministic=True)
        return action[0]


def _build_eval_env(config_path, live_snapshot_dir, pool_dir, min_duration_s: float):
    cfg = load_dogfight_config(config_path)

    live_dir = Path(live_snapshot_dir)
    model_path = live_dir / "model.zip"
    vecnorm_path = live_dir / "vecnormalize.pkl"
    if not (model_path.exists() and vecnorm_path.exists()):
        raise FileNotFoundError(
            f"Egitim anlik goruntusu bulunamadi: {live_dir}\n"
            f"Stage B en az bir snapshot yazana kadar bekleyin."
        )

    pool = CheckpointPool(pool_dir)
    if len(pool) == 0:
        raise FileNotFoundError(f"Havuz bos: {pool_dir}. Once seed-pool calistirin.")

    opp_controller = load_opponent_controller(*pool.latest(), deterministic=True)
    env = DogfightEnv(cfg.env, opponent_controller=opp_controller)
    env.set_shaped_weight(cfg.env.shaped_weight_end)
    env.set_curriculum_progress(1.0)
    # YENI: EGITIM config'ine (yaml) DOKUNMADAN, sadece BU kayit ortami
    # icin dengesizlik/egim/irtifa/sinir ihlalleri artik bolumu
    # sonlandirmiyor - mentorun istedigi 'kesintisiz uzun ucus kaydi'
    # tam olarak bu. Egitim guvenli/standart sinirlariyla (yaml'daki
    # terminate_on_fault: true) devam ediyor, TAMAMEN ETKILENMEDI.
    # Collision/HP=0/sayisal-sapma/felaket esikleri HALA aktif - yani
    # JSBSim gercekten kirilirsa kayit o bolumu bitirip devam eder,
    # cokme olmaz.
    env.set_terminate_on_fault(False)
    # YENI (KRITIK): bolum suresi, kayit hedefinden (min_duration_s)
    # DAHA UZUN yapiliyor - boylece 60s'lik bir kayit, egitimdeki 45s'lik
    # bolum sinirindan dolayi 2 parcaya BOLUNUP art arda 'birlestirilmis'
    # (reset'li/isinlanmali) olmuyor; TEK, KESINTISIZ bir bolumun tamami
    # kaydediliyor. +10s pay, hedefe TAM ulasilirken bolumun tam o anda
    # bitmemesini garanti eder.
    env.set_max_episode_seconds(min_duration_s + 10.0)

    training_ctrl = _TrainingController(model_path, vecnorm_path)
    return env, training_ctrl, pool.latest_version()


def write_continuous_acmi(path: Path, env, training_ctrl,
                          min_duration_s: float, max_episodes: int) -> tuple:
    """Ortami, TOPLAM sure en az min_duration_s olana kadar art arda
    bolumler halinde kosturup HEPSINI AYNI dosyaya, AYNI (kesintisiz)
    zaman eksenine yazar. Bir bolum crash/timeout ile biterse ortam
    otomatik resetlenip kayit devam eder - boylece kisa bir crash
    kaydi erken kesmez.

    max_episodes: guvenlik siniri - her bolum beklenenden cok kisa
    surerse (orn. surekli erken crash) sonsuz donguye girmemek icin.

    Doner: (toplam_sure_s, kosturulan_bolum_sayisi)
    """
    ref_time = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    lines = [
        "FileType=text/acmi/tacview",
        "FileVersion=2.2",
        f"0,ReferenceTime={ref_time}",
    ]

    total_time = 0.0
    episodes_run = 0

    while total_time < min_duration_s and episodes_run < max_episodes:
        env.reset()
        episodes_run += 1
        info = None
        while True:
            action = training_ctrl.compute_action(env)
            obs, reward, terminated, truncated, info = env.step(action)
            total_time += env.control_dt
            t = round(total_time, 3)

            self_n, self_e, self_alt = info["self_pos"]
            opp_n, opp_e, opp_alt = info["opp_pos"]
            s_roll, s_pitch, s_yaw = info["self_attitude"]
            o_roll, o_pitch, o_yaw = info["opp_attitude"]

            s_lon, s_lat = _north_east_to_lonlat(self_n, self_e)
            o_lon, o_lat = _north_east_to_lonlat(opp_n, opp_e)

            lines.append(f"#{t}")
            lines.append(_acmi_object_line(
                "1", s_lon, s_lat, self_alt * FT_TO_M,
                math.degrees(s_roll), math.degrees(s_pitch), math.degrees(s_yaw),
                name="TRAINING", color="Blue",
            ))
            lines.append(_acmi_object_line(
                "2", o_lon, o_lat, opp_alt * FT_TO_M,
                math.degrees(o_roll), math.degrees(o_pitch), math.degrees(o_yaw),
                name="BEST", color="Red",
            ))

            ended = terminated or truncated
            if ended:
                print(f"    bolum {episodes_run} bitti (t={total_time:.1f}s, "
                     f"sebep={info['reset_reason']}) - hedefe ulasilmadiysa devam ediliyor")
                break
            if total_time >= min_duration_s:
                break

    if total_time < min_duration_s:
        print(f"  UYARI: {max_episodes} bolume ragmen {min_duration_s}s hedefine "
             f"ulasilamadi (toplam {total_time:.1f}s). Bolumler beklenenden cok "
             f"kisa suruyor olabilir - env/egitim durumunu kontrol edin.")

    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    print(f"  {episodes_run} bolum birlestirildi, toplam {total_time:.1f}s -> {path}")
    return total_time, episodes_run


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", type=str, default=None)
    ap.add_argument("--live-snapshot-dir", type=str, default=None)
    ap.add_argument("--pool", type=str, default=None)
    ap.add_argument("--recordings", type=int, default=1,
                    help="Kac ayri .acmi dosyasi uretilecek (her biri en az --min-duration-s)")
    ap.add_argument("--min-duration-s", type=float, default=60.0,
                    help="Her kayit dosyasinin en az kac saniye surecegi (varsayilan 60)")
    ap.add_argument("--max-episodes", type=int, default=40,
                    help="Bir kayit icinde art arda kosturulacak en fazla bolum sayisi "
                         "(sonsuz donguyu onleyen guvenlik siniri)")
    args = ap.parse_args()

    config_path = args.config or str(REPO_ROOT / "configs" / "dogfight_stage_b.yaml")
    live_snapshot_dir = args.live_snapshot_dir or str(RUNS_DIR / "dogfight_stage_b" / "live_snapshot")
    pool_dir = args.pool or str(RUNS_DIR / "dogfight_pool")

    print(f"repo koku      : {REPO_ROOT}")
    print(f"config         : {config_path}")
    print(f"live-snapshot  : {live_snapshot_dir}")
    print(f"pool           : {pool_dir}")

    env, training_ctrl, pool_version = _build_eval_env(
        config_path, live_snapshot_dir, pool_dir, min_duration_s=args.min_duration_s)

    print(f"BEST (havuz)   : v{pool_version}")
    print(f"{args.recordings} kayit uretiliyor, her biri EN AZ {args.min_duration_s:.0f}s "
         f"SUREKLI/KESINTISIZ tek bir bolum olarak (bolum suresi bu kayit icin "
         f"{args.min_duration_s + 10:.0f}s'ye ayarlandi, egitim etkilenmedi)...")

    for rec in range(1, args.recordings + 1):
        out_path = EXPORT_DIR / f"dogfight_recording_{rec}.acmi"
        write_continuous_acmi(out_path, env, training_ctrl,
                              min_duration_s=args.min_duration_s,
                              max_episodes=args.max_episodes)

    print("\nTamamlandi. Uretilen .acmi dosyalarini dogrudan Tacview'a "
         "surukleyip birakabilirsiniz (TRAINING=Mavi, BEST=Kirmizi).")


if __name__ == "__main__":
    main()





Overwriting /content/repo/export_acmi.py


In [41]:
%%writefile /content/repo/src/drone_rl/dogfight/train.py
"""Dogfight egitim betigi - v8.

Degisiklikler:
  * StandoffCurriculumCallback -> ProgressCallback: hem shaped odul
    agirligini sonumler (AOS makalesindeki lambda_r decay) hem de rakip
    mufredatinin ilerlemesini ortamlara bildirir.
  * DiagnosticsCallback: TensorBoard'a ATA, menzil, closing, koni
    yakalama orani, HP ve reset sebebi dagilimi yazar. Bu metrikler
    olmadan "neden takip etmiyor" sorusunu olcerek cevaplayamiyorduk.
  * Lineer ogrenme hizi sonumlemesi, target_kl, max_grad_norm, vf_coef.
  * Kazanma olcutu artik koni adim sayisi degil, HP farki/dusurme.
  * SubprocVecEnv secenegi (--vec).
  * seed-pool artik --config'i dikkate aliyor (eskiden sessizce
    varsayilan config ile degerlendiriyordu).
"""

import argparse
import json
from pathlib import Path

import numpy as np
import torch.nn as nn
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback
from stable_baselines3.common.vec_env import VecNormalize

from drone_rl.dogfight.config import load_dogfight_config
from drone_rl.dogfight.env_factory import (
    make_dogfight_training_vec_env, make_dogfight_env, make_dummy_vecnorm_env,
)
from drone_rl.dogfight.checkpoint_pool import CheckpointPool
from drone_rl.dogfight.dogfight_env import NormalizerStats, OBS_DIM


ACTIVATION_MAP = {"tanh": nn.Tanh, "relu": nn.ReLU}


def build_policy_kwargs(cfg_ppo):
    kwargs = {}
    pi_arch = cfg_ppo.net_arch_pi or [256, 256]
    vf_arch = cfg_ppo.net_arch_vf or [256, 256]
    kwargs["net_arch"] = dict(pi=pi_arch, vf=vf_arch)
    if cfg_ppo.activation_fn:
        kwargs["activation_fn"] = ACTIVATION_MAP[cfg_ppo.activation_fn.lower()]
    return kwargs


def linear_schedule(initial: float, final_frac: float):
    """SB3 progress_remaining 1 -> 0 gider."""
    final = initial * float(final_frac)

    def _f(progress_remaining: float) -> float:
        return final + (initial - final) * progress_remaining

    return _f


def _model_has_nan(model) -> bool:
    """YENI: --resume ile yuklenen bir checkpoint'in agirliklarinda
    NaN/Inf olup olmadigini kontrol eder. Bir NaN cokusu SONRASI
    kaydedilmis (veya cokus TAM kaydetme anina denk gelmis) bir
    live_snapshot, temiz gozlemlerle bile SESSIZCE bozuk kalmaya
    devam eder - ag'in KENDISI zehirlenmis olur. Bu kontrol olmadan
    --resume, corken checkpoint'i sessizce yukleyip AYNI hatayla
    tekrar cokerdi (ilk cokus rollout toplarken, ikincisi egitim
    adimi sirasinda - farkli yerlerde ama AYNI kok neden)."""
    import torch
    for p in model.policy.parameters():
        if not torch.isfinite(p).all():
            return True
    return False


class ProgressCallback(BaseCallback):
    """Shaped odul agirligi + rakip mufredati ilerlemesi.

    Shaped (yogun) odul basta guclu olmali ki ajan hic sinyal almadan
    kaybolmasin; ilerledikce sonumlenmeli ki terminal (kazanma) sinyali
    baskin hale gelsin ve odul bicimlendirmesi politikayi carpitmasin."""

    def __init__(self, env_cfg, update_every: int = 200):
        super().__init__()
        self.env_cfg = env_cfg
        self.update_every = max(update_every, 1)
        self._last_w = None
        self._last_p = None

    def _on_step(self) -> bool:
        if self.n_calls % self.update_every != 0:
            return True
        cfg = self.env_cfg

        frac_s = min(1.0, self.num_timesteps / max(cfg.shaped_ramp_steps, 1))
        w = cfg.shaped_weight_start + (cfg.shaped_weight_end - cfg.shaped_weight_start) * frac_s
        self.training_env.env_method("set_shaped_weight", w)

        p = min(1.0, self.num_timesteps / max(cfg.curriculum_ramp_steps, 1))
        self.training_env.env_method("set_curriculum_progress", p)

        self.logger.record("curriculum/shaped_weight", w)
        self.logger.record("curriculum/progress", p)
        self._last_w, self._last_p = w, p
        return True


class DiagnosticsCallback(BaseCallback):
    """Rollout boyunca info dict'lerini toplayip TensorBoard'a yazar."""

    def __init__(self):
        super().__init__()
        self._reset()

    def _reset(self):
        self.ata = []
        self.rng = []
        self.closing = []
        self.lock = []
        self.exposed = []
        self.hp_self = []
        self.hp_opp = []
        self.reasons = {}
        # YENI: terminate_on_fault=False iken bolum bitmiyor, bu yuzden
        # 'reset_reason' istatistigi artik instabilite sikligini
        # YAKALAYAMIYOR. Bu iki liste, HER ADIMDA (bolum bitse de
        # bitmese de) dengesizlik olup olmadigini takip eder.
        self.self_fault_flags = []
        self.opp_fault_flags = []

    def _on_step(self) -> bool:
        infos = self.locals.get("infos", []) or []
        dones = self.locals.get("dones")

        for i, info in enumerate(infos):
            if "ata_deg" not in info:
                continue
            self.ata.append(info["ata_deg"])
            self.rng.append(info["range_ft"])
            self.closing.append(info["closing_fps"])
            self.lock.append(1.0 if info["opp_in_my_cone"] else 0.0)
            self.exposed.append(1.0 if info["me_in_opp_cone"] else 0.0)
            self.self_fault_flags.append(1.0 if info.get("self_fault") else 0.0)
            self.opp_fault_flags.append(1.0 if info.get("opp_fault") else 0.0)

            # Episode bittiyse: bitis sebebi ve kalan HP'ler
            if dones is not None and i < len(dones) and dones[i]:
                r = info.get("reset_reason") or "unknown"
                self.reasons[r] = self.reasons.get(r, 0) + 1
                self.hp_self.append(info.get("hp_self", 0.0))
                self.hp_opp.append(info.get("hp_opp", 0.0))
        return True

    def _on_rollout_end(self) -> None:
        if self.ata:
            self.logger.record("dogfight/ata_deg_mean", float(np.mean(self.ata)))
            self.logger.record("dogfight/range_ft_mean", float(np.mean(self.rng)))
            self.logger.record("dogfight/closing_fps_mean", float(np.mean(self.closing)))
            self.logger.record("dogfight/lock_rate", float(np.mean(self.lock)))
            self.logger.record("dogfight/exposed_rate", float(np.mean(self.exposed)))
        if self.self_fault_flags:
            self.logger.record("dogfight/self_fault_rate", float(np.mean(self.self_fault_flags)))
            self.logger.record("dogfight/opp_fault_rate", float(np.mean(self.opp_fault_flags)))
        if self.hp_self:
            self.logger.record("dogfight/hp_self_end", float(np.mean(self.hp_self)))
            self.logger.record("dogfight/hp_opp_end", float(np.mean(self.hp_opp)))
        total = sum(self.reasons.values())
        if total:
            for r, c in self.reasons.items():
                self.logger.record(f"reset_reason/{r}", c / total)
        self._reset()


class PeriodicSnapshotCallback(BaseCallback):
    def __init__(self, out_dir: Path, save_freq: int):
        super().__init__()
        self.snapshot_dir = Path(out_dir) / "live_snapshot"
        self.snapshot_dir.mkdir(parents=True, exist_ok=True)
        self.save_freq = max(save_freq, 1)

    def _on_step(self) -> bool:
        if self.n_calls % self.save_freq == 0:
            vecnorm = self.model.get_vec_normalize_env()
            self.model.save(str(self.snapshot_dir / "model"))
            if vecnorm is not None:
                vecnorm.save(str(self.snapshot_dir / "vecnormalize.pkl"))
            meta = {"num_timesteps": int(self.num_timesteps), "obs_dim": int(OBS_DIM)}
            (self.snapshot_dir / "meta.json").write_text(json.dumps(meta))
        return True


_OPPONENT_FAULT_REASONS = (
    "opponent_down", "opponent_ground", "opponent_ceiling",
    "opponent_tumble", "opponent_boundary",
)
_SELF_FAULT_REASONS = (
    "self_down", "self_ground", "self_ceiling", "self_tumble", "self_boundary",
)


def _episode_won(info) -> bool:
    """v8.1 DUZELTME: eskiden SADECE reset_reason=='opponent_down' acik
    galibiyet sayiliyordu; rakip kendi hatasiyla (tumble/boundary/ground/
    ceiling - yani sana hic kilit/hasar vermeden) duserse HICBIR dala
    girmiyor, en sonda hp_opp < hp_self karsilastirmasina dusuyordu.
    Ama boyle bir bolumde IKI TARAFIN DA HP'si hala baslangic degerinde
    (3.0 == 3.0) olabilir - '<' False cikip bu acik ustunluk GALIBIYET
    SAYILMIYORDU. Oysa ortamin odul fonksiyonu bunu zaten
    opponent_fault_bonus ile SENIN lehine puanliyordu (bkz.
    dogfight_env.py step()) - yani egitim sinyali ile degerlendirme
    metrigi CELISIYORDU. Bu, mean_reward surekli iyilesirken win_rate'in
    gurultulu/dusuk kalmasinin (ve promotion'un tikanmasinin) ana
    nedenlerinden biriydi.

    Simdi: rakibin HERHANGI bir kendi-hatasi (opponent_* reset_reason)
    ile bitmesi DOGRUDAN galibiyet sayilir - carpisma (collision) ve
    karsilikli dusme (mutual_down) haric, HP karsilastirmasina hic
    gerek kalmadan."""
    reason = info.get("reset_reason")
    if reason in _OPPONENT_FAULT_REASONS:
        return True
    if reason in _SELF_FAULT_REASONS:
        return False
    if reason in ("collision", "mutual_down"):
        return False
    # timeout ya da beklenmeyen bir sebep: HP farkina bak
    return float(info.get("hp_opp", 0.0)) < float(info.get("hp_self", 0.0))


def _evaluate_vs_fixed(model, vecnorm_stats: NormalizerStats, env_cfg,
                       fixed_opponent, n_episodes: int, shaped_weight=None):
    env = make_dogfight_env(env_cfg, fixed_opponent=fixed_opponent,
                            deterministic_opponent=True)
    if shaped_weight is not None:
        env.set_shaped_weight(shaped_weight)
    env.set_curriculum_progress(1.0)

    rewards, wins = [], 0
    for _ in range(n_episodes):
        obs, _ = env.reset()
        done = False
        ep_reward = 0.0
        info = {}
        while not done:
            norm_obs = vecnorm_stats.normalize(obs).reshape(1, -1)
            action, _ = model.predict(norm_obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action[0])
            ep_reward += reward
            done = terminated or truncated
        rewards.append(ep_reward)
        if _episode_won(info):
            wins += 1
    return float(np.mean(rewards)), wins / max(n_episodes, 1)


class PromotionCallback(BaseCallback):
    def __init__(self, pool: CheckpointPool, env_cfg, promo_cfg, out_dir: Path):
        super().__init__()
        self.pool = pool
        self.env_cfg = env_cfg
        self.promo_cfg = promo_cfg
        self.out_dir = out_dir
        self._consecutive_pass = 0
        self._next_eval = promo_cfg.eval_freq

    def _on_step(self) -> bool:
        if self.num_timesteps < self._next_eval:
            return True
        self._next_eval += self.promo_cfg.eval_freq

        latest = self.pool.latest()
        if latest is None:
            print("[promotion] havuz bos, atlaniyor")
            return True

        vecnorm_train = self.model.get_vec_normalize_env()
        stats = NormalizerStats(vecnorm_train)

        mean_reward, win_rate = _evaluate_vs_fixed(
            self.model, stats, self.env_cfg, latest, self.promo_cfg.n_eval_episodes
        )
        baseline = self.pool.latest_mean_reward()
        if baseline is None or abs(baseline) < 1e-6:
            improve_pct = 100.0
        else:
            improve_pct = (mean_reward - baseline) / abs(baseline) * 100.0

        passed = (win_rate >= self.promo_cfg.win_rate_threshold
                  and improve_pct >= self.promo_cfg.mean_reward_improve_pct)

        self.logger.record("promotion/win_rate", win_rate)
        self.logger.record("promotion/mean_reward", mean_reward)

        print(f"[promotion] step={self.num_timesteps} win_rate={win_rate:.2f} "
              f"mean_reward={mean_reward:.2f} (baseline={baseline}, "
              f"improve={improve_pct:+.1f}%) pass={passed} "
              f"({self._consecutive_pass + (1 if passed else 0)}/"
              f"{self.promo_cfg.consecutive_passes_required})")

        self._consecutive_pass = self._consecutive_pass + 1 if passed else 0

        if self._consecutive_pass >= self.promo_cfg.consecutive_passes_required:
            tmp_model = self.out_dir / f"_promo_candidate_{self.num_timesteps}.zip"
            tmp_vecnorm = self.out_dir / f"_promo_candidate_{self.num_timesteps}_vecnorm.pkl"
            self.model.save(str(tmp_model))
            vecnorm_train.save(str(tmp_vecnorm))

            version = self.pool.add(str(tmp_model), str(tmp_vecnorm), mean_reward,
                                    win_rate, note=f"step={self.num_timesteps}",
                                    obs_dim=OBS_DIM)
            print(f"[promotion] YENI VERSIYON: v{version} havuza eklendi!")

            tmp_model.unlink(missing_ok=True)
            tmp_vecnorm.unlink(missing_ok=True)
            self._consecutive_pass = 0

        return True


def _seed_pool(args):
    from drone_rl.dogfight.dogfight_env import DogfightEnv, OpponentCurriculum

    pool = CheckpointPool(args.pool)
    run_dir = Path(args.from_run)
    model_path = run_dir / "model_final.zip"
    vecnorm_path = run_dir / "vecnormalize.pkl"

    # DUZELTME: eskiden burada load_dogfight_config(None) cagriliyordu,
    # yani --config yok sayilip VARSAYILAN ayarlarla degerlendirme
    # yapiliyordu. Artik verilen config kullaniliyor.
    cfg = load_dogfight_config(args.config)

    vecnorm = VecNormalize.load(str(vecnorm_path), make_dummy_vecnorm_env())
    stats = NormalizerStats(vecnorm)
    if stats.obs_dim != OBS_DIM:
        raise ValueError(
            f"Stage A checkpoint'i {stats.obs_dim} boyutlu, bu surum {OBS_DIM} "
            f"bekliyor. Eski run'i kullanamazsiniz - Stage A'yi bu surumle "
            f"yeniden calistirin."
        )
    model = PPO.load(str(model_path), device="cpu")

    env = DogfightEnv(cfg.env, opponent_curriculum=OpponentCurriculum(cfg.env))
    env.set_curriculum_progress(1.0)
    env.set_shaped_weight(cfg.env.shaped_weight_end)

    rewards, wins = [], 0
    n = 20
    for _ in range(n):
        obs, _ = env.reset()
        done, ep_r, info = False, 0.0, {}
        while not done:
            norm_obs = stats.normalize(obs).reshape(1, -1)
            action, _ = model.predict(norm_obs, deterministic=True)
            obs, r, term, trunc, info = env.step(action[0])
            ep_r += r
            done = term or trunc
        rewards.append(ep_r)
        if _episode_won(info):
            wins += 1

    mean_reward = float(np.mean(rewards))
    win_rate = wins / n
    version = pool.add(str(model_path), str(vecnorm_path), mean_reward, win_rate,
                       note="stage-a seed", obs_dim=OBS_DIM)
    print(f"Havuz tohumlandi: v{version}, mean_reward={mean_reward:.2f}, "
          f"win_rate={win_rate:.2f}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--stage", type=str, choices=["a", "b"])
    ap.add_argument("--config", type=str)
    ap.add_argument("--out", type=str, default="/content/runs/dogfight_run")
    ap.add_argument("--pool", type=str, default=None)
    ap.add_argument("--timesteps", type=int, default=None)
    ap.add_argument("--n-envs", type=int, default=None)
    ap.add_argument("--vec", type=str, default=None, choices=["auto", "dummy", "subproc"])
    ap.add_argument("--seed", type=int, default=None)
    ap.add_argument("--snapshot-freq", type=int, default=10000)
    ap.add_argument("--seed-pool", action="store_true")
    ap.add_argument("--from", dest="from_run", type=str)
    # YENI: --resume - runs/<out>/live_snapshot/{model.zip,vecnormalize.pkl}
    # mevcutsa egitim SIFIRDAN degil, oradan devam eder. Colab runtime
    # kopmalarinda ilerlemenin kaybolmamasi icin eklendi.
    ap.add_argument("--resume", action="store_true",
                    help="live_snapshot'ta kayitli model/vecnormalize varsa oradan devam et")
    args = ap.parse_args()

    if args.seed_pool:
        if not args.from_run or not args.pool:
            raise ValueError("--seed-pool icin --from ve --pool gerekli")
        _seed_pool(args)
        return

    cfg = load_dogfight_config(args.config)
    timesteps = args.timesteps or cfg.train.timesteps
    n_envs = args.n_envs or cfg.train.n_envs
    vec = args.vec or cfg.train.vec
    seed = args.seed if args.seed is not None else cfg.train.seed
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)

    live_dir = out / "live_snapshot"
    resume_model_path = live_dir / "model.zip"
    resume_vecnorm_path = live_dir / "vecnormalize.pkl"
    do_resume = bool(args.resume and resume_model_path.exists() and resume_vecnorm_path.exists())
    if args.resume and not do_resume:
        print(f"[resume] --resume verildi ama '{live_dir}' altinda kayitli bir "
             f"snapshot bulunamadi (model.zip/vecnormalize.pkl) - SIFIRDAN baslaniyor.")

    venv = make_dogfight_training_vec_env(
        cfg.env, n_envs=n_envs, stage=args.stage, pool_dir=args.pool,
        training=True, norm_reward=True, clip_reward=cfg.ppo.clip_reward,
        vec=vec, seed=seed,
        vecnormalize_path=str(resume_vecnorm_path) if do_resume else None,
    )

    policy_kwargs = build_policy_kwargs(cfg.ppo)

    if do_resume:
        # DIKKAT: policy_kwargs / mimari buradan gecilmiyor - PPO.load
        # kaydedilmis modelin KENDI mimarisini ve agirliklarini geri
        # yukler. Sadece cevre (env) YENI (guncel kod/odul duzeltmeleri
        # ile) baglaniyor - boylece TRAIN.PY'DA yapilan bir duzeltme
        # (orn. _episode_won mantigi ya da odul fonksiyonu) agirliklari
        # SIFIRLAMADAN devreye girer.
        model = PPO.load(str(resume_model_path), env=venv, device="cpu")

        # YENI: cokmus/zehirlenmis bir checkpoint'i SESSIZCE yukleyip
        # AYNI hatayla tekrar cokmek yerine, burada erkenden ve ACIKCA
        # durur.
        if _model_has_nan(model):
            raise RuntimeError(
                f"\n[resume] KRITIK: '{resume_model_path}' agirliklarinda "
                f"NaN/Inf tespit edildi - bu checkpoint bir NaN cokusu "
                f"SIRASINDA ya da SONRASINDA kaydedilmis, artik KULLANILAMAZ "
                f"(gozlemler temiz olsa bile ag kendi kendine NaN uretmeye "
                f"devam eder).\n\n"
                f"Secenekler:\n"
                f"  1) '{out}/ckpt/' klasorunde daha ESKI bir 'ppo_*_steps.zip' "
                f"var mi kontrol edin (varsa NaN icermeyen birini secip "
                f"--resume YERINE bu betigi elle o dosyayi yukleyecek "
                f"sekilde calistirmak gerekir).\n"
                f"  2) Havuzdaki (CheckpointPool) en son PROMOTE EDILMIS "
                f"versiyon KESINLIKLE NaN icermez (promosyon degerlendirmesi "
                f"NaN aksiyonla asla basarili olamaz) - 'python main.py "
                f"pool-info' ile kontrol edip o versiyonu yeni bir egitimin "
                f"baslangici olarak kullanabilirsiniz.\n"
                f"  3) --resume KULLANMADAN sifirdan baslatin - gozlem "
                f"kirpma duzeltmesi sayesinde bu sorun BIR DAHA olusmayacak."
            )

        already_done = int(model.num_timesteps)
        remaining = max(timesteps - already_done, 0)
        print(f"[resume] {resume_model_path} yuklendi.")
        print(f"[resume] su ana kadar tamamlanan adim : {already_done}")
        print(f"[resume] yaml'daki hedef adim          : {timesteps}")
        print(f"[resume] bu calistirmada kalan adim    : {remaining}")
    else:
        model = PPO(
            cfg.ppo.policy, venv,
            n_steps=cfg.ppo.n_steps, batch_size=cfg.ppo.batch_size, n_epochs=cfg.ppo.n_epochs,
            gamma=cfg.ppo.gamma, gae_lambda=cfg.ppo.gae_lambda, clip_range=cfg.ppo.clip_range,
            learning_rate=linear_schedule(cfg.ppo.learning_rate, cfg.ppo.lr_final_frac),
            ent_coef=cfg.ppo.ent_coef, vf_coef=cfg.ppo.vf_coef,
            max_grad_norm=cfg.ppo.max_grad_norm, target_kl=cfg.ppo.target_kl,
            policy_kwargs=policy_kwargs, verbose=1, device="cpu", seed=seed,
            tensorboard_log=str(out / "tb"),
        )
        already_done = 0
        remaining = timesteps

    # DUZELTME: save_vecnormalize=True eklendi - eskiden ckpt/ klasoru
    # SADECE model agirliklarini kaydediyordu, eslesen vecnormalize.pkl'i
    # DEGIL. Bu yuzden bir NaN cokusu sonrasi geri donulebilecek, hem
    # model hem normalizasyon istatistikleri birlikte SAGLAM bir ara
    # nokta yoktu - sadece live_snapshot vardi, o da corken onunla
    # birlikte bozuluyordu.
    ckpt_cb = CheckpointCallback(save_freq=max(20_000 // n_envs, 1),
                                 save_path=str(out / "ckpt"), name_prefix="ppo",
                                 save_vecnormalize=True)
    progress_cb = ProgressCallback(cfg.env)
    diag_cb = DiagnosticsCallback()
    snapshot_cb = PeriodicSnapshotCallback(out, save_freq=max(args.snapshot_freq // n_envs, 1))
    callbacks = [ckpt_cb, progress_cb, diag_cb, snapshot_cb]

    if args.stage == "b":
        if not args.pool:
            raise ValueError("--stage b icin --pool gerekli")
        pool = CheckpointPool(args.pool)
        if len(pool) == 0:
            raise ValueError("Havuz bos - once --seed-pool calistirin")
        promo_cb = PromotionCallback(pool, cfg.env, cfg.promotion, out)
        if do_resume:
            # YENI: resume aninda _next_eval'i mevcut ilerlemeye gore
            # HIZALA - yoksa (varsayilan _next_eval=eval_freq oldugu
            # icin) num_timesteps zaten cok ilerideyken ilk birkac
            # _on_step cagrisinda ART ARDA, GEREKSIZ COK SAYIDA
            # degerlendirme (her biri n_eval_episodes bolum!) tetiklenir.
            promo_cb._next_eval = (
                (already_done // cfg.promotion.eval_freq) + 1
            ) * cfg.promotion.eval_freq
            print(f"[resume] promotion degerlendirmesi bir sonraki "
                 f"adim={promo_cb._next_eval}'de tetiklenecek.")
        callbacks.append(promo_cb)

    if remaining > 0:
        model.learn(total_timesteps=remaining, reset_num_timesteps=False, callback=callbacks)
    else:
        print(f"[resume] zaten hedef adim sayisina ({timesteps}) ulasilmis, "
             f"egitim atlaniyor.")

    model.save(out / "model_final")
    venv.save(str(out / "vecnormalize.pkl"))
    print(f"Egitim tamamlandi: {out} (toplam num_timesteps={model.num_timesteps})")


if __name__ == "__main__":
    main()


Overwriting /content/repo/src/drone_rl/dogfight/train.py


In [15]:
%%writefile /content/repo/src/drone_rl/dogfight/realtime_dogfight_server.py
"""Gercek zamanli dogfight demo sunucusu - v8.

Degisiklikler:
  * Payload yeni odul/geometri anahtarlarina gore guncellendi
    (ATA/AA, HP, angajman ekseni, track/threat/dist/close kirilimi).
  * Koni yonelimi artik Euler acilarindan degil, ortamin hesapladigi
    ANGAJMAN EKSENI vektorunden (self_axis/opp_axis) cizilir - hem
    isaret/siralama hatalarini ortadan kaldirir hem de ekranda gorulen
    koni ile odulde kullanilan koni BIREBIR ayni olur.
  * VecNormalize yuklenirken artik gercek bir DogfightEnv (2 JSBSim
    motoru) kurulmuyor; _DummyObsEnv kullaniliyor.
"""

import asyncio
import json
import os

from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.responses import FileResponse

from drone_rl.dogfight.config import load_dogfight_config
from drone_rl.dogfight.dogfight_env import DogfightEnv, NormalizerStats, OBS_DIM
from drone_rl.dogfight.env_factory import load_opponent_controller, make_dummy_vecnorm_env
from drone_rl.dogfight.checkpoint_pool import CheckpointPool

app = FastAPI()
STATE = {
    "html_path": None, "env": None, "training_controller": None,
    "demo_max_steps": None, "live_snapshot_dir": None, "pool_dir": None,
    "reload_interval_s": 2.0, "_last_training_mtime": None,
    "_last_pool_version": None, "_last_event": None,
    "training_timesteps": None,
}


@app.get("/")
def index():
    return FileResponse(STATE["html_path"])


def _load_training_controller(model_path, vecnorm_path):
    from stable_baselines3 import PPO
    from stable_baselines3.common.vec_env import VecNormalize

    vecnorm = VecNormalize.load(vecnorm_path, make_dummy_vecnorm_env())
    stats = NormalizerStats(vecnorm)
    if stats.obs_dim != OBS_DIM:
        raise ValueError(
            f"Snapshot {stats.obs_dim} boyutlu, bu surum {OBS_DIM} bekliyor "
            f"({vecnorm_path}). Eski run'lar v8 ile uyumlu degil."
        )
    model = PPO.load(model_path, device="cpu")
    return _TrainingSelfController(model, stats)


def _read_training_meta():
    """Hafif json okuma - model reload'undan bagimsiz, boylece timestep
    sayaci arayuzde akici ilerler."""
    meta_path = os.path.join(STATE["live_snapshot_dir"], "meta.json")
    if not os.path.exists(meta_path):
        return
    try:
        with open(meta_path, "r") as f:
            meta = json.load(f)
        STATE["training_timesteps"] = meta.get("num_timesteps")
    except Exception:
        pass


def _check_and_reload_training():
    """DIKKAT: agir, bloklayan islemler icerir. reload_watcher() bunu
    asyncio.to_thread ile ayri thread'de calistirir."""
    _read_training_meta()

    model_path = os.path.join(STATE["live_snapshot_dir"], "model.zip")
    vecnorm_path = os.path.join(STATE["live_snapshot_dir"], "vecnormalize.pkl")
    if not (os.path.exists(model_path) and os.path.exists(vecnorm_path)):
        return
    mtime = os.path.getmtime(model_path)
    if STATE["_last_training_mtime"] is not None and mtime <= STATE["_last_training_mtime"]:
        return

    STATE["training_controller"] = _load_training_controller(model_path, vecnorm_path)
    STATE["_last_training_mtime"] = mtime
    STATE["_last_event"] = {"type": "training_updated"}
    print("[reload] TRAINING modeli guncellendi")


def _check_and_reload_best():
    pool = CheckpointPool(STATE["pool_dir"])
    version = pool.latest_version()
    if version is None:
        return
    if STATE["_last_pool_version"] is not None and version <= STATE["_last_pool_version"]:
        return

    new_controller = load_opponent_controller(*pool.latest(), deterministic=True)
    STATE["env"].set_opponent_controller(new_controller)
    STATE["_last_pool_version"] = version
    STATE["_last_event"] = {"type": "best_updated", "version": version}
    print(f"[reload] BEST modeli guncellendi -> v{version}")


async def reload_watcher():
    while True:
        await asyncio.sleep(STATE["reload_interval_s"])
        try:
            if STATE["env"] is not None:
                await asyncio.to_thread(_check_and_reload_training)
                await asyncio.to_thread(_check_and_reload_best)
        except Exception as e:
            print(f"[reload] hata (atlaniyor): {e}")


@app.on_event("startup")
async def _on_startup():
    asyncio.create_task(reload_watcher())


@app.websocket("/ws")
async def dogfight_loop(websocket: WebSocket):
    await websocket.accept()
    env: DogfightEnv = STATE["env"]
    demo_max_steps = STATE["demo_max_steps"]

    obs, _ = env.reset()
    total_steps = 0
    finished = False
    cumulative_my_score = 0
    cumulative_opp_score = 0
    episode_count = 0
    last_reset_reason = None
    info = None

    try:
        while True:
            if not finished:
                training_action = STATE["training_controller"].compute_action_for_self(env)
                obs, reward, terminated, truncated, info = env.step(training_action)
                total_steps += 1

                if terminated or truncated:
                    cumulative_my_score += info["my_score"]
                    cumulative_opp_score += info["opp_score"]
                    episode_count += 1
                    last_reset_reason = info.get("reset_reason")
                    obs, _ = env.reset()

                if demo_max_steps and total_steps >= demo_max_steps:
                    finished = True

            payload = {
                "self_pos": info["self_pos"], "opp_pos": info["opp_pos"],
                "self_attitude": info["self_attitude"], "opp_attitude": info["opp_attitude"],
                "self_axis": list(info["self_axis"]), "opp_axis": list(info["opp_axis"]),
                "self_hdot_fps": info["self_hdot_fps"], "opp_hdot_fps": info["opp_hdot_fps"],
                "self_speed_fps": info["self_speed_fps"], "opp_speed_fps": info["opp_speed_fps"],
                "range_ft": info["range_ft"], "closing_fps": info["closing_fps"],
                "ata_deg": info["ata_deg"], "aa_deg": info["aa_deg"],
                "opp_ata_deg": info["opp_ata_deg"],
                "opp_in_my_cone": info["opp_in_my_cone"], "me_in_opp_cone": info["me_in_opp_cone"],
                "hp_self": info["hp_self"], "hp_opp": info["hp_opp"],
                "hp_initial": info["hp_initial"],
                # odul kirilimi
                "track_reward": info["track_reward"], "threat_penalty": info["threat_penalty"],
                "dist_penalty": info["dist_penalty"], "close_reward": info["close_reward"],
                "lock_reward": info["lock_reward"], "exposed_penalty": info["exposed_penalty"],
                "control_penalty": info["control_penalty"], "safety_penalty": info["safety_penalty"],
                "opp_track_reward": info["opp_track_reward"],
                "opp_threat_penalty": info["opp_threat_penalty"],
                "opp_dist_penalty": info["opp_dist_penalty"],
                "opp_close_reward": info["opp_close_reward"],
                "opp_control_penalty": info["opp_control_penalty"],
                "opp_safety_penalty": info["opp_safety_penalty"],
                "self_reward": info["self_reward"], "opp_reward": info["opp_reward"],
                "shaped_weight": info["shaped_weight"],
                "training_timesteps": STATE["training_timesteps"],
                "my_score": cumulative_my_score + info["my_score"],
                "opp_score": cumulative_opp_score + info["opp_score"],
                "episode_count": episode_count, "last_reset_reason": last_reset_reason,
                "cone_half_angle_deg": env.cfg.cone_half_angle_deg,
                "cone_range_ft": env.cfg.cone_range_ft,
                "step": total_steps, "finished": finished,
            }
            if STATE["_last_event"] is not None:
                payload["event"] = STATE["_last_event"]
                STATE["_last_event"] = None

            await websocket.send_text(json.dumps(payload))
            await asyncio.sleep(env.control_dt if not finished else 1.0)

    except WebSocketDisconnect:
        pass


class _TrainingSelfController:
    def __init__(self, model, stats):
        self.model = model
        self.stats = stats

    def compute_action_for_self(self, env: DogfightEnv):
        obs = env._get_obs_for(env.fdm_self, env.fdm_opp, env.prev_action_self)
        norm_obs = self.stats.normalize(obs).reshape(1, -1)
        action, _ = self.model.predict(norm_obs, deterministic=True)
        return action[0]


def start_server(live_snapshot_dir: str, pool_dir: str, config_path: str, html_path: str,
                 demo_max_steps: int = None, reload_interval_s: float = 2.0, port: int = 8020):
    import threading
    import uvicorn

    cfg = load_dogfight_config(config_path)

    model_path = os.path.join(live_snapshot_dir, "model.zip")
    vecnorm_path = os.path.join(live_snapshot_dir, "vecnormalize.pkl")
    if not (os.path.exists(model_path) and os.path.exists(vecnorm_path)):
        raise FileNotFoundError(f"Henuz bir egitim anlik goruntusu yok: {live_snapshot_dir}")

    pool = CheckpointPool(pool_dir)
    if len(pool) == 0:
        raise FileNotFoundError(f"Havuz bos: {pool_dir}. Once --seed-pool calistirin.")

    opp_controller = load_opponent_controller(*pool.latest(), deterministic=True)
    env = DogfightEnv(cfg.env, opponent_controller=opp_controller)
    env.set_shaped_weight(cfg.env.shaped_weight_end)
    env.set_curriculum_progress(1.0)

    STATE["env"] = env
    STATE["training_controller"] = _load_training_controller(model_path, vecnorm_path)
    STATE["html_path"] = html_path
    STATE["demo_max_steps"] = demo_max_steps
    STATE["live_snapshot_dir"] = live_snapshot_dir
    STATE["pool_dir"] = pool_dir
    STATE["reload_interval_s"] = reload_interval_s
    STATE["_last_training_mtime"] = os.path.getmtime(model_path)
    STATE["_last_pool_version"] = pool.latest_version()
    STATE["_last_event"] = None

    meta_path = os.path.join(live_snapshot_dir, "meta.json")
    if os.path.exists(meta_path):
        try:
            with open(meta_path, "r") as f:
                STATE["training_timesteps"] = json.load(f).get("num_timesteps")
        except Exception:
            STATE["training_timesteps"] = None

    thread = threading.Thread(
        target=lambda: uvicorn.run(app, host="0.0.0.0", port=port, log_level="warning"),
        daemon=True,
    )
    thread.start()
    print(f"Dogfight sunucusu baslatildi (port {port}). Her {reload_interval_s}s kontrol edilecek.")

Overwriting /content/repo/src/drone_rl/dogfight/realtime_dogfight_server.py


In [111]:
%%writefile /content/repo/configs/dogfight_stage_a.yaml

# Stage A: mufredat rakiplerine karsi temel takip/nisan alma.
# v8 - standoff_* ve reward_align_* parametreleri KALDIRILDI.
env:
  episode_seconds: 45.0
  physics_hz: 240
  control_hz: 20

  hover_throttle: 0.420
  throttle_range: 0.25
  roll_authority: 0.6
  pitch_authority: 0.6
  yaw_authority: 0.45
  control_surface_tau_s: 0.08
  # `python main.py calibrate` ciktisina gore ayarlayin (+1.0 veya -1.0)
  forward_pitch_sign: 1.0

  velocity_norm_fps: 30.0
  range_norm_ft: 100.0
  closing_norm_fps: 20.0
  dz_norm_ft: 50.0
  engage_axis_blend_fps: 8.0

  cone_half_angle_deg: 25.0
  cone_range_ft: 60.0

  reward_track_weight: 0.50
  reward_threat_weight: 0.35
  track_sigma_rad: 0.80

  reward_dist_weight: 0.15
  dist_ref_ft: 150.0
  reward_close_weight: 0.30
  closing_ref_fps: 15.0
  too_close_ft: 25.0
  reward_too_close_weight: 0.30

  reward_cone_hold: 0.15

  reward_tilt_weight: 0.02
  reward_spin_weight: 0.03
  reward_yawrate_weight: 0.02
  reward_jerk_weight: 0.03

  hp_initial: 3.0
  hp_damage_rate: 1.0
  win_bonus: 50.0
  timeout_hp_bonus: 15.0

  crash_penalty: 25.0
  opponent_fault_bonus: 10.0
  crash_min_alt_ft: 20.0
  crash_max_alt_ft: 300.0
  crash_max_tilt_rad: 1.40
  tilt_soft_rad: 0.60
  tilt_soft_weight: 0.25
  yawrate_soft_rps: 4.0
  yawrate_soft_weight: 0.10
  max_horizontal_range_ft: 400.0
  boundary_soft_margin_ft: 120.0
  boundary_soft_weight: 0.25
  min_separation_ft: 8.0

  # YENI (mentor istegi): egim/irtifa/sinir ihlalleri artik episode'u
  # SONLANDIRMIYOR, sadece cezalandiriliyor. Carpisma ve HP=0 (kazanma/
  # kaybetme) bu ayardan BAGIMSIZ, her zaman episode'u bitirir.
  terminate_on_fault: true

  # YENI: terminate_on_fault=False olsa BILE HER ZAMAN sonlandiran,
  # cok gevsek bir emniyet freni - JSBSim'in fizik motoru gercekten
  # sayisal olarak kirilmadan (NaN/Inf) once devreye girer.
  extreme_tilt_rad: 2.5
  extreme_altitude_min_ft: -100.0
  extreme_altitude_max_ft: 2000.0
  extreme_boundary_ft: 2000.0

  # YENI: irtifa tabani/tavani icin kademeli ceza (yer carpmasi sorunu icin)
  alt_floor_soft_margin_ft: 60.0
  alt_floor_soft_weight: 0.35
  alt_ceiling_soft_margin_ft: 40.0
  alt_ceiling_soft_weight: 0.15

  # YENI: asiri hizli inisi dogrudan cezalandirir
  descent_rate_soft_fps: 12.0
  descent_rate_soft_weight: 0.12

  shaped_weight_start: 1.00
  shaped_weight_end: 0.55
  shaped_ramp_steps: 500000
  curriculum_ramp_steps: 350000

  base_altitude_ft: 150.0
  altitude_jitter_ft: 25.0
  spawn_range_min_ft: 80.0
  spawn_range_max_ft: 220.0
  spawn_speed_max_fps: 18.0
  spawn_attitude_jitter_rad: 0.12

  opponent_latest_prob: 0.60
  opponent_pfsp_temp: 0.25
  opponent_deterministic: false

ppo:
  policy: MlpPolicy
  n_steps: 1024
  batch_size: 256
  n_epochs: 10
  gamma: 0.995
  gae_lambda: 0.95
  clip_range: 0.2
  learning_rate: 0.0003
  lr_final_frac: 0.1
  ent_coef: 0.005
  vf_coef: 0.5
  max_grad_norm: 0.5
  target_kl: 0.05
  clip_reward: 10.0
  net_arch_pi: [256, 256]
  net_arch_vf: [256, 256]
  activation_fn: tanh

train:
  timesteps: 800000
  n_envs: 6
  vec: auto
  seed: 0

promotion:
  eval_freq: 30000
  n_eval_episodes: 20
  win_rate_threshold: 0.55
  mean_reward_improve_pct: 5.0
  consecutive_passes_required: 2




Overwriting /content/repo/configs/dogfight_stage_a.yaml


In [113]:
%%writefile /content/repo/configs/dogfight_stage_b.yaml
# Stage B: havuzdan orneklenen dondurulmus PPO rakiplerine karsi self-play.
# v8 - standoff_* ve reward_align_* parametreleri KALDIRILDI.
env:
  episode_seconds: 45.0
  physics_hz: 240
  control_hz: 20

  hover_throttle: 0.420
  throttle_range: 0.25
  roll_authority: 0.6
  pitch_authority: 0.6
  yaw_authority: 0.45
  control_surface_tau_s: 0.08
  # `python main.py calibrate` ciktisina gore ayarlayin (+1.0 veya -1.0)
  forward_pitch_sign: 1.0

  velocity_norm_fps: 30.0
  range_norm_ft: 100.0
  closing_norm_fps: 20.0
  dz_norm_ft: 50.0
  engage_axis_blend_fps: 8.0

  cone_half_angle_deg: 25.0
  cone_range_ft: 60.0

  reward_track_weight: 0.50
  reward_threat_weight: 0.35
  track_sigma_rad: 0.80

  reward_dist_weight: 0.15
  dist_ref_ft: 150.0
  reward_close_weight: 0.30
  closing_ref_fps: 15.0
  too_close_ft: 25.0
  reward_too_close_weight: 0.30

  reward_cone_hold: 0.15

  reward_tilt_weight: 0.02
  reward_spin_weight: 0.03
  reward_yawrate_weight: 0.02
  reward_jerk_weight: 0.03

  hp_initial: 3.0
  hp_damage_rate: 1.0
  win_bonus: 50.0
  timeout_hp_bonus: 15.0

  crash_penalty: 25.0
  opponent_fault_bonus: 10.0
  crash_min_alt_ft: 20.0
  crash_max_alt_ft: 300.0
  crash_max_tilt_rad: 1.40
  tilt_soft_rad: 0.60
  tilt_soft_weight: 0.25
  yawrate_soft_rps: 4.0
  yawrate_soft_weight: 0.10
  max_horizontal_range_ft: 400.0
  boundary_soft_margin_ft: 120.0
  boundary_soft_weight: 0.25
  min_separation_ft: 8.0

  # YENI (mentor istegi): egim/irtifa/sinir ihlalleri artik episode'u
  # SONLANDIRMIYOR, sadece cezalandiriliyor. Carpisma ve HP=0 (kazanma/
  # kaybetme) bu ayardan BAGIMSIZ, her zaman episode'u bitirir.
  terminate_on_fault: true

  # YENI: terminate_on_fault=False olsa BILE HER ZAMAN sonlandiran,
  # cok gevsek bir emniyet freni - JSBSim'in fizik motoru gercekten
  # sayisal olarak kirilmadan (NaN/Inf) once devreye girer.
  extreme_tilt_rad: 2.5
  extreme_altitude_min_ft: -100.0
  extreme_altitude_max_ft: 2000.0
  extreme_boundary_ft: 2000.0

  # YENI: irtifa tabani/tavani icin kademeli ceza (yer carpmasi sorunu icin)
  alt_floor_soft_margin_ft: 60.0
  alt_floor_soft_weight: 0.35
  alt_ceiling_soft_margin_ft: 40.0
  alt_ceiling_soft_weight: 0.15

  # YENI: asiri hizli inisi dogrudan cezalandirir
  descent_rate_soft_fps: 12.0
  descent_rate_soft_weight: 0.12

  shaped_weight_start: 1.00
  shaped_weight_end: 0.45
  shaped_ramp_steps: 600000
  curriculum_ramp_steps: 1

  base_altitude_ft: 150.0
  altitude_jitter_ft: 25.0
  spawn_range_min_ft: 80.0
  spawn_range_max_ft: 220.0
  spawn_speed_max_fps: 18.0
  spawn_attitude_jitter_rad: 0.12

  opponent_latest_prob: 0.60
  opponent_pfsp_temp: 0.25
  opponent_deterministic: false

ppo:
  policy: MlpPolicy
  n_steps: 1024
  batch_size: 256
  n_epochs: 10
  gamma: 0.995
  gae_lambda: 0.95
  clip_range: 0.2
  learning_rate: 0.0003
  lr_final_frac: 0.1
  ent_coef: 0.005
  vf_coef: 0.5
  max_grad_norm: 0.5
  target_kl: 0.05
  clip_reward: 10.0
  net_arch_pi: [256, 256]
  net_arch_vf: [256, 256]
  activation_fn: tanh

train:
  timesteps: 2000000
  n_envs: 6
  vec: auto
  seed: 1

promotion:
  eval_freq: 30000
  n_eval_episodes: 20
  win_rate_threshold: 0.55
  mean_reward_improve_pct: 5.0
  consecutive_passes_required: 2






Overwriting /content/repo/configs/dogfight_stage_b.yaml


In [18]:
%%writefile /content/repo/main.py
#!/usr/bin/env python3
import argparse
import sys
import time
import webbrowser
from pathlib import Path

REPO_ROOT = Path(__file__).resolve().parent
SRC_DIR = REPO_ROOT / "src"
sys.path.insert(0, str(SRC_DIR))

CONFIGS_DIR = REPO_ROOT / "configs"
RUNS_DIR = REPO_ROOT / "runs"
HTML_PATH = REPO_ROOT / "dogfightSim_realtime.html"


def _common_train_argv(args, base):
    argv = list(base)
    if args.timesteps:
        argv += ["--timesteps", str(args.timesteps)]
    if args.n_envs:
        argv += ["--n-envs", str(args.n_envs)]
    if getattr(args, "vec", None):
        argv += ["--vec", args.vec]
    if getattr(args, "seed", None) is not None:
        argv += ["--seed", str(args.seed)]
    if getattr(args, "resume", False):
        argv += ["--resume"]
    argv += ["--snapshot-freq", str(args.snapshot_freq)]
    return argv


def cmd_train_a(args):
    from drone_rl.dogfight import train as train_mod
    out = args.out or str(RUNS_DIR / "dogfight_stage_a")
    config = args.config or str(CONFIGS_DIR / "dogfight_stage_a.yaml")
    sys.argv = _common_train_argv(
        args, ["train.py", "--stage", "a", "--config", config, "--out", out])
    train_mod.main()


def cmd_seed_pool(args):
    from drone_rl.dogfight import train as train_mod
    from_run = args.from_run or str(RUNS_DIR / "dogfight_stage_a")
    pool = args.pool or str(RUNS_DIR / "dogfight_pool")
    config = args.config or str(CONFIGS_DIR / "dogfight_stage_a.yaml")
    sys.argv = ["train.py", "--seed-pool", "--from", from_run,
                "--pool", pool, "--config", config]
    train_mod.main()


def cmd_train_b(args):
    from drone_rl.dogfight import train as train_mod
    out = args.out or str(RUNS_DIR / "dogfight_stage_b")
    config = args.config or str(CONFIGS_DIR / "dogfight_stage_b.yaml")
    pool = args.pool or str(RUNS_DIR / "dogfight_pool")
    sys.argv = _common_train_argv(
        args, ["train.py", "--stage", "b", "--config", config,
               "--out", out, "--pool", pool])
    train_mod.main()


def cmd_calibrate(args):
    from drone_rl.dogfight import calibrate as cal_mod
    config = args.config or str(CONFIGS_DIR / "dogfight_stage_a.yaml")
    cal_mod.main(config)


def cmd_pool_info(args):
    from drone_rl.dogfight.checkpoint_pool import CheckpointPool
    pool = CheckpointPool(args.pool or str(RUNS_DIR / "dogfight_pool"))
    print(pool.summary())


def cmd_demo(args):
    from drone_rl.dogfight.realtime_dogfight_server import start_server
    live_snapshot_dir = args.live_snapshot_dir or str(RUNS_DIR / "dogfight_stage_b" / "live_snapshot")
    pool_dir = args.pool or str(RUNS_DIR / "dogfight_pool")
    config = args.config or str(CONFIGS_DIR / "dogfight_stage_b.yaml")
    html_path = args.html or str(HTML_PATH)

    start_server(
        live_snapshot_dir=live_snapshot_dir, pool_dir=pool_dir,
        config_path=config, html_path=html_path,
        demo_max_steps=args.max_steps, reload_interval_s=args.reload_interval,
        port=args.port,
    )

    url = f"http://localhost:{args.port}"
    print(f"\nSunucu hazir: {url}")
    if not args.no_browser:
        try:
            webbrowser.open(url)
        except Exception:
            pass
    print("Durdurmak icin Ctrl+C.\n")
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        print("\nKapatiliyor.")


def build_parser():
    ap = argparse.ArgumentParser(prog="main.py")
    sub = ap.add_subparsers(dest="command", required=True)

    def add_train_args(p):
        p.add_argument("--config", type=str, default=None)
        p.add_argument("--out", type=str, default=None)
        p.add_argument("--timesteps", type=int, default=None)
        p.add_argument("--n-envs", type=int, default=None)
        p.add_argument("--vec", type=str, default=None,
                       choices=["auto", "dummy", "subproc"])
        p.add_argument("--seed", type=int, default=None)
        p.add_argument("--resume", action="store_true",
                       help="live_snapshot'ta kayitli model/vecnormalize varsa oradan devam et")
        p.add_argument("--snapshot-freq", type=int, default=10000)

    p_a = sub.add_parser("train-a")
    add_train_args(p_a)
    p_a.set_defaults(func=cmd_train_a)

    p_seed = sub.add_parser("seed-pool")
    p_seed.add_argument("--from", dest="from_run", type=str, default=None)
    p_seed.add_argument("--pool", type=str, default=None)
    p_seed.add_argument("--config", type=str, default=None)
    p_seed.set_defaults(func=cmd_seed_pool)

    p_b = sub.add_parser("train-b")
    add_train_args(p_b)
    p_b.add_argument("--pool", type=str, default=None)
    p_b.set_defaults(func=cmd_train_b)

    p_cal = sub.add_parser("calibrate")
    p_cal.add_argument("--config", type=str, default=None)
    p_cal.set_defaults(func=cmd_calibrate)

    p_pool = sub.add_parser("pool-info")
    p_pool.add_argument("--pool", type=str, default=None)
    p_pool.set_defaults(func=cmd_pool_info)

    p_demo = sub.add_parser("demo")
    p_demo.add_argument("--live-snapshot-dir", dest="live_snapshot_dir", type=str, default=None)
    p_demo.add_argument("--pool", type=str, default=None)
    p_demo.add_argument("--config", type=str, default=None)
    p_demo.add_argument("--html", type=str, default=None)
    p_demo.add_argument("--port", type=int, default=8020)
    p_demo.add_argument("--max-steps", dest="max_steps", type=int, default=None)
    p_demo.add_argument("--reload-interval", dest="reload_interval", type=float, default=2.0)
    p_demo.add_argument("--no-browser", action="store_true")
    p_demo.set_defaults(func=cmd_demo)

    return ap


def main():
    parser = build_parser()
    args = parser.parse_args()
    args.func(args)


if __name__ == "__main__":
    main()


Overwriting /content/repo/main.py


In [31]:
%%writefile /content/repo/export_tacview_csv.py
"""Iki drone'un (TRAINING, BEST) ucusunu Tacview'un RESMI "Real-life CSV"
formatinda kaydeder - onceki export_acmi.py'den FARKLI: o Tacview'un
KENDI native ACMI formatini uretiyordu, bu ise Tacview'un DUZ CSV
IMPORT ozelligine uygun, dokumantasyona birebir uyan bir cikti uretir:

    https://raia-software-inc.gitbook.io/tacview/real-life-data/real-life-csv-data

O sayfadaki kurallar:
  * Sutunlar: Time, Longitude, Latitude, Altitude, Roll, Pitch, Yaw
    (Time=kayit basindan saniye; Longitude/Latitude=derece;
    Altitude=metre; Roll/Pitch/Yaw=derece)
  * BIR CSV DOSYASI = BIR UCAK. Iki drone icin IKI AYRI dosya uretilir;
    Tacview'da once birini File->Open ile acip sonra digerini
    File->Merge ile eklersiniz - boylece ikisi ayni sahnede,
    senkronize gorunur.
  * Ucak adi/pilot/renk METADATASI DOSYA ADINDAN okunur:
    "<NATO adi> (<pilot>) [<renk>].csv"

DUZELTME (surekli kayit): Eskiden TEK bolum kaydediliyordu ve bir
carpisma/crash olunca kayit o anda kesiliyordu - bolumler bazen
~10 saniyede bitince 'daha ne oldugunu anlamadan' kayit sona eriyordu.
Simdi varsayilan olarak EN AZ --min-duration-s (varsayilan 60s)
uzunlugunda KESINTISIZ bir kayit uretiliyor: bir bolum biterse ortam
otomatik resetlenip Time sutunu AYNI kesintisiz eksende artmaya
devam ediyor - hedef sureye ulasana kadar.

DIKKAT: Onceki export_episode_csv.py (analiz/rapor icin duz tablo CSV)
DOKUNULMADI, degistirilmedi - bu TAMAMEN AYRI, Tacview'a OZEL bir
script'tir.

Egitim (Stage B) ARKA PLANDA calisirken de guvenle calistirilabilir:
sadece diskteki mevcut snapshot/havuz dosyalarini OKUR, kendi ayri bir
degerlendirme ortaminda calisir.

Kullanim (repo kokunden):
    cd /content/repo && python tools/export_tacview_csv.py --min-duration-s 60

Cikti (exports/ klasorunde), her kayit icin:
    Quadrotor (TRAINING) [Blue] - rec1.csv
    Quadrotor (BEST) [Red] - rec1.csv

Tacview'da acma:
    1) "Quadrotor (TRAINING) [Blue] - rec1.csv" dosyasini File->Open ile ac
    2) "Quadrotor (BEST) [Red] - rec1.csv" dosyasini File->Merge ile ekle
"""

import argparse
import csv
import math
from pathlib import Path

from drone_rl.dogfight.config import load_dogfight_config
from drone_rl.dogfight.dogfight_env import (
    DogfightEnv, NormalizerStats, LAT0_DEG, LON0_DEG, FT_PER_DEG_LAT,
)
from drone_rl.dogfight.env_factory import load_opponent_controller, make_dummy_vecnorm_env
from drone_rl.dogfight.checkpoint_pool import CheckpointPool

FT_TO_M = 0.3048

# Tacview'un resmi CSV sutun basliklari (dokumantasyona birebir uygun)
CSV_HEADER = ["Time", "Longitude", "Latitude", "Altitude", "Roll", "Pitch", "Yaw"]


def _find_repo_root() -> Path:
    candidates = [Path.cwd()]
    here = Path(__file__).resolve()
    candidates += [here.parent, here.parent.parent, here.parent.parent.parent]
    for c in candidates:
        if (c / "configs").is_dir() and (c / "src" / "drone_rl").is_dir():
            return c
    tried = "\n".join(f"  - {c}" for c in candidates)
    raise FileNotFoundError(
        "Repo koku otomatik bulunamadi. Denenen yerler:\n" + tried +
        "\n'cd /content/repo && python tools/export_tacview_csv.py' seklinde "
        "calistirin ya da --config/--live-snapshot-dir/--pool verin."
    )


REPO_ROOT = _find_repo_root()
RUNS_DIR = REPO_ROOT / "runs"
EXPORT_DIR = REPO_ROOT / "exports"


def _north_east_to_lonlat(north_ft: float, east_ft: float):
    """DogfightEnv._position()'un TAM TERSI - ayni sabitlerle, boylece
    CSV'deki konum ortamin kendi ic tutarliligiyla BIREBIR eslesir."""
    ft_per_deg_lon = FT_PER_DEG_LAT * math.cos(math.radians(LAT0_DEG))
    lat = LAT0_DEG + north_ft / FT_PER_DEG_LAT
    lon = LON0_DEG + east_ft / ft_per_deg_lon
    return lon, lat


def _heading_deg(yaw_rad: float) -> float:
    """Tacview 'Yaw' alani: gercek kuzeye (true north) gore, derece.
    JSBSim'in psi'si zaten kuzeyden saat yonunde olcup ayni
    konvansiyonu kullandigi icin sadece radyandan dereceye cevirip
    0-360 araligina sariyoruz."""
    return math.degrees(yaw_rad) % 360.0


class _TrainingController:
    def __init__(self, model_path, vecnorm_path):
        from stable_baselines3 import PPO
        from stable_baselines3.common.vec_env import VecNormalize

        vecnorm = VecNormalize.load(str(vecnorm_path), make_dummy_vecnorm_env())
        self.stats = NormalizerStats(vecnorm)
        self.model = PPO.load(str(model_path), device="cpu")

    def compute_action(self, env):
        obs = env._get_obs_for(env.fdm_self, env.fdm_opp, env.prev_action_self)
        norm_obs = self.stats.normalize(obs).reshape(1, -1)
        action, _ = self.model.predict(norm_obs, deterministic=True)
        return action[0]


def _build_eval_env(config_path, live_snapshot_dir, pool_dir):
    cfg = load_dogfight_config(config_path)

    live_dir = Path(live_snapshot_dir)
    model_path = live_dir / "model.zip"
    vecnorm_path = live_dir / "vecnormalize.pkl"
    if not (model_path.exists() and vecnorm_path.exists()):
        raise FileNotFoundError(
            f"Egitim anlik goruntusu bulunamadi: {live_dir}\n"
            f"Stage B en az bir snapshot yazana kadar bekleyin."
        )

    pool = CheckpointPool(pool_dir)
    if len(pool) == 0:
        raise FileNotFoundError(f"Havuz bos: {pool_dir}. Once seed-pool calistirin.")

    opp_controller = load_opponent_controller(*pool.latest(), deterministic=True)
    env = DogfightEnv(cfg.env, opponent_controller=opp_controller)
    env.set_shaped_weight(cfg.env.shaped_weight_end)
    env.set_curriculum_progress(1.0)
    # YENI: sadece bu kayit ortami icin - egitim config'i etkilenmez.
    env.set_terminate_on_fault(False)

    training_ctrl = _TrainingController(model_path, vecnorm_path)
    return env, training_ctrl, pool.latest_version()


def run_continuous_recording(env, training_ctrl, min_duration_s: float, max_episodes: int):
    """Ortami, TOPLAM sure en az min_duration_s olana kadar art arda
    bolumler halinde kosturur; Time sutunu KESINTISIZ (bolumler arasi
    sifirlanmadan) artmaya devam eder. Bir bolum crash/timeout ile
    biterse ortam otomatik resetlenip kayit devam eder.

    Doner: (training_rows, best_rows, toplam_sure_s, kosturulan_bolum_sayisi)
    """
    training_rows, best_rows = [], []
    total_time = 0.0
    episodes_run = 0

    while total_time < min_duration_s and episodes_run < max_episodes:
        env.reset()
        episodes_run += 1
        info = None
        while True:
            action = training_ctrl.compute_action(env)
            obs, reward, terminated, truncated, info = env.step(action)
            total_time += env.control_dt
            t = round(total_time, 3)

            self_n, self_e, self_alt = info["self_pos"]
            opp_n, opp_e, opp_alt = info["opp_pos"]
            s_roll, s_pitch, s_yaw = info["self_attitude"]
            o_roll, o_pitch, o_yaw = info["opp_attitude"]

            s_lon, s_lat = _north_east_to_lonlat(self_n, self_e)
            o_lon, o_lat = _north_east_to_lonlat(opp_n, opp_e)

            training_rows.append([
                f"{t:.2f}", f"{s_lon:.9f}", f"{s_lat:.9f}", f"{self_alt * FT_TO_M:.2f}",
                f"{math.degrees(s_roll):.3f}", f"{math.degrees(s_pitch):.3f}",
                f"{_heading_deg(s_yaw):.3f}",
            ])
            best_rows.append([
                f"{t:.2f}", f"{o_lon:.9f}", f"{o_lat:.9f}", f"{opp_alt * FT_TO_M:.2f}",
                f"{math.degrees(o_roll):.3f}", f"{math.degrees(o_pitch):.3f}",
                f"{_heading_deg(o_yaw):.3f}",
            ])

            ended = terminated or truncated
            if ended:
                print(f"    bolum {episodes_run} bitti (t={total_time:.1f}s, "
                     f"sebep={info['reset_reason']}) - hedefe ulasilmadiysa devam ediliyor")
                break
            if total_time >= min_duration_s:
                break

    if total_time < min_duration_s:
        print(f"  UYARI: {max_episodes} bolume ragmen {min_duration_s}s hedefine "
             f"ulasilamadi (toplam {total_time:.1f}s). Bolumler beklenenden cok "
             f"kisa suruyor olabilir - env/egitim durumunu kontrol edin.")

    return training_rows, best_rows, total_time, episodes_run


def write_tacview_csv(path: Path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(CSV_HEADER)
        writer.writerows(rows)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", type=str, default=None)
    ap.add_argument("--live-snapshot-dir", type=str, default=None)
    ap.add_argument("--pool", type=str, default=None)
    ap.add_argument("--recordings", type=int, default=1,
                    help="Kac ayri kayit (dosya cifti) uretilecek, her biri en az --min-duration-s")
    ap.add_argument("--min-duration-s", type=float, default=60.0,
                    help="Her kaydin en az kac saniye surecegi (varsayilan 60)")
    ap.add_argument("--max-episodes", type=int, default=40,
                    help="Bir kayit icinde art arda kosturulacak en fazla bolum sayisi")
    args = ap.parse_args()

    config_path = args.config or str(REPO_ROOT / "configs" / "dogfight_stage_b.yaml")
    live_snapshot_dir = args.live_snapshot_dir or str(RUNS_DIR / "dogfight_stage_b" / "live_snapshot")
    pool_dir = args.pool or str(RUNS_DIR / "dogfight_pool")

    print(f"repo koku      : {REPO_ROOT}")
    print(f"config         : {config_path}")
    print(f"live-snapshot  : {live_snapshot_dir}")
    print(f"pool           : {pool_dir}")

    env, training_ctrl, pool_version = _build_eval_env(config_path, live_snapshot_dir, pool_dir)

    print(f"BEST (havuz)   : v{pool_version}")
    print(f"{args.recordings} kayit Tacview CSV olarak uretiliyor, her biri en az "
         f"{args.min_duration_s:.0f}s (gerekirse bolumler otomatik birlestirilecek)...")

    for rec in range(1, args.recordings + 1):
        training_rows, best_rows, total_time, episodes_run = run_continuous_recording(
            env, training_ctrl, min_duration_s=args.min_duration_s,
            max_episodes=args.max_episodes)

        training_path = EXPORT_DIR / f"Quadrotor (TRAINING) [Blue] - rec{rec}.csv"
        best_path = EXPORT_DIR / f"Quadrotor (BEST) [Red] - rec{rec}.csv"

        write_tacview_csv(training_path, training_rows)
        write_tacview_csv(best_path, best_rows)

        print(f"  kayit {rec}: {episodes_run} bolum birlestirildi, toplam {total_time:.1f}s")
        print(f"    -> {training_path.name}")
        print(f"    -> {best_path.name}")

    print("\nTamamlandi. Tacview'da:")
    print("  1) 'Quadrotor (TRAINING) ...csv' dosyasini File->Open ile acin")
    print("  2) 'Quadrotor (BEST) ...csv' dosyasini File->Merge ile ekleyin")


if __name__ == "__main__":
    main()



Overwriting /content/repo/export_tacview_csv.py


In [19]:
%%writefile /content/repo/requirements.txt
jsbsim
torch
stable-baselines3>=2.0
gymnasium
fastapi
uvicorn
pyyaml
numpy
tensorboard

Overwriting /content/repo/requirements.txt


In [20]:
%%writefile /content/repo/dogfightSim_realtime.html
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8" />
<title>F450 Dogfight — Self-Play Simulator</title>
<meta name="viewport" content="width=device-width, initial-scale=1" />
<style>
  :root{
    --bg-0:#0b0f14; --bg-1:#111820; --bg-2:#161f29; --line:#26323e;
    --ink-0:#e8eef3; --ink-1:#9fb0bd; --ink-2:#5f7180;
    --training:#4fd1c5; --best:#e0a05a; --danger:#e0596a;
    --mono: "JetBrains Mono","SF Mono",Consolas,monospace;
    --sans: -apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,sans-serif;
  }
  *{box-sizing:border-box;}
  html,body{ margin:0; height:100%; background:var(--bg-0); color:var(--ink-0); font-family:var(--sans); }
  #app{ display:flex; flex-direction:column; height:100vh; }

  header{
    display:flex; align-items:center; gap:14px; padding:12px 18px;
    background:var(--bg-1); border-bottom:1px solid var(--line); flex-wrap:wrap;
  }
  header h1{ font-size:15px; font-weight:600; margin:0; white-space:nowrap; }
  header h1 span{ color:var(--training); }

  .toggle-group{ display:flex; border:1px solid var(--line); border-radius:6px; overflow:hidden; }
  .toggle-group button{
    background:var(--bg-2); color:var(--ink-1); border:none; padding:7px 14px;
    font-size:12px; font-family:var(--mono); cursor:pointer;
  }
  .toggle-group button.active{ background:var(--training); color:#04211e; font-weight:700; }

  .badge{
    display:flex; align-items:center; gap:6px;
    font-family:var(--mono); font-size:12px; color:var(--ink-1);
    background:var(--bg-2); border:1px solid var(--line); border-radius:6px;
    padding:7px 12px; white-space:nowrap;
  }
  .badge .k{ color:var(--ink-2); }
  .badge .v{ color:var(--ink-0); font-weight:700; }

  .conn-status{
    margin-left:auto; display:flex; align-items:center; gap:8px;
    font-family:var(--mono); font-size:12px;
  }
  .conn-dot{ width:8px; height:8px; border-radius:50%; background:var(--ink-2); }
  .conn-dot.connected{ background:var(--training); }
  .conn-dot.disconnected{ background:var(--danger); }

  main{ flex:1; display:grid; grid-template-columns: 1fr 320px; min-height:0; }
  #viewport-wrap{ position:relative; }
  #three-canvas{ width:100%; height:100%; display:block; }

  #scoreboard{
    position:absolute; top:14px; left:50%; transform:translateX(-50%);
    display:flex; gap:0; background:rgba(17,24,32,.88); border:1px solid var(--line);
    border-radius:10px; overflow:hidden; font-family:var(--mono); backdrop-filter: blur(4px);
  }
  .score-cell{ padding:10px 22px; text-align:center; min-width:130px; }
  .score-cell.training{ border-right:1px solid var(--line); }
  .score-cell .label{ font-size:10px; color:var(--ink-2); letter-spacing:.5px; }
  .score-cell.training .label{ color:var(--training); }
  .score-cell.best .label{ color:var(--best); }
  .score-cell .value{ font-size:26px; font-weight:700; margin-top:2px; }
  .score-cell.scoring{ animation: flash 0.5s ease-in-out; }
  @keyframes flash{ 0%,100%{background:transparent;} 50%{background:rgba(255,255,255,.15);} }

  /* YENI: HP cubugu (WEZ hasar modeli) */
  .hp-track{ height:5px; border-radius:3px; background:#22303c; margin-top:7px; overflow:hidden; }
  .hp-fill{ height:100%; width:100%; transition:width .15s linear; }
  .score-cell.training .hp-fill{ background:var(--training); }
  .score-cell.best .hp-fill{ background:var(--best); }
  .hp-fill.low{ background:var(--danger) !important; }

  #hud{
    position:absolute; bottom:14px; left:14px;
    background:rgba(17,24,32,.85); border:1px solid var(--line);
    border-radius:8px; padding:10px 14px; font-family:var(--mono); font-size:12px;
    line-height:1.7; color:var(--ink-0); backdrop-filter: blur(4px); min-width:210px;
  }
  #hud .row{ display:flex; justify-content:space-between; gap:18px; }
  #hud .row .k{ color:var(--ink-2); }
  #hud .cone-status.active{ color:var(--danger); font-weight:700; }
  #hud .reset-reason{ color:var(--best); }
  #hud .closing.pos{ color:var(--training); }
  #hud .closing.neg{ color:var(--danger); }

  #finished-overlay{
    position:absolute; inset:0; display:none; align-items:center; justify-content:center;
    background:rgba(11,15,20,.75); backdrop-filter: blur(3px);
  }
  #finished-overlay.show{ display:flex; }
  #finished-overlay .box{
    background:var(--bg-1); border:1px solid var(--line); border-radius:12px;
    padding:28px 40px; text-align:center;
  }
  #finished-overlay h2{ margin:0 0 8px; color:var(--training); }
  #finished-overlay p{ margin:0; color:var(--ink-1); font-family:var(--mono); font-size:13px; }

  aside#compare{
    background:var(--bg-1); border-left:1px solid var(--line);
    padding:14px; overflow:auto; font-family:var(--mono); font-size:11px;
  }
  aside#compare h3{ margin:0 0 10px; font-size:11px; color:var(--ink-2); font-weight:600; letter-spacing:.5px; }
  .compare-table{ width:100%; border-collapse:collapse; }
  .compare-table th{
    text-align:right; padding:5px 6px; font-size:10px; font-weight:700;
    border-bottom:1px solid var(--line);
  }
  .compare-table th.metric-col{ text-align:left; color:var(--ink-2); font-weight:400; }
  .compare-table th.training-col{ color:var(--training); }
  .compare-table th.best-col{ color:var(--best); }
  .compare-table td{ padding:5px 6px; text-align:right; border-bottom:1px solid rgba(38,50,62,.5); }
  .compare-table td.metric-name{ text-align:left; color:var(--ink-2); }
  .compare-table tr.section-row td{
    padding-top:12px; color:var(--ink-1); font-weight:700; border-bottom:1px solid var(--line);
  }
  .compare-table tr.total-row td{
    padding-top:8px; border-top:1px solid var(--line); font-weight:700; color:var(--ink-0);
  }

  #toast-container{
    position:absolute; top:14px; right:14px; display:flex; flex-direction:column;
    gap:8px; align-items:flex-end; pointer-events:none; z-index:20;
  }
  .toast{
    background:rgba(17,24,32,.95); border:1px solid var(--line); border-left:3px solid var(--training);
    border-radius:8px; padding:10px 16px; font-family:var(--mono); font-size:12px;
    color:var(--ink-0); box-shadow:0 4px 14px rgba(0,0,0,.4);
    animation: toast-in 0.3s ease-out, toast-out 0.4s ease-in 4.6s forwards;
    max-width:280px;
  }
  .toast.best{ border-left-color:var(--best); }
  .toast .toast-title{ font-weight:700; margin-bottom:2px; }
  .toast.training .toast-title{ color:var(--training); }
  .toast.best .toast-title{ color:var(--best); }
  @keyframes toast-in{ from{ opacity:0; transform:translateX(20px);} to{ opacity:1; transform:translateX(0);} }
  @keyframes toast-out{ from{ opacity:1; } to{ opacity:0; transform:translateX(20px);} }
</style>
</head>
<body>
<div id="app">
  <header>
    <h1>F450 <span>Dogfight</span> — Self-Play</h1>
    <div class="toggle-group">
      <button id="btn-3d" class="active">3D</button>
      <button id="btn-2d">2D (top-down)</button>
    </div>
    <div class="badge">
      <span class="k">timestep</span><span class="v" id="hud-timestep">–</span>
    </div>
    <div class="badge">
      <span class="k">shaped w</span><span class="v" id="hud-shaped">–</span>
    </div>
    <div class="conn-status">
      <span class="conn-dot" id="conn-dot"></span>
      <span id="conn-text">connecting…</span>
    </div>
  </header>

  <main>
    <div id="viewport-wrap">
      <canvas id="three-canvas"></canvas>

      <div id="toast-container"></div>

      <div id="scoreboard">
        <div class="score-cell training" id="cell-training">
          <div class="label">TRAINING</div>
          <div class="value" id="score-training">0</div>
          <div class="hp-track"><div class="hp-fill" id="hp-training"></div></div>
        </div>
        <div class="score-cell best" id="cell-best">
          <div class="label">BEST</div>
          <div class="value" id="score-best">0</div>
          <div class="hp-track"><div class="hp-fill" id="hp-best"></div></div>
        </div>
      </div>

      <div id="hud">
        <div class="row"><span class="k">range</span><span class="v" id="hud-range">–</span></div>
        <div class="row"><span class="k">closing</span><span class="v closing" id="hud-closing">–</span></div>
        <div class="row"><span class="k">ATA / AA</span><span class="v" id="hud-ata">–</span></div>
        <div class="row"><span class="k">step</span><span class="v" id="hud-step">–</span></div>
        <div class="row"><span class="k">episode</span><span class="v" id="hud-episode">–</span></div>
        <div class="row"><span class="k">son reset</span><span class="v reset-reason" id="hud-reset-reason">–</span></div>
        <div class="row"><span class="k">TRAINING → WEZ</span><span class="v cone-status" id="hud-cone-mine">–</span></div>
        <div class="row"><span class="k">BEST → WEZ</span><span class="v cone-status" id="hud-cone-opp">–</span></div>
      </div>

      <div id="finished-overlay">
        <div class="box">
          <h2>SIMULATION COMPLETE</h2>
          <p id="finished-summary">–</p>
        </div>
      </div>
    </div>

    <aside id="compare">
      <h3>CANLI KARŞILAŞTIRMA</h3>
      <table class="compare-table">
        <thead>
          <tr>
            <th class="metric-col">metrik</th>
            <th class="training-col">TRAINING</th>
            <th class="best-col">BEST</th>
          </tr>
        </thead>
        <tbody>
          <tr class="section-row"><td colspan="3">Kinematik</td></tr>
          <tr><td class="metric-name">roll (°)</td><td id="cmp-roll-t">–</td><td id="cmp-roll-b">–</td></tr>
          <tr><td class="metric-name">pitch (°)</td><td id="cmp-pitch-t">–</td><td id="cmp-pitch-b">–</td></tr>
          <tr><td class="metric-name">yaw (°)</td><td id="cmp-yaw-t">–</td><td id="cmp-yaw-b">–</td></tr>
          <tr><td class="metric-name">hız (ft/s)</td><td id="cmp-spd-t">–</td><td id="cmp-spd-b">–</td></tr>
          <tr><td class="metric-name">irtifa (ft)</td><td id="cmp-alt-t">–</td><td id="cmp-alt-b">–</td></tr>
          <tr><td class="metric-name">dikey hız (ft/s)</td><td id="cmp-hdot-t">–</td><td id="cmp-hdot-b">–</td></tr>
          <tr><td class="metric-name">HP</td><td id="cmp-hp-t">–</td><td id="cmp-hp-b">–</td></tr>
          <tr class="section-row"><td colspan="3">Ödül bileşenleri (adım başı)</td></tr>
          <tr><td class="metric-name">track (ATA·AA)</td><td id="cmp-track-t">–</td><td id="cmp-track-b">–</td></tr>
          <tr><td class="metric-name">threat</td><td id="cmp-threat-t">–</td><td id="cmp-threat-b">–</td></tr>
          <tr><td class="metric-name">distance</td><td id="cmp-dist-t">–</td><td id="cmp-dist-b">–</td></tr>
          <tr><td class="metric-name">closing</td><td id="cmp-close-t">–</td><td id="cmp-close-b">–</td></tr>
          <tr><td class="metric-name">lock / exposed</td><td id="cmp-lock-t">–</td><td id="cmp-lock-b">–</td></tr>
          <tr><td class="metric-name">control</td><td id="cmp-control-t">–</td><td id="cmp-control-b">–</td></tr>
          <tr><td class="metric-name">safety</td><td id="cmp-safety-t">–</td><td id="cmp-safety-b">–</td></tr>
          <tr class="total-row"><td class="metric-name">TOPLAM ödül</td><td id="cmp-reward-t">–</td><td id="cmp-reward-b">–</td></tr>
        </tbody>
      </table>
    </aside>
  </main>
</div>

<script type="importmap">
{
  "imports": {
    "three": "https://unpkg.com/three@0.160.0/build/three.module.js",
    "three/addons/": "https://unpkg.com/three@0.160.0/examples/jsm/"
  }
}
</script>

<script type="module">
import * as THREE from "three";
import { OrbitControls } from "three/addons/controls/OrbitControls.js";

// max_horizontal_range_ft = 400 (v8)
const ARENA_HALF_FT = 400;

const canvas = document.getElementById("three-canvas");
const connDot = document.getElementById("conn-dot");
const connText = document.getElementById("conn-text");
const scoreTrainingEl = document.getElementById("score-training");
const scoreBestEl = document.getElementById("score-best");
const cellTraining = document.getElementById("cell-training");
const cellBest = document.getElementById("cell-best");
const hpTrainingEl = document.getElementById("hp-training");
const hpBestEl = document.getElementById("hp-best");
const toastContainer = document.getElementById("toast-container");
const timestepValueEl = document.getElementById("hud-timestep");
const shapedValueEl = document.getElementById("hud-shaped");

const hud = {
  range: document.getElementById("hud-range"),
  closing: document.getElementById("hud-closing"),
  ata: document.getElementById("hud-ata"),
  step: document.getElementById("hud-step"),
  episode: document.getElementById("hud-episode"),
  resetReason: document.getElementById("hud-reset-reason"),
  coneMine: document.getElementById("hud-cone-mine"),
  coneOpp: document.getElementById("hud-cone-opp"),
};
const cmp = {
  rollT: document.getElementById("cmp-roll-t"), rollB: document.getElementById("cmp-roll-b"),
  pitchT: document.getElementById("cmp-pitch-t"), pitchB: document.getElementById("cmp-pitch-b"),
  yawT: document.getElementById("cmp-yaw-t"), yawB: document.getElementById("cmp-yaw-b"),
  spdT: document.getElementById("cmp-spd-t"), spdB: document.getElementById("cmp-spd-b"),
  altT: document.getElementById("cmp-alt-t"), altB: document.getElementById("cmp-alt-b"),
  hdotT: document.getElementById("cmp-hdot-t"), hdotB: document.getElementById("cmp-hdot-b"),
  hpT: document.getElementById("cmp-hp-t"), hpB: document.getElementById("cmp-hp-b"),
  trackT: document.getElementById("cmp-track-t"), trackB: document.getElementById("cmp-track-b"),
  threatT: document.getElementById("cmp-threat-t"), threatB: document.getElementById("cmp-threat-b"),
  distT: document.getElementById("cmp-dist-t"), distB: document.getElementById("cmp-dist-b"),
  closeT: document.getElementById("cmp-close-t"), closeB: document.getElementById("cmp-close-b"),
  lockT: document.getElementById("cmp-lock-t"), lockB: document.getElementById("cmp-lock-b"),
  controlT: document.getElementById("cmp-control-t"), controlB: document.getElementById("cmp-control-b"),
  safetyT: document.getElementById("cmp-safety-t"), safetyB: document.getElementById("cmp-safety-b"),
  rewardT: document.getElementById("cmp-reward-t"), rewardB: document.getElementById("cmp-reward-b"),
};
const finishedOverlay = document.getElementById("finished-overlay");
const finishedSummary = document.getElementById("finished-summary");

const RESET_REASON_LABELS = {
  collision: "ÇARPIŞMA",
  opponent_down: "BEST düşürüldü ✓",
  self_down: "TRAINING düşürüldü",
  mutual_down: "ikisi de düştü",
  self_ground: "TRAINING yere çarptı",
  self_ceiling: "TRAINING tavana çıktı",
  self_tumble: "TRAINING kontrolü kaybetti",
  self_boundary: "TRAINING sınır dışı",
  opponent_ground: "BEST yere çarptı",
  opponent_ceiling: "BEST tavana çıktı",
  opponent_tumble: "BEST kontrolü kaybetti",
  opponent_boundary: "BEST sınır dışı",
  timeout: "süre doldu",
};

function showToast(kind, title, detail) {
  const el = document.createElement("div");
  el.className = `toast ${kind}`;
  el.innerHTML = `<div class="toast-title">${title}</div><div>${detail}</div>`;
  toastContainer.appendChild(el);
  setTimeout(() => el.remove(), 5000);
}

function handleEvent(event) {
  if (!event) return;
  if (event.type === "training_updated") {
    showToast("training", "TRAINING güncellendi", "Eğitim ilerledi, yeni model yüklendi.");
  } else if (event.type === "best_updated") {
    showToast("best", "BEST güncellendi", `Promotion gerçekleşti → v${event.version}`);
  }
}

const scene = new THREE.Scene();
const camera = new THREE.PerspectiveCamera(50, 1, 0.1, 4000);
camera.up.set(0, 0, 1);

const renderer = new THREE.WebGLRenderer({ canvas, antialias: true });
renderer.setPixelRatio(Math.min(window.devicePixelRatio, 2));

const controls = new OrbitControls(camera, renderer.domElement);
controls.enableDamping = true;
controls.dampingFactor = 0.08;
controls.target.set(0, 0, 150);
camera.position.set(350, -350, 400);
controls.update();

scene.add(new THREE.AmbientLight(0xffffff, 0.6));
const sun = new THREE.DirectionalLight(0xffffff, 0.9);
sun.position.set(200, -150, 300);
scene.add(sun);

const grid = new THREE.GridHelper(ARENA_HALF_FT * 2 * 1.15, 46, 0x2a3644, 0x1a232c);
grid.rotation.x = Math.PI / 2;
scene.add(grid);

/* ==================================================================
   DUZELTME (v8) - GORSELLESTIRME
   ------------------------------------------------------------------
   1) Govde yonelimi: eskiden rotation.set(roll, pitch, yaw, "XYZ")
      kullaniliyordu. Ucak icin dogru sira ZYX'tir (R = Rz*Ry*Rx).
      Ayrica x=kuzey / y=dogu / z=yukari eslemesinde +Y ekseni
      etrafinda POZITIF theta burnu ASAGI cevirir, oysa JSBSim'de
      pozitif theta burun YUKARI demektir -> pitch isareti de ters
      cevriliyor. Sonuc: set(roll, -pitch, yaw, "ZYX").
   2) Koni yonelimi: artik Euler acilarindan turetilmiyor. Sunucu,
      ortamin odulde KULLANDIGI angajman ekseni vektorunu (hiz
      vektoru tabanli) gonderiyor; koni dogrudan bu vektore
      dogrultuluyor. Yani ekranda gordugun koni ile odul/WEZ
      hesabindaki koni BIREBIR ayni.
   ================================================================== */
const ARM_ANGLES_DEG = [45, 135, 225, 315];
const ARM_LEN = 3.0, ARM_W = 0.4, HUB_R = 0.6;

function buildAircraftIcon(bodyColorHex) {
  const group = new THREE.Group();

  const mesh3d = new THREE.Group();
  const bodyMat3d = new THREE.MeshStandardMaterial({ color: bodyColorHex, metalness: 0.2, roughness: 0.6 });
  const armGeom3d = new THREE.BoxGeometry(ARM_LEN, ARM_W, ARM_W);
  ARM_ANGLES_DEG.forEach((deg) => {
    const arm = new THREE.Mesh(armGeom3d, bodyMat3d);
    const rad = THREE.MathUtils.degToRad(deg);
    arm.position.set(Math.cos(rad) * (ARM_LEN / 2), Math.sin(rad) * (ARM_LEN / 2), 0);
    arm.rotation.z = rad;
    mesh3d.add(arm);
  });
  mesh3d.add(new THREE.Mesh(new THREE.SphereGeometry(HUB_R, 12, 12), bodyMat3d));
  // burun isaretcisi (+X): yonelim dogrulugu gozle kontrol edilebilsin
  const nose = new THREE.Mesh(new THREE.BoxGeometry(ARM_LEN * 0.55, 0.25, 0.25), bodyMat3d);
  nose.position.set(ARM_LEN * 0.45, 0, 0);
  mesh3d.add(nose);
  group.add(mesh3d);

  const mesh2d = new THREE.Group();
  const bodyMat2d = new THREE.MeshBasicMaterial({ color: bodyColorHex, side: THREE.DoubleSide });
  const armGeom2d = new THREE.PlaneGeometry(ARM_LEN, ARM_W);
  ARM_ANGLES_DEG.forEach((deg) => {
    const arm = new THREE.Mesh(armGeom2d, bodyMat2d);
    const rad = THREE.MathUtils.degToRad(deg);
    arm.position.set(Math.cos(rad) * (ARM_LEN / 2), Math.sin(rad) * (ARM_LEN / 2), 0);
    arm.rotation.z = rad;
    mesh2d.add(arm);
  });
  mesh2d.visible = false;
  group.add(mesh2d);

  const conePivot3d = new THREE.Group();
  group.add(conePivot3d);
  const conePivot2d = new THREE.Group();
  conePivot2d.visible = false;
  group.add(conePivot2d);

  return { group, mesh3d, mesh2d, conePivot3d, conePivot2d };
}

function makeCone3d(colorHex, halfAngleDeg, rangeFt) {
  const radius = rangeFt * Math.tan(THREE.MathUtils.degToRad(halfAngleDeg));
  const geom = new THREE.ConeGeometry(radius, rangeFt, 24, 1, true);
  geom.translate(0, -rangeFt / 2, 0);
  geom.rotateZ(Math.PI / 2); // +X (on) yonune isaret eder
  const mat = new THREE.MeshBasicMaterial({
    color: colorHex, transparent: true, opacity: 0.10, side: THREE.DoubleSide, depthWrite: false,
  });
  return new THREE.Mesh(geom, mat);
}

function makeCone2d(colorHex, halfAngleDeg, rangeFt) {
  const halfRad = THREE.MathUtils.degToRad(halfAngleDeg);
  const geom = new THREE.CircleGeometry(rangeFt, 24, -halfRad, halfRad * 2);
  const mat = new THREE.MeshBasicMaterial({
    color: colorHex, transparent: true, opacity: 0.18, side: THREE.DoubleSide, depthWrite: false,
  });
  return new THREE.Mesh(geom, mat);
}

const trainingIcon = buildAircraftIcon(0x4fd1c5);
scene.add(trainingIcon.group);
const trainingCone3d = makeCone3d(0x4fd1c5, 25, 60);
trainingIcon.conePivot3d.add(trainingCone3d);
const trainingCone2d = makeCone2d(0x4fd1c5, 25, 60);
trainingIcon.conePivot2d.add(trainingCone2d);

const bestIcon = buildAircraftIcon(0xe0a05a);
scene.add(bestIcon.group);
const bestCone3d = makeCone3d(0xe0a05a, 25, 60);
bestIcon.conePivot3d.add(bestCone3d);
const bestCone2d = makeCone2d(0xe0a05a, 25, 60);
bestIcon.conePivot2d.add(bestCone2d);

let conesSized = false;
function ensureConeSize(f) {
  if (conesSized) return;
  conesSized = true;
  [trainingCone3d, bestCone3d].forEach((c) => {
    c.geometry.dispose();
    const radius = f.cone_range_ft * Math.tan(THREE.MathUtils.degToRad(f.cone_half_angle_deg));
    c.geometry = new THREE.ConeGeometry(radius, f.cone_range_ft, 24, 1, true);
    c.geometry.translate(0, -f.cone_range_ft / 2, 0);
    c.geometry.rotateZ(Math.PI / 2);
  });
  [trainingCone2d, bestCone2d].forEach((c) => {
    c.geometry.dispose();
    const halfRad = THREE.MathUtils.degToRad(f.cone_half_angle_deg);
    c.geometry = new THREE.CircleGeometry(f.cone_range_ft, 24, -halfRad, halfRad * 2);
  });
}

const trailTraining = [];
const trailBest = [];
const TRAIL_LEN = 240;
let trailLineTraining = new THREE.Line(new THREE.BufferGeometry(),
  new THREE.LineBasicMaterial({ color: 0x4fd1c5, transparent: true, opacity: 0.5 }));
let trailLineBest = new THREE.Line(new THREE.BufferGeometry(),
  new THREE.LineBasicMaterial({ color: 0xe0a05a, transparent: true, opacity: 0.5 }));
scene.add(trailLineTraining);
scene.add(trailLineBest);

function resizeRenderer() {
  const w = canvas.clientWidth, h = canvas.clientHeight;
  renderer.setSize(w, h, false);
  camera.aspect = w / Math.max(h, 1);
  camera.updateProjectionMatrix();
}
window.addEventListener("resize", resizeRenderer);

let is2D = false;
document.getElementById("btn-3d").addEventListener("click", () => setMode(false));
document.getElementById("btn-2d").addEventListener("click", () => setMode(true));

function setMode(twoD) {
  is2D = twoD;
  document.getElementById("btn-3d").classList.toggle("active", !twoD);
  document.getElementById("btn-2d").classList.toggle("active", twoD);

  trainingIcon.mesh3d.visible = !twoD;
  trainingIcon.mesh2d.visible = twoD;
  trainingIcon.conePivot3d.visible = !twoD;
  trainingIcon.conePivot2d.visible = twoD;
  bestIcon.mesh3d.visible = !twoD;
  bestIcon.mesh2d.visible = twoD;
  bestIcon.conePivot3d.visible = !twoD;
  bestIcon.conePivot2d.visible = twoD;

  if (twoD) {
    controls.minPolarAngle = 0;
    controls.maxPolarAngle = 0.001;
    camera.position.set(controls.target.x, controls.target.y, controls.target.z + 600);
  } else {
    controls.minPolarAngle = 0;
    controls.maxPolarAngle = Math.PI;
  }
  controls.update();
}

let latestFrame = null;
let lastMyScore = 0, lastOppScore = 0;

const X_AXIS = new THREE.Vector3(1, 0, 0);
const _axisVec = new THREE.Vector3();
const _axisQuat = new THREE.Quaternion();
const CONE_SMOOTH = 0.35;

function applyConeAxis(conePivot3d, conePivot2d, axis) {
  if (!axis) return;
  _axisVec.set(axis[0], axis[1], axis[2]);
  if (_axisVec.lengthSq() < 1e-9) return;
  _axisVec.normalize();
  // group'un kendi donusu yok -> yerel eksenler = dunya eksenleri,
  // dolayisiyla bu quaternion dogrudan dogru sonucu verir.
  _axisQuat.setFromUnitVectors(X_AXIS, _axisVec);
  conePivot3d.quaternion.slerp(_axisQuat, CONE_SMOOTH);
  conePivot2d.rotation.z = Math.atan2(axis[1], axis[0]);
}

function connect() {
  const wsProtocol = window.location.protocol === "https:" ? "wss:" : "ws:";
  const ws = new WebSocket(`${wsProtocol}//${window.location.host}/ws`);

  ws.onopen = () => { connDot.className = "conn-dot connected"; connText.textContent = "connected"; };
  ws.onclose = () => {
    connDot.className = "conn-dot disconnected"; connText.textContent = "disconnected — retrying…";
    setTimeout(connect, 1500);
  };
  ws.onerror = () => ws.close();
  ws.onmessage = (msg) => {
    const f = JSON.parse(msg.data);
    if (f.event) handleEvent(f.event);
    latestFrame = f;
  };
}
connect();

function fmt(v, digits = 2) { return (typeof v === "number" ? v.toFixed(digits) : "–"); }

function setHp(el, hp, hp0) {
  const frac = Math.max(0, Math.min(1, (hp || 0) / (hp0 || 1)));
  el.style.width = `${frac * 100}%`;
  el.classList.toggle("low", frac < 0.34);
}

function renderFrame(f) {
  ensureConeSize(f);

  const [sn, se, salt] = f.self_pos;
  const [on, oe, oalt] = f.opp_pos;
  const [sroll, spitch, syaw] = f.self_attitude;
  const [oroll, opitch, oyaw] = f.opp_attitude;

  trainingIcon.group.position.set(sn, se, salt);
  // DUZELTME: ZYX sirasi + ters pitch isareti
  trainingIcon.mesh3d.rotation.set(sroll, -spitch, syaw, "ZYX");
  trainingIcon.mesh2d.rotation.set(0, 0, syaw);
  applyConeAxis(trainingIcon.conePivot3d, trainingIcon.conePivot2d, f.self_axis);

  bestIcon.group.position.set(on, oe, oalt);
  bestIcon.mesh3d.rotation.set(oroll, -opitch, oyaw, "ZYX");
  bestIcon.mesh2d.rotation.set(0, 0, oyaw);
  applyConeAxis(bestIcon.conePivot3d, bestIcon.conePivot2d, f.opp_axis);

  trailTraining.push(new THREE.Vector3(sn, se, salt));
  if (trailTraining.length > TRAIL_LEN) trailTraining.shift();
  trailLineTraining.geometry.dispose();
  trailLineTraining.geometry = new THREE.BufferGeometry().setFromPoints(trailTraining);

  trailBest.push(new THREE.Vector3(on, oe, oalt));
  if (trailBest.length > TRAIL_LEN) trailBest.shift();
  trailLineBest.geometry.dispose();
  trailLineBest.geometry = new THREE.BufferGeometry().setFromPoints(trailBest);

  hud.range.textContent = `${f.range_ft.toFixed(1)} ft`;
  hud.closing.textContent = `${f.closing_fps >= 0 ? "+" : ""}${f.closing_fps.toFixed(1)} ft/s`;
  hud.closing.classList.toggle("pos", f.closing_fps > 0.5);
  hud.closing.classList.toggle("neg", f.closing_fps < -0.5);
  hud.ata.textContent = `${f.ata_deg.toFixed(0)}° / ${f.aa_deg.toFixed(0)}°`;
  hud.step.textContent = f.step;
  hud.episode.textContent = f.episode_count;
  hud.resetReason.textContent = f.last_reset_reason ? (RESET_REASON_LABELS[f.last_reset_reason] || f.last_reset_reason) : "—";
  hud.coneMine.textContent = f.opp_in_my_cone ? "LOCK ✓" : "—";
  hud.coneMine.classList.toggle("active", f.opp_in_my_cone);
  hud.coneOpp.textContent = f.me_in_opp_cone ? "LOCK ✓" : "—";
  hud.coneOpp.classList.toggle("active", f.me_in_opp_cone);

  timestepValueEl.textContent = (typeof f.training_timesteps === "number")
    ? f.training_timesteps.toLocaleString("tr-TR")
    : "–";
  shapedValueEl.textContent = fmt(f.shaped_weight, 2);

  setHp(hpTrainingEl, f.hp_self, f.hp_initial);
  setHp(hpBestEl, f.hp_opp, f.hp_initial);

  cmp.rollT.textContent = fmt(sroll * 180 / Math.PI, 1);
  cmp.rollB.textContent = fmt(oroll * 180 / Math.PI, 1);
  cmp.pitchT.textContent = fmt(spitch * 180 / Math.PI, 1);
  cmp.pitchB.textContent = fmt(opitch * 180 / Math.PI, 1);
  cmp.yawT.textContent = fmt(((syaw * 180 / Math.PI) + 360) % 360, 1);
  cmp.yawB.textContent = fmt(((oyaw * 180 / Math.PI) + 360) % 360, 1);
  cmp.spdT.textContent = fmt(f.self_speed_fps, 1);
  cmp.spdB.textContent = fmt(f.opp_speed_fps, 1);
  cmp.altT.textContent = fmt(salt, 1);
  cmp.altB.textContent = fmt(oalt, 1);
  cmp.hdotT.textContent = fmt(f.self_hdot_fps, 2);
  cmp.hdotB.textContent = fmt(f.opp_hdot_fps, 2);
  cmp.hpT.textContent = fmt(f.hp_self, 2);
  cmp.hpB.textContent = fmt(f.hp_opp, 2);

  cmp.trackT.textContent = fmt(f.track_reward, 3);
  cmp.trackB.textContent = fmt(f.opp_track_reward, 3);
  cmp.threatT.textContent = fmt(-f.threat_penalty, 3);
  cmp.threatB.textContent = fmt(-f.opp_threat_penalty, 3);
  cmp.distT.textContent = fmt(-f.dist_penalty, 3);
  cmp.distB.textContent = fmt(-f.opp_dist_penalty, 3);
  cmp.closeT.textContent = fmt(f.close_reward, 3);
  cmp.closeB.textContent = fmt(f.opp_close_reward, 3);
  cmp.lockT.textContent = fmt(f.lock_reward - f.exposed_penalty, 3);
  cmp.lockB.textContent = fmt(f.exposed_penalty - f.lock_reward, 3);
  cmp.controlT.textContent = fmt(-f.control_penalty, 3);
  cmp.controlB.textContent = fmt(-f.opp_control_penalty, 3);
  cmp.safetyT.textContent = fmt(-f.safety_penalty, 3);
  cmp.safetyB.textContent = fmt(-f.opp_safety_penalty, 3);
  cmp.rewardT.textContent = fmt(f.self_reward, 3);
  cmp.rewardB.textContent = fmt(f.opp_reward, 3);

  scoreTrainingEl.textContent = f.my_score;
  scoreBestEl.textContent = f.opp_score;
  if (f.my_score > lastMyScore) { cellTraining.classList.add("scoring"); setTimeout(() => cellTraining.classList.remove("scoring"), 500); }
  if (f.opp_score > lastOppScore) { cellBest.classList.add("scoring"); setTimeout(() => cellBest.classList.remove("scoring"), 500); }
  lastMyScore = f.my_score; lastOppScore = f.opp_score;

  if (f.finished) {
    finishedOverlay.classList.add("show");
    finishedSummary.textContent = `TRAINING ${f.my_score} — ${f.opp_score} BEST  (${f.episode_count} episode, ${f.step} adım)`;
  }

  if (!is2D) {
    const midX = (sn + on) / 2, midY = (se + oe) / 2, midZ = (salt + oalt) / 2;
    controls.target.lerp(new THREE.Vector3(midX, midY, midZ), 0.03);
  }
}

function animate() {
  requestAnimationFrame(animate);
  if (latestFrame) renderFrame(latestFrame);
  controls.update();
  resizeRenderer();
  renderer.render(scene, camera);
}
requestAnimationFrame(animate);
</script>
</body>
</html>

Overwriting /content/repo/dogfightSim_realtime.html


In [21]:
%%writefile /content/repo/src/drone_rl/dogfight/calibrate.py

"""pitch_cmd isaret kalibrasyonu.

`_apply_action` icinde `elevator = -pitch_cmd * pitch_authority` esleme
var, ama JSBSim'de elevator komutunun BURNU YUKARI mi ASAGI mi cevirdigi
ucak modeline (ve F450 + ScasEngage kurulumuna) bagli. Bir multikopter
ILERLEMEK icin burnunu ASAGI egmek zorunda oldugundan, scripted rakip
kontrolculerin (HoverOpponent, ScriptedCircleOpponent,
KappaPursuitOpponent) dogru calismasi bu isarete baglidir.

Bu betik dronu havada sabitleyip SABIT pozitif bir pitch_cmd uygular ve
3 saniye sonra burun yonunde mi geriye mi gittigini olcer. Sonucu
`configs/*.yaml` icindeki `env.forward_pitch_sign` alanina yazin.

Kullanim:
    python main.py calibrate
"""

import math

import numpy as np
import jsbsim

from drone_rl.dogfight.config import DogfightEnvConfig, load_dogfight_config


def measure_forward_pitch_sign(cfg: DogfightEnvConfig, hold_s: float = 3.0,
                               verbose: bool = True) -> float:
    physics_dt = 1.0 / cfg.physics_hz
    fdm = jsbsim.FGFDMExec(None)
    fdm.set_debug_level(0)
    if not fdm.load_model("F450"):
        raise RuntimeError("F450 yuklenemedi")
    fdm.set_dt(physics_dt)

    fdm["ic/lat-gc-deg"] = 0.0
    fdm["ic/long-gc-deg"] = 0.0
    fdm["ic/h-agl-ft"] = cfg.base_altitude_ft
    fdm["ic/u-fps"] = 0.0
    fdm["ic/v-fps"] = 0.0
    fdm["ic/w-fps"] = 0.0
    fdm["ic/phi-rad"] = 0.0
    fdm["ic/theta-rad"] = 0.0
    fdm["ic/psi-true-rad"] = 0.0          # burun KUZEYE bakiyor
    fdm.run_ic()
    for i in range(4):
        fdm[f"propulsion/engine[{i}]/set-running"] = 1
    fdm["fcs/ScasEngage"] = 1
    fdm["fcs/aileron-cmd-norm"] = 0.0
    fdm["fcs/elevator-cmd-norm"] = 0.0
    fdm["fcs/rudder-cmd-norm"] = 0.0
    fdm["fcs/throttle-cmd-norm"] = cfg.hover_throttle

    surface = np.zeros(3, dtype=np.float64)
    alpha = physics_dt / (cfg.control_surface_tau_s + physics_dt)
    elevator_target = float(np.clip(-1.0 * cfg.pitch_authority, -1.0, 1.0))  # pitch_cmd = +1

    n_steps = int(hold_s * cfg.physics_hz)
    for _ in range(n_steps):
        surface += alpha * (np.array([0.0, elevator_target, 0.0]) - surface)
        fdm["fcs/aileron-cmd-norm"] = float(surface[0])
        fdm["fcs/elevator-cmd-norm"] = float(surface[1])
        fdm["fcs/rudder-cmd-norm"] = float(surface[2])
        fdm["fcs/throttle-cmd-norm"] = cfg.hover_throttle
        fdm.run()

    v_north = fdm["velocities/v-north-fps"]
    theta = fdm["attitude/theta-rad"]
    u_body = fdm["velocities/u-fps"]

    sign = 1.0 if v_north > 0.0 else -1.0

    if verbose:
        print("=" * 58)
        print("pitch_cmd = +1.0 uygulandiktan sonra (burun kuzeye bakiyordu):")
        print(f"  kuzey hizi      : {v_north:+.3f} ft/s")
        print(f"  govde ileri hizi: {u_body:+.3f} ft/s")
        print(f"  pitch acisi     : {math.degrees(theta):+.2f} deg")
        print("-" * 58)
        print(f"  ONERILEN DEGER  : env.forward_pitch_sign = {sign:+.1f}")
        if abs(v_north) < 0.5:
            print("  UYARI: hareket cok kucuk. pitch_authority/hover_throttle")
            print("         degerlerini kontrol edin veya hold_s'i artirin.")
        print("=" * 58)
    return sign


def main(config_path=None):
    cfg = load_dogfight_config(config_path).env
    return measure_forward_pitch_sign(cfg)


if __name__ == "__main__":
    main()

Overwriting /content/repo/src/drone_rl/dogfight/calibrate.py


In [ ]:
!cd /content/repo && python main.py seed-pool

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


     JSBSim Flight Dynamics Model v1.2.4 Feb  7 2026 11:12:49
            [JSBSim-ML v2.0]

JSBSim startup beginning ...


YOU HAVE AN INCOMPATIBLE CFG FILE FOR THIS AIRCRAFT. RESULTS WILL BE UNPREDICTABLE !!
Current version needed is: 2.0
         You have version: 3.0

Failed to tie property fcs/accelx/malfunction/fail_low to object methods
Failed to tie property fcs/accelx/malfunction/fail_high to object methods
Failed to tie property fcs/accelx/malfunction/fail_stuck to object methods
Failed to tie property fcs/accelx/randomseed to object methods
Failed to tie property fcs/accely/malfunction/fail_low to object methods
Faile

In [42]:
!rm -rf /content/repo/runs/dogfight_stage_b

In [114]:
train_b_process.terminate()
!sleep 2


In [115]:
import os, subprocess
env = {**os.environ, "PYTHONUNBUFFERED": "1"}
train_b_process = subprocess.Popen(
    ["python", "main.py", "train-b", "--resume"],
    cwd="/content/repo", env=env,
    stdout=open("/content/repo/train_b_log.txt", "w"),
    stderr=subprocess.STDOUT,
)
print(f"Stage B (resume) başladı, PID={train_b_process.pid}")


Stage B (resume) başladı, PID=35242


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [122]:
print("Süreç durumu:", "ÇALIŞIYOR" if train_b_process.poll() is None else f"BİTTİ/ÇÖKTÜ (kod={train_b_process.poll()})")
print("-" * 60)
!tail -40 /content/repo/train_b_log.txt

Süreç durumu: ÇALIŞIYOR
------------------------------------------------------------
|    exposed_rate         | 0.00505     |
|    hp_opp_end           | 2.94        |
|    hp_self_end          | 2.95        |
|    lock_rate            | 0.00977     |
|    opp_fault_rate       | 0.00309     |
|    range_ft_mean        | 152         |
|    self_fault_rate      | 0.0013      |
| promotion/              |             |
|    mean_reward          | -39.3       |
|    win_rate             | 0.75        |
| reset_reason/           |             |
|    collision            | 0.0714      |
|    opponent_boundary    | 0.214       |
|    opponent_ceiling     | 0.0357      |
|    opponent_ground      | 0.179       |
|    opponent_tumble      | 0.214       |
|    self_boundary        | 0.143       |
|    self_ground          | 0.107       |
|    self_tumble          | 0.0357      |
| rollout/                |             |
|    ep_len_mean          | 218         |
|    ep_rew_mean          | -29.4

In [60]:

train_b_process.terminate()  # 17604'ü de durdur
!sleep 2
!rm -rf /content/repo/runs/dogfight_stage_b


In [98]:
!cd /content/repo && python export_acmi.py --min-duration-s 60 2>&1 | tail -25

[reload] TRAINING modeli guncellendi
Failed to tie property fcs/accelx/randomseed to object methods
Failed to tie property fcs/accely/malfunction/fail_low to object methods
Failed to tie property fcs/accely/malfunction/fail_high to object methods
Failed to tie property fcs/accely/malfunction/fail_stuck to object methods
Failed to tie property fcs/accely/randomseed to object methods
Failed to tie property fcs/accelz/malfunction/fail_low to object methods
Failed to tie property fcs/accelz/malfunction/fail_high to object methods
Failed to tie property fcs/accelz/malfunction/fail_stuck to object methods
Failed to tie property fcs/accelz/randomseed to object methods
Failed to tie property fcs/temp_c/malfunction/fail_low to object methods
Failed to tie property fcs/temp_c/malfunction/fail_high to object methods
Failed to tie property fcs/temp_c/malfunction/fail_stuck to object methods
Failed to tie property fcs/temp_c/randomseed to object methods
repo koku      : /content/repo
config        

In [99]:
!ls -la /content/repo/exports/dogfight_recording_1.acmi
!wc -l /content/repo/exports/dogfight_recording_1.acmi

-rw-r--r-- 1 root root 277954 Sep 15 06:11 /content/repo/exports/dogfight_recording_1.acmi
3606 /content/repo/exports/dogfight_recording_1.acmi


In [105]:
def parse_lonlat_alt(line):
    t = line.split("T=")[1].split(",")[0]
    p = t.split("|")
    return float(p[0]), float(p[1]), float(p[2])  # lon, lat, alt(m)

def find_resets(lines, label, jump_threshold_m=5.0):
    vals = [parse_lonlat_alt(l) for l in lines]
    resets = []
    for i in range(1, len(vals)):
        lon0, lat0, alt0 = vals[i-1]
        lon1, lat1, alt1 = vals[i]
        alt_jump = abs(alt1 - alt0)
        # lon/lat farkini kabaca metreye cevir (kaba yaklasim, sadece
        # buyuk siçramayi yakalamak icin yeterli)
        lat_jump_m = abs(lat1 - lat0) * 111320.0
        lon_jump_m = abs(lon1 - lon0) * 111320.0
        if alt_jump > jump_threshold_m or lat_jump_m > jump_threshold_m or lon_jump_m > jump_threshold_m:
            resets.append((i, i * 0.05, alt_jump, lat_jump_m, lon_jump_m))
    print(f"\n=== {label}: {len(resets)} reset/sicrama bulundu ===")
    for i, t, aj, latj, lonj in resets:
        print(f"  frame {i:5d}  t={t:6.2f}s  irtifa_sicrama={aj:7.2f}m  lat_sicrama={latj:7.2f}m  lon_sicrama={lonj:7.2f}m")
    return resets

resets_t = find_resets(training_lines, "TRAINING")
resets_b = find_resets(best_lines, "BEST")

print(f"\nTOPLAM: TRAINING={len(resets_t)} reset, BEST={len(resets_b)} reset")
print("(NOT: reset ayni bolum degisikliginde HER IKI drone da birden")
print(" resetlenir - o yuzden ayni frame numarasinda cikmalari beklenir)")



=== TRAINING: 3 reset/sicrama bulundu ===
  frame   447  t= 22.35s  irtifa_sicrama=  76.35m  lat_sicrama= 292.80m  lon_sicrama=  36.53m
  frame   893  t= 44.65s  irtifa_sicrama=  42.11m  lat_sicrama=  91.74m  lon_sicrama= 104.02m
  frame  1161  t= 58.05s  irtifa_sicrama=  34.74m  lat_sicrama= 129.00m  lon_sicrama=  51.41m

=== BEST: 3 reset/sicrama bulundu ===
  frame   447  t= 22.35s  irtifa_sicrama=  35.58m  lat_sicrama= 144.84m  lon_sicrama= 192.18m
  frame   893  t= 44.65s  irtifa_sicrama=  27.51m  lat_sicrama=  45.04m  lon_sicrama= 122.47m
  frame  1161  t= 58.05s  irtifa_sicrama=  79.65m  lat_sicrama= 100.25m  lon_sicrama= 160.65m

TOPLAM: TRAINING=3 reset, BEST=3 reset
(NOT: reset ayni bolum degisikliginde HER IKI drone da birden
 resetlenir - o yuzden ayni frame numarasinda cikmalari beklenir)


In [109]:
!grep -B 2 -A 2 "ata_deg_mean\|lock_rate\|self_fault_rate" /content/repo/train_b_log.txt | tail -100

| reset_reason/           |             |
|    collision            | 0.0323      |
--
|    shaped_weight        | 0.629      |
| dogfight/               |            |
|    ata_deg_mean         | 94.6       |
|    closing_fps_mean     | -12.6      |
|    exposed_rate         | 0.0142     |
|    hp_opp_end           | 3          |
|    hp_self_end          | 2.9        |
|    lock_rate            | 0.000488   |
|    opp_fault_rate       | 0.00277    |
|    range_ft_mean        | 152        |
|    self_fault_rate      | 0.00195    |
| reset_reason/           |            |
|    opponent_boundary    | 0.103      |
--
|    shaped_weight        | 0.623       |
| dogfight/               |             |
|    ata_deg_mean         | 94.3        |
|    closing_fps_mean     | -14.1       |
|    exposed_rate         | 0.00358     |
|    hp_opp_end           | 2.96        |
|    hp_self_end          | 2.98        |
|    lock_rate            | 0.00505     |
|    opp_fault_rate       | 0.00277     |

In [ ]:
!cd /content/repo && python main.py pool-info

Pool: /content/repo/runs/dogfight_pool (1 versiyon)
  v1: mean_reward=-39.32 win_rate=0.00 obs_dim=30 note=stage-a seed


In [33]:
!ls -la /content/repo/runs/dogfight_stage_b/live_snapshot/ 2>&1

total 1800
drwxr-xr-x 2 root root    4096 Sep 15 04:25 .
drwxr-xr-x 5 root root    4096 Sep 15 04:29 ..
-rw-r--r-- 1 root root      40 Sep 15 04:40 meta.json
-rw-r--r-- 1 root root 1824087 Sep 15 04:40 model.zip
-rw-r--r-- 1 root root    3363 Sep 15 04:40 vecnormalize.pkl


In [123]:
import sys
sys.path.insert(0, "/content/repo/src")
from drone_rl.dogfight.realtime_dogfight_server import start_server

start_server(
    live_snapshot_dir="/content/repo/runs/dogfight_stage_b/live_snapshot",
    pool_dir="/content/repo/runs/dogfight_pool",
    config_path="/content/repo/configs/dogfight_stage_b.yaml",
    html_path="/content/repo/dogfightSim_realtime.html",
    reload_interval_s=15.0,
    port=8030,
)

Dogfight sunucusu baslatildi (port 8030). Her 15.0s kontrol edilecek.


In [124]:
from google.colab.output import serve_kernel_port_as_window
serve_kernel_port_as_window(8030)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [ ]:
train_b_process.terminate()

In [ ]:
!rm -rf /content/repo/runs /content/repo/train_a_log.txt /content/repo/train_b_log.txt

In [ ]:
!cd /content/repo && python main.py calibrate



     JSBSim Flight Dynamics Model v1.2.4 Feb  7 2026 11:12:49
            [JSBSim-ML v2.0]

JSBSim startup beginning ...


YOU HAVE AN INCOMPATIBLE CFG FILE FOR THIS AIRCRAFT. RESULTS WILL BE UNPREDICTABLE !!
Current version needed is: 2.0
         You have version: 3.0

Failed to tie property fcs/accelx/malfunction/fail_low to object methods
Failed to tie property fcs/accelx/malfunction/fail_high to object methods
Failed to tie property fcs/accelx/malfunction/fail_stuck to object methods
Failed to tie property fcs/accelx/randomseed to object methods
Failed to tie property fcs/accely/malfunction/fail_low to object methods
Failed to tie property fcs/accely/malfunction/fail_high to object methods
Failed to tie property fcs/accely/malfunction/fail_stuck to object methods
Failed to tie property fcs/accely/randomseed to object methods
Failed to tie property fcs/accelz/malfunction/fail_low to object methods
Failed to tie property fcs/accelz/malfunction/fail_high to object methods
Failed to

In [ ]:
import os, subprocess
env = {**os.environ, "PYTHONUNBUFFERED": "1"}
train_a_process = subprocess.Popen(
    ["python", "main.py", "train-a"],
    cwd="/content/repo", env=env,
    stdout=open("/content/repo/train_a_log.txt", "w"),
    stderr=subprocess.STDOUT,
)
print(f"Stage A arka planda başladı, PID={train_a_process.pid}")

Stage A arka planda başladı, PID=2896


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/repo/runs/dogfight_stage_b/tb

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


<IPython.core.display.Javascript object>

[reload] TRAINING modeli guncellendi


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
print("Süreç durumu:", "ÇALIŞIYOR" if train_a_process.poll() is None else f"BİTTİ/ÇÖKTÜ (kod={train_a_process.poll()})")
print("-" * 60)
!tail -40 /content/repo/train_a_log.txt

NameError: name 'train_a_process' is not defined